# Barbershop Baseline vs HairFastGAN B24 — LOOKLAB Native Sharp
ใช้วิธีจาก `LOOKLAB_Diffusion_Native_Sharp.ipynb` กับผล GAN ทั้งสองโมเดลด้วยฟังก์ชันและค่าเดียวกัน

**4 ขั้น × 2 โมเดล = 8 PNG:** ก่อน diffusion → raw diffusion → ปรับพื้นหลัง → Native Sharp
- ขนาดเดิม 1024×1024 ไม่ใช้ Real-ESRGAN / Swin2SR และไม่เพิ่ม diffusion pass
- diffusion: seed 3407, strength .15, steps 30, guidance 2.5 ตามไฟล์แนบ
- Native Sharp: hair .65, face .25, max delta 5 ระดับ RGB, threshold .8 ตามไฟล์แนบ
- Barbershop ใช้ Baseline จาก validation protocol 27f7f146ef0d; HFG ใช้ B epoch24 ที่ตรวจ SHA256
- ใช้ aligned crops ชุดเดียวกัน; Barbershop รับ PNG quantized จาก crops ของ HFG
- จับเวลาแยก GAN / mask / diffusion / background / native_sharp / รวม และประเมิน MUSIQ, ArcFace, สีผม, shape preservation, protected drift ทุกขั้น
- รายงาน delta แยก diffusion, background_finish, native_sharp, total เพื่อไม่เอาผลเพิ่มความคมไปอ้างว่าเป็นผลของ diffusion
- ส่วน 19–20 สร้างภาพ blind A/B รวม 6 คู่ และรับคะแนนจริง; ไม่สร้างคะแนนคนอัตโนมัติ

## วิธีรัน
ครั้งแรกใช้ Colab GPU รันส่วน 1–17 และ 19 ตามลำดับ ส่วน 20 ใช้เมื่อได้คำตอบผู้ประเมิน
ชุดถัดไปใช้ส่วน 18 ตั้ง RUN_NEW_IMAGES=True รันครั้งเดียว ไม่ต้อง Restart runtime
เปลี่ยนจากรุ่นก่อนที่มี RESULTS อยู่ใน session: นำส่วน 9 ของไฟล์นี้ไปนิยามฟังก์ชันใหม่ แล้วใช้ส่วน 21 เพื่อปรับผล raw เดิมโดยไม่สร้าง GAN/diffusion ใหม่ จากนั้นประเมินใหม่ได้

ตรวจ syntax, source parity และขั้นควบคุมด้วยภาพจำลอง; ยังไม่ได้รัน inference GPU หรือ MUSIQ weights จริงในสภาพแวดล้อมที่สร้างไฟล์นี้


## 1. ติดตั้งและตรวจ GPU

In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub requests
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('เลือก Runtime > Change runtime type > GPU แล้วรันใหม่')
from PIL import Image
if not hasattr(Image, 'ANTIALIAS'): Image.ANTIALIAS=Image.Resampling.LANCZOS
print('GPU:', torch.cuda.get_device_name(0))


## 2. เตรียม HairFastGAN และพื้นที่ผลลัพธ์

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time
from huggingface_hub import snapshot_download

drive.mount('/content/drive')
LOCAL_REPO = '/content/HairFastGAN'
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')
RESULTS_ROOT = Path('/content/drive/MyDrive/Barbershop_Baseline_vs_HairFastGAN_B24_Diffusion')
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
# ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
if not Path(LOCAL_REPO, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
if HAIRFAST_REVISION:
    subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
os.chdir(LOCAL_REPO)
local_models = Path(LOCAL_REPO, 'pretrained_models')
if not local_models.exists() or not any(local_models.iterdir()):
    if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
        shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
    else:
        snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
        shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)
        shutil.copytree(local_models, DRIVE_CACHE_DIR, dirs_exist_ok=True)
# ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
    p = local_models / relative
    if not p.exists() or p.stat().st_size < 1000000:
        raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', REPO_COMMIT)


## 3. Native torch backend — ใช้โค้ดเดิม

In [ ]:
# Compatibility backend: native torch operators also execute on CUDA tensors.
# No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
import ast, hashlib, importlib, sys
from pathlib import Path
STYLEGAN_OPS_BACKEND = 'torch_native'

def _native_stylegan_source(source, kind):
    marker = '# HAIRFAST_TORCH_NATIVE_V1'
    if marker in source:
        return source
    tree = ast.parse(source)
    extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
    entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
    edits = []
    found_load = found_entry = False
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
            if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
            edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
            found_load = True
        if isinstance(node, ast.FunctionDef) and node.name == entry:
            edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
            found_entry = True
    if not found_load or not found_entry:
        raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
    lines = source.splitlines(True)
    for start,end,text in sorted(edits, reverse=True):
        lines[start:end] = [text]
    updated = marker + '\n' + ''.join(lines)
    if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
        updated += '\n\n' + UP_NATIVE_IMPL + '\n'
    compile(updated, kind + '.py', 'exec')
    return updated

FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'

def install_native_stylegan_ops(repo):
    root = Path(repo).resolve()
    pairs = sorted((root / 'models').rglob('op/fused_act.py'))
    if not pairs:
        raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
    planned = []
    for fused_path in pairs:
        for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
            if not path.is_file():
                raise RuntimeError('ขาด operator: ' + str(path))
            module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
            if module_name in sys.modules:
                raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
            old = path.read_text()
            planned.append((path, old, _native_stylegan_source(old,kind)))
    # Validate every candidate before writing; preserve each original once.
    records = []
    for path,old,new in planned:
        if old != new:
            backup = path.with_suffix('.py.before_native')
            if not backup.exists(): backup.write_text(old)
            path.write_text(new)
        records.append({'path':str(path.relative_to(root)),
                        'sha256':hashlib.sha256(new.encode()).hexdigest()})
    importlib.invalidate_caches()
    return records

STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))

# Small on-device smoke check before loading model weights.
import torch
_check = {'torch':torch, 'F':torch.nn.functional}
exec(FUSED_NATIVE + '\n' + UP_NATIVE_IMPL + '\n' + UP_NATIVE, _check)
_device = 'cuda' if torch.cuda.is_available() else 'cpu'
_x = torch.randn(1, 2, 5, 5, device=_device, requires_grad=True)
_b = torch.randn(2, device=_device, requires_grad=True)
_y = _check['fused_leaky_relu'](_x, _b, .1)
torch.testing.assert_close(_y, torch.nn.functional.leaky_relu(_x + _b.view(1,2,1,1), .1) * (2 ** .5))
_z = _check['upfirdn2d'](_x, torch.ones(1,1,device=_device))
torch.testing.assert_close(_z, _x)
(_y.sum() + _z.sum()).backward()
assert torch.isfinite(_x.grad).all() and torch.isfinite(_b.grad).all()
print('Native operator smoke check passed on', _device)
del _check, _x, _b, _y, _z


## 4. โหลด HairFastGAN B24 — checkpoint เดิมที่ตรวจ hash แล้ว
หากหา checkpoint ไม่พบ ให้แก้ `CHECKPOINT` เป็นตำแหน่งไฟล์ B24 เดิม ห้ามเปลี่ยนเป็น weights อื่นเพื่อข้ามการตรวจ hash

In [ ]:
import sys, torch
sys.path.insert(0, LOCAL_REPO)
from hair_swap import HairFast, get_parser
hair_fast = HairFast(get_parser().parse_args([]))
print('HairFastGAN พร้อม')

import hashlib, inspect, textwrap, types
CHECKPOINT = Path('/content/drive/MyDrive/HairFastGAN_BPP_4Trials_PretrainedStart/9be83b9c2e/trial_B/best.pt')
EXPECTED_SHA256 = '6195edd247917e7c34887945413e08b731faae736f5239dd2b28d9636e65f98b'
def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
if not CHECKPOINT.is_file():
    raise FileNotFoundError('ไม่พบ B best.pt ใน Drive: '+str(CHECKPOINT))
if sha_file(CHECKPOINT)!=EXPECTED_SHA256:raise RuntimeError('ไฟล์ไม่ใช่ B epoch 24 ที่ตรวจไว้')
ck=torch.load(CHECKPOINT,map_location='cpu',weights_only=True)
assert ck['epoch']==24 and ck['config']['trial']=='B'
b=hair_fast.blend
assert sha_file(b.opts.blending_checkpoint)==ck['start_blending_sha256']
enc=b.blending_encoder
expected={k for k,_ in enc.named_parameters() if k.startswith('modulation_module_list.')}
assert set(ck['blending_trainable_state_dict'])==expected
state=enc.state_dict();state.update(ck['blending_trainable_state_dict'])
enc.load_state_dict(state,strict=True)
b.post_process.load_state_dict(ck['post_process_state_dict'],strict=True)
enc.eval();b.post_process.eval()
del state,ck
# Match the training cache: mixing .95; shape rotation OFF; color rotation ON.
for obj in (hair_fast.args,hair_fast.embed.opts,hair_fast.align.opts,b.opts):
    obj.mixing=.95;obj.save_all=False
method=hair_fast.align.shape_module
raw=textwrap.dedent(inspect.getsource(method))
raw='\n'.join(line for line in raw.splitlines() if not line.startswith('@'))
anchor='    # Rotate stage\n    if img1_in is not img2_in:'
replacement=('    # Shape OFF, color ON, same policy as training\n'
             '    if im_name2 == "shape":\n'
             '        I_rot = None\n        rot_mask = inp_mask2\n'
             '    elif img1_in is not img2_in:')
assert raw.count(anchor)==1,'Alignment source ไม่ตรงรุ่น'
ns = dict(inspect.unwrap(method.__func__).__globals__)
exec(compile(raw.replace(anchor,replacement),'<shape_rotation_policy>','exec'),ns)
hair_fast.align.shape_module=types.MethodType(torch.inference_mode()(ns['shape_module']),hair_fast.align)
print('โหลด B epoch 24 ครบ: Blending + Post-process | mix .95 | shape rotate OFF / color ON')

# Per-parameter/per-buffer placement: preserve mixed CPU/CUDA modules.
def hfg_modules(root):
    import types
    seen=set(); modules=[]
    def walk(obj):
        if id(obj) in seen: return
        seen.add(id(obj))
        if isinstance(obj,torch.nn.Module):
            modules.append(obj)
            for child in obj.children(): walk(child)
        elif isinstance(obj,dict):
            for value in obj.values(): walk(value)
        elif isinstance(obj,(list,tuple)):
            for value in obj: walk(value)
        elif hasattr(obj,'__dict__') and not isinstance(obj,(type,types.ModuleType,types.FunctionType,types.MethodType)):
            for value in vars(obj).values(): walk(value)
    walk(root)
    return modules

def capture_hfg_devices(root):
    return [(module,kind,name,str(value.device))
        for module in hfg_modules(root)
        for kind in ('_parameters','_buffers')
        for name,value in getattr(module,kind).items() if value is not None]

def restore_hfg_devices(layout):
    # Restore original slots, not module.to('cuda'), which would move CPU-only buffers too.
    buffer_cache={}
    with torch.no_grad():
        for module,kind,name,device in layout:
            value=getattr(module,kind)[name]
            if value is None: raise RuntimeError('HFG tensor disappeared: '+name)
            if kind=='_parameters':
                if str(value.device)!=device: value.data=value.data.to(device)
                if value.grad is not None and str(value.grad.device)!=device:
                    value.grad.data=value.grad.data.to(device)
            elif str(value.device)!=device:
                key=(id(value),device)
                if key not in buffer_cache: buffer_cache[key]=value.to(device)
                module._buffers[name]=buffer_cache[key]
    assert all(str(getattr(m,k)[name].device)==dev for m,k,name,dev in layout)

def park_postprocess():
    # Remove Accelerate hooks before parking; re-enable them in section 8.
    pipe=globals().get('inpaint_pipe')
    if pipe is not None:
        from accelerate.hooks import remove_hook_from_module
        for component in pipe.components.values():
            if isinstance(component,torch.nn.Module):
                remove_hook_from_module(component,recurse=True)
                component.cpu()
        if hasattr(pipe,'_all_hooks'): pipe._all_hooks=[]
        globals()['_DIFFUSION_PARKED']=True
    if 'seg_model' in globals(): seg_model.cpu()
    import gc
    gc.collect();torch.cuda.empty_cache()

_HFG_DEVICE_LAYOUT=capture_hfg_devices(hair_fast)
_HFG_LAYOUT_OWNER=id(hair_fast)
_HFG_OFFLOADED=[]


## 5. เปลี่ยนรูปและเริ่มชุดใหม่ใน runtime เดิม
ครั้งแรก: รันส่วน 1–17 ตามลำดับ
ชุดถัดไป: รันส่วน **5 → 17** ใหม่ อัปโหลดสามรูปใหม่ได้เลย ไม่ต้องรันส่วน 1–4 และไม่ต้อง Restart runtime
แต่ละชุดมีโฟลเดอร์และชื่อไฟล์ใหม่; คงโมเดลไว้ใน RAM สลับเข้า GPU ตามขั้นตอน
เวลา restore/offload ไม่รวมอยู่ในเวลา inference; Barbershop ยังโหลด worker และ optimize รูปใหม่ตามปกติ
อย่ารันส่วนอื่นของชุดใหม่ก่อนส่วน 5 และอย่ารันเซลล์พร้อมกัน

In [ ]:
import io, uuid, gc
import numpy as np
from PIL import Image, ImageOps
from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import files
from IPython.display import display

INPUT_PATHS = []  # ["/content/drive/.../face.jpg", ".../shape.jpg", ".../color.jpg"]
SEED = 3407
STRENGTH = 0.15
STEPS = 30
GUIDANCE = 2.5
FACE_BLEND = 0.35
COMPARE_CROP = None  # (left, top, right, bottom); None = เลือกจาก union hair masks
AUTO_DOWNLOAD_IMAGES = False  # True = ดาวน์โหลด 6 รูป; browser ต้องอนุญาตหลายไฟล์
if globals().get('_HFG_LAYOUT_OWNER') != id(hair_fast) or not globals().get('_HFG_DEVICE_LAYOUT'):
    raise RuntimeError('ใช้ส่วน 4 ของไฟล์รุ่นใหม่โหลด B24 ก่อน เพื่อบันทึก device layout ก่อน offload')
park_postprocess()
restore_hfg_devices(_HFG_DEVICE_LAYOUT)
# Invalidate only per-case outputs in memory; prior Drive folders stay intact.
for _name in ['RESULTS','EXPORT_IMAGES','TIMINGS','METRIC_FRAME','DELTA_FRAME','TIME_SUMMARY','TIME_DETAIL']:
    globals().pop(_name,None)
if not 0 <= int(SEED) <= 4294967295: raise ValueError('seed ไม่ถูกต้อง')
RUN_ID=datetime.now(ZoneInfo('Asia/Bangkok')).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
RUN_DIR=RESULTS_ROOT/RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
INPUT_DIR=RUN_DIR/'inputs_original'; INPUT_DIR.mkdir()
CROP_DIR=RUN_DIR/'inputs_aligned'; CROP_DIR.mkdir()
BASE_DIR=RUN_DIR/'base_outputs'; BASE_DIR.mkdir()
if INPUT_PATHS and len(INPUT_PATHS)!=3: raise ValueError('ต้องระบุ 3 paths ตามลำดับ หน้า/ทรง/สี')
INPUTS=[]
for i,(role,label) in enumerate([('face','รูปหน้า'),('shape','รูปทรงผม'),('color','รูปสีผม')]):
    if INPUT_PATHS:
        with Image.open(INPUT_PATHS[i]) as im: normalized=ImageOps.exif_transpose(im).convert('RGB')
    else:
        print('อัปโหลด '+label+' เพียง 1 ภาพ')
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError('กรุณาเลือกครั้งละ 1 ภาพ')
        with Image.open(io.BytesIO(next(iter(uploaded.values())))) as im:
            normalized=ImageOps.exif_transpose(im).convert('RGB')
        del uploaded
    path=INPUT_DIR/(role+'.png'); normalized.save(path); INPUTS.append(path)

def tensor_to_pil(value, value_range='zero_one'):
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if torch.is_tensor(value):
        t = value.detach().cpu()
        if t.ndim == 4 and t.shape[0] == 1:
            t = t[0]
        if t.ndim != 3 or t.shape[0] != 3:
            raise ValueError(f'Unexpected tensor shape: {tuple(t.shape)}')
        a = t.permute(1, 2, 0).numpy()
    else:
        a = np.asarray(value)
    if a.dtype == np.uint8:
        return Image.fromarray(a).convert('RGB')
    if value_range == 'minus_one_one':
        a = (a + 1) / 2
    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:
        raise ValueError('Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ')
    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert('RGB')

def clock():
    torch.cuda.synchronize()
    return time.perf_counter()
# Device restore is done above, before timing the swap.
_HFG_OFFLOADED=[]
t=clock()
with torch.inference_mode():
    swapped=hair_fast.swap(*(str(p) for p in INPUTS),align=True,seed=int(SEED))
HFG_SECONDS=clock()-t
if not isinstance(swapped,tuple) or len(swapped)!=4: raise RuntimeError('swap return format ไม่ตรงรุ่น')
tensor_to_pil(swapped[0]).save(BASE_DIR/'HairFastGAN_B24.png')
for role,value in zip(('face','shape','color'),swapped[1:]):
    image=tensor_to_pil(value)
    if image.size!=(1024,1024): raise ValueError('Aligned crop ต้องเป็น 1024x1024')
    image.save(CROP_DIR/(role+'.png'))
del swapped, value
# ย้ายโมเดล HFG ลง CPU ก่อนเปิด subprocess Barbershop ป้องกัน VRAM ซ้อน
# เดินเฉพาะ object tree ของ HFG; nn.Module ย้ายลูกทั้งหมดด้วย .cpu()
def offload_hfg(root):
    import types
    seen=set(); moved=[]
    def walk(obj):
        if id(obj) in seen: return
        seen.add(id(obj))
        if isinstance(obj,torch.nn.Module):
            # CPU + CUDA in one module is valid. .cpu() moves all registered children.
            obj.cpu()
            moved.append(obj)
        elif isinstance(obj,dict):
            for value in obj.values(): walk(value)
        elif isinstance(obj,(list,tuple)):
            for value in obj: walk(value)
        elif hasattr(obj,'__dict__') and not isinstance(obj,(type,types.ModuleType,types.FunctionType,types.MethodType)):
            for value in vars(obj).values(): walk(value)
    walk(root)
    return moved
_HFG_OFFLOADED=offload_hfg(hair_fast)
gc.collect();torch.cuda.empty_cache()
print('HFG seconds:',round(HFG_SECONDS,2),'| offloaded modules:',len(_HFG_OFFLOADED))
print('ผลชุดนี้:',RUN_DIR)
display(Image.open(BASE_DIR/'HairFastGAN_B24.png'))


## 6. เตรียม Barbershop Baseline ตาม revision และ weights ในรายงาน
ตรวจ hash ของ ffhq.pt และ seg.pth ตรงกับรายงานเดิม ไม่มีการโหลด checkpoint ที่ fine-tune

In [ ]:
# Barbershop stages use the SAME Python/Torch/CUDA runtime as HairFastGAN via subprocess isolation.
BARBER_REV='088d7d0f0623b348732fc15d8fbdea8b8cb730c7'
BARBER_REPO=Path('/content/Barbershop_baseline_comparison')
BARBER_ROOT=RUN_DIR/'barbershop_frontend';BARBER_ROOT.mkdir(exist_ok=True)
sha=sha_file
BARBER_WEIGHTS=Path('/content/drive/MyDrive/Barbershop_matched_weights');BARBER_WEIGHTS.mkdir(exist_ok=True)
if not (BARBER_REPO/'.git').exists():subprocess.run(['git','clone','https://github.com/ZPdesu/Barbershop.git',str(BARBER_REPO)],check=True)
# No destructive checkout of unrelated work: this is a dedicated experiment checkout.
head=subprocess.check_output(['git','-C',str(BARBER_REPO),'rev-parse','HEAD'],text=True).strip()
if head!=BARBER_REV:
    assert not subprocess.check_output(['git','-C',str(BARBER_REPO),'status','--porcelain'],text=True).strip(), 'Dedicated checkout has modifications at another commit'
    subprocess.run(['git','-C',str(BARBER_REPO),'checkout',BARBER_REV],check=True)
import gdown, ast, platform
# Download once; use the SAME source weights through a local/Drive path or manual upload on failure.
# Optional: fill paths to the original Barbershop weights from your earlier notebook.
LOCAL_WEIGHT_PATHS = {'ffhq.pt': '', 'seg.pth': ''}
WEIGHT_IDS = {'ffhq.pt': '1NVlMTYO4wyXZYVGVlGjP_xoCzBlfXSR1',
              'seg.pth': '1bDOOJpST8dJDcSPuPKpTbdAmLKUGUji5'}

def check_barber_weight(path, name):
    if not path.is_file() or path.stat().st_size < 1000000:
        raise ValueError('Missing/incomplete weight file: '+str(path))
    # Only open original weights from the sources above or your trusted existing Barbershop copy.
    obj=torch.load(path,map_location='cpu',weights_only=False)
    if name=='ffhq.pt':
        if not isinstance(obj,dict) or not {'g_ema','latent_avg'}<=set(obj):
            raise ValueError('Not a Barbershop ffhq.pt checkpoint (requires g_ema and latent_avg)')
        if not isinstance(obj['g_ema'],dict) or not torch.is_tensor(obj['latent_avg']):
            raise ValueError('Invalid generator checkpoint structure')
    elif not isinstance(obj,dict) or not any(torch.is_tensor(v) for v in obj.values()):
        raise ValueError('Expected original Barbershop segmentation state_dict')
    del obj

for name,drive_id in WEIGHT_IDS.items():
    target=BARBER_WEIGHTS/name
    if target.is_file():
        check_barber_weight(target,name)
        print('Using cached weight:',target)
        continue
    temp=BARBER_WEIGHTS/(name+'.download')
    supplied=LOCAL_WEIGHT_PATHS[name].strip()
    if supplied:
        source=Path(supplied).expanduser()
        check_barber_weight(source,name)
        shutil.copy2(source,temp)
    else:
        try:
            result=gdown.download(id=drive_id,output=str(temp),quiet=False)
            if result is None:raise RuntimeError('gdown returned no file')
            check_barber_weight(temp,name)
        except Exception as err:
            print('Automatic download unavailable for',name,':',type(err).__name__)
            print('Original source: https://drive.google.com/file/d/'+drive_id+'/view')
            print('Use this original file or your existing Barbershop copy. Do not substitute another model checkpoint.')
            supplied=input('Paste the full /content/drive/... path to '+name+'; or press Enter to upload it: ').strip().strip('"')
            if supplied:
                source=Path(supplied).expanduser();check_barber_weight(source,name)
                shutil.copy2(source,temp)
            else:
                from google.colab import files as colab_files
                # Upload into a dedicated folder; avoid overwriting unrelated notebook files.
                import tempfile
                with tempfile.TemporaryDirectory(prefix='barber_weight_') as upload_dir:
                    before=os.getcwd()
                    try:
                        os.chdir(upload_dir);uploaded_weights=colab_files.upload()
                    finally:os.chdir(before)
                    if len(uploaded_weights)!=1:raise ValueError('Upload only '+name+' in this dialog')
                    source=Path(upload_dir)/next(iter(uploaded_weights))
                    check_barber_weight(source,name);shutil.copy2(source,temp)
                    del uploaded_weights
    check_barber_weight(temp,name)
    os.replace(temp,target)
    print('Saved validated weight:',target)
# Record byte identities; structural validation alone does not establish source provenance.
weight_hashes={name:sha(BARBER_WEIGHTS/name) for name in WEIGHT_IDS}
weight_lock=BARBER_WEIGHTS/'weight_hashes.json'
if weight_lock.exists():
    assert json.loads(weight_lock.read_text())==weight_hashes, 'Cached weights changed; do not mix checkpoints within an experiment'
else:weight_lock.write_text(json.dumps(weight_hashes,indent=2))

EXPECTED_BARBER_HASHES = {'ffhq.pt': '1a0c25c17013e0bb99bd3de4e80ea52cb3efc2b71224ff902e0a29bafac9c531', 'seg.pth': '590a79b3bea1f45e22d39e36f9dc80434075367c2e8d131ca3f5275e54dd3c00'}
if weight_hashes != EXPECTED_BARBER_HASHES:
    raise RuntimeError('Barbershop weights ไม่ตรงรายงานที่เลือก Baseline: ตรวจไฟล์เดิมใน Drive')

# Use the repository's fixed PCA; do not rebuild a new random PCA each trial.
pca=BARBER_WEIGHTS/'ffhq_PCA.npz'
if not pca.is_file():shutil.copy2(BARBER_REPO/'pretrained_models/ffhq_PCA.npz',pca)
assert sha(pca)==sha(BARBER_REPO/'pretrained_models/ffhq_PCA.npz')
weights=torch.load(BARBER_WEIGHTS/'ffhq.pt',map_location='cpu',weights_only=False)
assert {'g_ema','latent_avg'}<=set(weights);del weights

# Use the same native-PyTorch operator implementation used by HairFastGAN.
patches=[]
for kind in ['fused_act','upfirdn2d']:
    path=BARBER_REPO/'models/stylegan2/op'/(kind+'.py')
    original=subprocess.check_output(['git','-C',str(BARBER_REPO),'show',BARBER_REV+':models/stylegan2/op/'+kind+'.py'],text=True)
    patched=_native_stylegan_source(original,kind)
    compile(patched,str(path),'exec');path.write_text(patched)
    patches.append(dict(file=str(path.relative_to(BARBER_REPO)),sha256=sha(path)))

# Instrument existing optimizer steps without changing losses or latent update rules.
class LogOptimizerSteps(ast.NodeTransformer):
    def __init__(self,stage):self.stage=stage;self.counter=0
    def visit_Expr(self,node):
        self.generic_visit(node)
        call=node.value
        if isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute) and call.func.attr=='step' and isinstance(call.func.value,ast.Name) and call.func.value.id in ['optimizer_W','optimizer_FS','optimizer_align','opt_blend']:
            self.counter+=1
            log=ast.parse(f"record('{self.stage}_{self.counter}', locals().get('step'), locals().get('loss'))").body[0]
            return [node,ast.copy_location(log,node)]
        return node
for name,expected in [('Embedding',2),('Alignment',4),('Blending',1)]:
    rel='models/'+name+'.py';original=subprocess.check_output(['git','-C',str(BARBER_REPO),'show',BARBER_REV+':'+rel],text=True)
    transformer=LogOptimizerSteps(name);tree=transformer.visit(ast.parse(original));assert transformer.counter==expected,(name,transformer.counter)
    tree.body.insert(0,ast.ImportFrom(module='experiment_trace',names=[ast.alias(name='record')],level=0))
    ast.fix_missing_locations(tree);code=ast.unparse(tree);compile(code,rel,'exec');(BARBER_REPO/rel).write_text(code)
    patches.append(dict(file=rel,sha256=sha(BARBER_REPO/rel)))

(BARBER_REPO/'experiment_worker.py').write_text('"""Runs only inside the pinned Barbershop checkout in a separate Python process."""\nimport os,sys,json,time,random,hashlib,copy,gc,ast,argparse\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom PIL import Image\nif not hasattr(Image,\'ANTIALIAS\'):Image.ANTIALIAS=Image.Resampling.LANCZOS\ncfg=json.loads(Path(sys.argv[1]).read_text())\n# Read the original CLI defaults without executing its main pipeline.\nsource=Path(\'main.py\').read_text();tree=ast.parse(source)\nblock=next(n for n in tree.body if isinstance(n,ast.If) and \'__name__\' in ast.unparse(n.test))\nparser_nodes=[]\nfor node in block.body:\n    if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'args\' for t in node.targets):break\n    parser_nodes.append(node)\nns={\'argparse\':argparse};exec(compile(ast.Module(body=parser_nodes,type_ignores=[]),\'defaults\',\'exec\'),ns)\nopts=ns[\'parser\'].parse_args([])\nopts.ckpt=cfg[\'ckpt\'];opts.seg_ckpt=cfg[\'seg\'];opts.seed=int(cfg.get(\'seed\',3407))\nopts.W_steps=1100;opts.FS_steps=250;opts.align_steps1=140;opts.align_steps2=100;opts.blend_steps=400\nopts.save_intermediate=False;opts.verbose=False;opts.sign=\'realistic\';opts.smooth=5\n# All torch.load calls are restricted to the trusted local model files and repository LPIPS weights.\n_original_load=torch.load\ndef legacy_load(*args,**kwargs):\n    kwargs.setdefault(\'weights_only\',False)\n    return _original_load(*args,**kwargs)\ntorch.load=legacy_load\nfrom models.Embedding import Embedding\nfrom models.Alignment import Alignment\nfrom models.Blending import Blending\nimport experiment_trace as trace\n\ndef seed(key):\n    n=(int(cfg.get(\'seed\',3407))+int(hashlib.sha256(key.encode()).hexdigest()[:8],16))%(2**31)\n    random.seed(n);np.random.seed(n);torch.manual_seed(n);torch.cuda.manual_seed_all(n)\n    torch.backends.cudnn.benchmark=False\n    torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False\n    torch.backends.cudnn.allow_tf32=False\n\ndef sha(p):\n    h=hashlib.sha256()\n    with open(p,\'rb\') as f:\n        for chunk in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(chunk)\n    return h.hexdigest()\ndef fingerprint(module):\n    h=hashlib.sha256()\n    for k,v in module.state_dict().items():h.update(k.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())\n    return h.hexdigest()\ndef atomic(obj,p):\n    p=Path(p);tmp=p.with_suffix(\'.tmp\');tmp.write_text(json.dumps(obj,indent=2));os.replace(tmp,p)\ndef seconds(fn):\n    torch.cuda.synchronize();tick=time.perf_counter();result=fn();torch.cuda.synchronize()\n    return time.perf_counter()-tick,result\n\nroot=Path(cfg[\'root\']);inv=root/\'inversion\';inv.mkdir(exist_ok=True)\ntrace.configure(root/\'optimization_steps.csv\')\nseed(\'initialization\')\nopts.learning_rate=.01;opts.output_dir=str(inv)\nembedding=Embedding(copy.deepcopy(opts))\n# Use fixed pretrained noise in both frozen preprocessing and subsequent shared-weight decoding.\nimport functools\nembedding.net.generator.forward=functools.partial(embedding.net.generator.forward,randomize_noise=False)\ninitial=fingerprint(embedding.net.generator)\n# Reuse the exact same generator for all modules; these weights remain frozen.\naligner=Alignment(copy.deepcopy(opts),net=embedding.net)\nblender=Blending(copy.deepcopy(opts),net=embedding.net)\nfor module in [embedding.loss_builder,aligner.loss_builder,blender.loss_builder]:\n    module.eval().requires_grad_(False)\n# No GPU training of shared network weights occurs.\natomic(dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),gpu=torch.cuda.get_device_name(0),defaults=vars(opts),weights_sha256=sha(cfg[\'ckpt\']),seg_sha256=sha(cfg[\'seg\']),pca_sha256=sha(cfg[\'pca\'])),root/\'worker_environment.json\')\nids=sorted({r[k] for r in cfg[\'cases\'] for k in [\'face\',\'shape\',\'color\']})\nfor ident in ids:\n    marker=inv/(ident+\'_complete.json\');image=str(Path(cfg[\'input_dir\'])/(ident+\'.png\'))\n    if marker.exists():\n        previous=json.loads(marker.read_text());assert previous[\'input_sha256\']==sha(image)\n        continue\n    seed(\'inversion/\'+ident);trace.context(model=\'shared_inversion\',case=ident,operation=\'inversion\')\n    wsec,_=seconds(lambda:embedding.invert_images_in_W([image]))\n    fsec,_=seconds(lambda:embedding.invert_images_in_FS([image]))\n    trace.flush()\n    atomic(dict(input_sha256=sha(image),W_seconds=wsec,FS_seconds=fsec,total_seconds=wsec+fsec,W_steps=opts.W_steps,FS_steps=opts.FS_steps),marker)\n\n# ALIGNMENT_SAVE_GUARD_V1: preserve exact computed latents, never substitute another pair.\n_alignment_latents={}\n_original_save_align=aligner.save_align_results\ndef checked_save_align(im_name_1,im_name_2,sign,gen_im,latent_in,latent_F,save_intermediate=True):\n    _original_save_align(im_name_1,im_name_2,sign,gen_im,latent_in,latent_F,save_intermediate=save_intermediate)\n    path=Path(aligner.opts.output_dir)/(\'Align_\'+sign)/(im_name_1+\'_\'+im_name_2+\'.npz\')\n    payload=dict(latent_in=latent_in.detach().cpu().numpy().copy(),latent_F=latent_F.detach().cpu().numpy().copy())\n    _alignment_latents[str(path)]=payload\n    write_checked_alignment(path,payload)\ndef write_checked_alignment(path,payload):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    if not all(np.isfinite(v).all() for v in payload.values()):raise RuntimeError(\'Nonfinite alignment: \'+str(path))\n    tmp=path.with_name(path.name+\'.pending\')\n    with tmp.open(\'wb\') as stream:\n        np.savez(stream,**payload);stream.flush();os.fsync(stream.fileno())\n    os.replace(tmp,path)\n    with np.load(path,allow_pickle=False) as saved:\n        if set(saved.files)!=set(payload) or any(not np.array_equal(saved[k],v) for k,v in payload.items()):\n            raise RuntimeError(\'Alignment write verification failed: \'+str(path))\n    print(\'Verified aligned latent:\',path,flush=True)\naligner.save_align_results=checked_save_align\n\ndef ensure_alignment_pair(path):\n    path=Path(path)\n    if str(path) not in _alignment_latents:raise RuntimeError(\'Alignment did not produce expected pair: \'+str(path))\n    # Re-persist both exact computed pairs before blending; avoids stale/missing Drive entries.\n    write_checked_alignment(path,_alignment_latents[str(path)])\n\nrows=[]\nfor spec in cfg[\'trials\']:\n    for row in cfg[\'cases\']:\n        key=row[\'key\'];dest=root/spec[\'name\']/key;dest.mkdir(parents=True,exist_ok=True)\n        result_path=dest/\'complete.json\'\n        if result_path.exists():\n            old=json.loads(result_path.read_text());assert old[\'spec\']==spec\n            assert Path(old[\'final_float\']).is_file() and Path(old[\'final_png\']).is_file(), \'Completed result files missing\'\n            rows.append(old);continue\n        # Shared inversion bytes are read-only and linked into each isolated case directory.\n        for sub in [\'W+\',\'FS\']:\n            link=dest/sub\n            if not link.exists():link.symlink_to(inv/sub,target_is_directory=True)\n        a=copy.deepcopy(opts);a.output_dir=str(dest);a.learning_rate=spec[\'align_lr\'];aligner.opts=a;aligner.loss_builder.opt=a\n        b=copy.deepcopy(opts);b.output_dir=str(dest);b.learning_rate=spec[\'blend_lr\'];blender.opts=b;blender.loss_builder.opt=b\n        f,s,c=[str(Path(cfg[\'input_dir\'])/(row[k]+\'.png\')) for k in [\'face\',\'shape\',\'color\']]\n        seed(\'case/\'+key) # same latent/noise initialization for all trials of this case\n        trace.context(model=spec[\'name\'],case=key,operation=\'align_shape\')\n        align1,_=seconds(lambda:aligner.align_images(f,s,sign=\'realistic\',align_more_region=False,smooth=5))\n        trace.context(model=spec[\'name\'],case=key,operation=\'align_color\')\n        align2=0.0\n        if s!=c:\n            align2,_=seconds(lambda:aligner.align_images(f,c,sign=\'realistic\',align_more_region=False,smooth=5,save_intermediate=False))\n        for ref in sorted({s,c}):\n            ensure_alignment_pair(dest/\'Align_realistic\'/(Path(f).stem+\'_\'+Path(ref).stem+\'.npz\'))\n        trace.context(model=spec[\'name\'],case=key,operation=\'blend\')\n        # Capture the float output before PNG quantization for the common evaluator.\n        capture={};original_save=blender.save_blend_results\n        def save_float(*args,**kwargs):\n            capture[\'full\']=((args[4].detach().float().cpu()+1)/2).clamp(0,1).numpy()\n            return original_save(*args,**kwargs)\n        blender.save_blend_results=save_float\n        try:blendsec,_=seconds(lambda:blender.blend_images(f,s,c,sign=\'realistic\'))\n        finally:blender.save_blend_results=original_save\n        np.save(dest/\'final_float.npy\',capture[\'full\']);del capture\n        from PIL import Image\n        png=dest/f\'{Path(f).stem}_{Path(s).stem}_{Path(c).stem}_realistic.png\'\n        assert png.is_file()\n        inv_seconds=sum(json.loads((inv/(ident+\'_complete.json\')).read_text())[\'total_seconds\'] for ident in sorted({row[k] for k in [\'face\',\'shape\',\'color\']}))\n        result=dict(case=key,split=cfg[\'split\'],model=spec[\'name\'],spec=spec,align_shape_seconds=align1,align_color_seconds=align2,blend_seconds=blendsec,inversion_cold_seconds=inv_seconds,cached_pipeline_seconds=align1+align2+blendsec,estimated_cold_compute_seconds=inv_seconds+align1+align2+blendsec,final_float=str(dest/\'final_float.npy\'),final_png=str(png))\n        trace.flush()\n        atomic(result,result_path);rows.append(result)\n        atomic(rows,root/\'results.json\');gc.collect();torch.cuda.empty_cache()\nassert fingerprint(embedding.net.generator)==initial,\'Frozen generator changed\'\natomic(rows,root/\'results.json\')\natomic(dict(status=\'complete\',frozen_generator_unchanged=True,cases=len(cfg[\'cases\']),trials=len(cfg[\'trials\']),rows=len(rows)),root/\'completion.json\')\n')
(BARBER_REPO/'experiment_trace.py').write_text("import csv,torch,time\n_handle=None\n_writer=None\n_context={}\ndef configure(path):\n    global _handle,_writer\n    fields=['model','case','operation','stage','step','loss']\n    # Keep only completed work when retrying; partial cases restart from the shared inversion.\n    if path.exists():\n        with open(path,newline='') as src:rows=list(csv.DictReader(src))\n        def complete(r):\n            marker=(path.parent/'inversion'/(r['case']+'_complete.json')) if r['model']=='shared_inversion' else (path.parent/r['model']/r['case']/'complete.json')\n            return marker.is_file()\n        temp=path.with_suffix('.tmp')\n        with open(temp,'w',newline='') as dst:\n            writer=csv.DictWriter(dst,fieldnames=fields);writer.writeheader();writer.writerows(r for r in rows if complete(r))\n        temp.replace(path)\n    exists=path.exists() and path.stat().st_size>0\n    _handle=open(path,'a',newline='')\n    _writer=csv.DictWriter(_handle,fieldnames=fields)\n    if not exists:_writer.writeheader()\ndef flush():\n    if _handle:_handle.flush()\ndef context(**kwargs):\n    global _context\n    _context=kwargs\n    flush()\ndef record(stage,step,loss):\n    if _writer is None:return\n    value=float(loss.detach()) if torch.is_tensor(loss) else float(loss)\n    if not __import__('math').isfinite(value):raise RuntimeError('Nonfinite optimization loss')\n    _writer.writerow(dict(**_context,stage=stage,step=step,loss=value))\n")

ALIGNMENT_GUARD_PREVIOUS_WORKER_SHA256='5baf0074bda54939722b5cc374e8a4b2233332ba283451c63a85eea50d2ce2f8'


## 7. สร้าง Barbershop Baseline จาก aligned crops ชุดเดียวกัน
ใช้ subprocess แยก import ของสอง repository; หากขั้นนี้ล้มเหลว รันเซลล์นี้ซ้ำได้โดยเก็บ inversion ที่เสร็จแล้ว ตรวจ config/hash ก่อนใช้ cache
แก้ alignment save guard: บันทึก latent ที่คำนวณจริงแบบ atomic ตรวจอ่านกลับและตรวจคู่ face/shape กับ face/color ก่อน blending ถ้าอัปเกรดจาก error นี้ ให้รันส่วน 6 และ 7 ใหม่ ใช้ inversion เดิมที่เสร็จแล้ว ไม่สร้าง input ใหม่

In [ ]:
BARBER_CASE=dict(key='comparison_000',face='face',shape='shape',color='color')
BARBER_CONFIG=dict(root=str(BARBER_ROOT),input_dir=str(CROP_DIR),
    ckpt=str(BARBER_WEIGHTS/'ffhq.pt'),seg=str(BARBER_WEIGHTS/'seg.pth'),
    pca=str(pca),trials=[dict(name='BS_Pretrained',align_lr=.01,blend_lr=.01)],
    cases=[BARBER_CASE],split='user_comparison',seed=int(SEED),
    crop_hashes={role:sha_file(CROP_DIR/(role+'.png')) for role in ['face','shape','color']},
    weight_hashes=weight_hashes,revision=BARBER_REV,
    worker_sha256=sha_file(BARBER_REPO/'experiment_worker.py'),patches=patches)
config_path=BARBER_ROOT/'worker_config.json'
if config_path.exists():
    previous=json.loads(config_path.read_text())
    if previous!=BARBER_CONFIG:
        old_fields={k:v for k,v in previous.items() if k!='worker_sha256'}
        new_fields={k:v for k,v in BARBER_CONFIG.items() if k!='worker_sha256'}
        compatible=(old_fields==new_fields and previous.get('worker_sha256')==ALIGNMENT_GUARD_PREVIOUS_WORKER_SHA256)
        if not compatible or (BARBER_ROOT/'completion.json').exists():
            raise RuntimeError('Config เปลี่ยนหรือเป็นผลที่เสร็จแล้ว ห้ามใช้ cache ปนกัน ให้เริ่มชุดใหม่')
        backup=config_path.with_name('worker_config.before_alignment_guard.json')
        if not backup.exists():backup.write_text(json.dumps(previous,indent=2))
        config_path.write_text(json.dumps(BARBER_CONFIG,indent=2))
        print('อัปเดตเฉพาะ alignment save guard; คง input/weights/settings และ inversion cache เดิม')
else: config_path.write_text(json.dumps(BARBER_CONFIG,indent=2))
t=clock()
with (BARBER_ROOT/'worker.log').open('a') as log:
    proc=subprocess.Popen([sys.executable,str(BARBER_REPO/'experiment_worker.py'),str(config_path)],
        cwd=BARBER_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,
        bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1'})
    for line in proc.stdout:
        log.write(line);log.flush();print(line,end='')
    returncode=proc.wait()
if returncode: raise RuntimeError('Barbershop ล้มเหลว ดู worker.log ด้านบน แล้วรันส่วน 7 ซ้ำ')
BARBER_WALL_SECONDS=clock()-t
completion=json.loads((BARBER_ROOT/'completion.json').read_text())
assert completion['rows']==1 and completion['frozen_generator_unchanged']
BARBER_RESULT=json.loads((BARBER_ROOT/'results.json').read_text())[0]
# ใช้ float output จาก worker แล้ว quantize แบบเดียวกับ HFG เพื่อไม่ต่างกันจาก PNG writer
raw_bs=np.load(BARBER_RESULT['final_float'])
if raw_bs.ndim!=4 or raw_bs.shape[:2]!=(1,3): raise ValueError('Barbershop tensor format ไม่ตรงรุ่น')
barber_image=tensor_to_pil(torch.from_numpy(raw_bs))
if barber_image.size!=(1024,1024): raise ValueError('Barbershop output ต้องเป็น 1024x1024')
barber_image.save(BASE_DIR/'Barbershop_Baseline.png')
del raw_bs,barber_image
print('Barbershop เสร็จ | wall time รอบนี้:',round(BARBER_WALL_SECONDS,2),'วินาที')
print('เวลาคำนวณรวม inversion ของผลนี้:',round(BARBER_RESULT['estimated_cold_compute_seconds'],2),'วินาที')
display(Image.open(BASE_DIR/'Barbershop_Baseline.png'))


## 8. โหลด segmentation และ diffusion — ตั้งค่าตามต้นฉบับ

In [ ]:
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageDraw
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from diffusers import StableDiffusionInpaintPipeline, UniPCMultistepScheduler

DEVICE = 'cuda'
SEG_MODEL_ID = 'mattmdjaga/segformer_b2_clothes'
DIFFUSION_MODEL_ID = 'stablediffusionapi/realistic-vision-v51'
SEG_REVISION = 'main'           # เปลี่ยนเป็น commit hash เพื่อทำซ้ำข้าม session
DIFFUSION_REVISION = 'main'
DIFFUSION_CPU_OFFLOAD = True
seg_key=(SEG_MODEL_ID,SEG_REVISION,DEVICE)
if globals().get('_SEG_LOADED_KEY') != seg_key or 'seg_model' not in globals():
    seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID, revision=SEG_REVISION)
    seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID, revision=SEG_REVISION).to(DEVICE).eval()
    _SEG_LOADED_KEY=seg_key
else:
    seg_model.to(DEVICE).eval()
    print('คืน segmentation จาก RAM ไป GPU แล้ว')
label_ids = {str(name).lower(): int(i) for i, name in seg_model.config.id2label.items()}
HAIR_ID, FACE_ID = label_ids['hair'], label_ids['face']
assert (HAIR_ID, FACE_ID) == (2, 11), 'ตรวจ mapping ของโมเดลก่อนทดลอง'
diffusion_key=(DIFFUSION_MODEL_ID,DIFFUSION_REVISION,DIFFUSION_CPU_OFFLOAD,DEVICE)
if globals().get('_DIFFUSION_LOADED_KEY') != diffusion_key or 'inpaint_pipe' not in globals():
    # # ไม่ปิด safety checker ที่โมเดลมีมา
    # inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    #     DIFFUSION_MODEL_ID, revision=DIFFUSION_REVISION, torch_dtype=torch.float16)

    # ปิด Safety Checker
    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        DIFFUSION_MODEL_ID,
        revision=DIFFUSION_REVISION,
        torch_dtype=torch.float16,
        safety_checker=None,
    )

    inpaint_pipe.scheduler = UniPCMultistepScheduler.from_config(inpaint_pipe.scheduler.config)
    if DIFFUSION_CPU_OFFLOAD:
        inpaint_pipe.enable_model_cpu_offload()
    else:
        inpaint_pipe.to(DEVICE)
    # รองรับทั้ง pipeline API เดิม และ VAE API; ถ้าไม่มีให้ข้าม optimization นี้
    vae = getattr(inpaint_pipe, 'vae', None)
    if callable(getattr(vae, 'enable_slicing', None)):
        vae.enable_slicing()
    elif callable(getattr(inpaint_pipe, 'enable_vae_slicing', None)):
        inpaint_pipe.enable_vae_slicing()
    else:
        print('VAE slicing ไม่รองรับในรุ่นนี้: ใช้งานต่อโดยไม่เปิด slicing')
    _DIFFUSION_LOADED_KEY=diffusion_key
else:
    if globals().get('_DIFFUSION_PARKED',False):
        if DIFFUSION_CPU_OFFLOAD: inpaint_pipe.enable_model_cpu_offload()
        else: inpaint_pipe.to(DEVICE)
    print('ใช้ diffusion pipeline ที่โหลดไว้แล้ว')
_DIFFUSION_PARKED=False
print('Labels:', HAIR_ID, FACE_ID, '| UNet channels:', inpaint_pipe.unet.config.in_channels)
# บันทึกรุ่นที่ใช้จริง ไม่เก็บ secrets
ENVIRONMENT = {
    'repo_commit': REPO_COMMIT, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda,
    'seg_model': SEG_MODEL_ID, 'seg_revision_requested': SEG_REVISION,
    'seg_revision_resolved': getattr(seg_model.config, '_commit_hash', None),
    'diffusion_model': DIFFUSION_MODEL_ID, 'diffusion_revision_requested': DIFFUSION_REVISION,
    'unet_channels': inpaint_pipe.unet.config.in_channels,
    'scheduler': type(inpaint_pipe.scheduler).__name__,
    'cpu_offload': DIFFUSION_CPU_OFFLOAD,
}
(RESULTS_ROOT / 'requirements.freeze.txt').write_text(
    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## 9. ฟังก์ชัน LOOKLAB Native Sharp จากไฟล์แนบ
ใช้ฟังก์ชันเดิมโดยตรง รวมการใช้ segmentation ของ raw ร่วมกันในขั้น background และ sharpen ปรับ NATIVE_HAIR_AMOUNT / NATIVE_FACE_AMOUNT ที่ท้ายเซลล์ฟังก์ชัน

In [ ]:
import io, threading, uuid, zipfile
from scipy.ndimage import binary_fill_holes, distance_transform_edt
GPU_LOCK=threading.Lock()
DEFAULT_SEED=3407
INPAINT_STRENGTH=.15
INPAINT_STEPS=30
INPAINT_GUIDANCE=2.5
JOINT_FACE_BLEND=0.35
BACKGROUND_BLEND=0.0  # พื้นหลังซ่อมแยก ไม่ผ่าน hair/face diffusion
BACKGROUND_REPAIR_MODE="auto"  # auto: สีเรียบเมื่อมั่นใจ มิฉะนั้น inpaint
BACKGROUND_REPAIR_STRENGTH=0.65
BACKGROUND_REPAIR_STEPS=40
BACKGROUND_REPAIR_FEATHER_PX=24.0
BACKGROUND_REPAIR_EXTRA_MASK=""  # optional: PNG mask ขนาดเท่า aligned image; white=repair
DIFFUSION_SIDE=1024
JOINT_TRANSITION_PX=24
JOINT_MARGIN_PX=12
JOINT_EYE_MARGIN_PX=3
JOINT_EYE_FEATHER_PX=6
_EYE_LANDMARK_MODEL=globals().get('_EYE_LANDMARK_MODEL')
_EYE_INIT_LOCK=globals().get('_EYE_INIT_LOCK',threading.Lock())
_DLIB_DETECTOR=globals().get('_DLIB_DETECTOR')
# Limited seam edits share the existing diffusion pass.
SEAM_WIDTH_PX=4.0
SEAM_BLEND=0.35
HAIR_DIFFUSION_BLEND=1.0
PROMPT = ('Natural portrait photograph, same person, haircut and hair color. '
          'Real hair with varied strand thickness, irregular fine strands, subtle flyaways, '
          'soft directional highlights and depth between locks. Natural skin texture, '
          'same eyes and gaze. Consistent lighting, natural hair edges against the existing background.')
NEGATIVE = ('wig, plastic hair, painted streaks, uniform parallel strands, flat clumps, '
            'cutout edges, bright halo, oversharpening, blur, waxy skin, '
            'changed face, changed haircut, changed hair color, new background objects')
for text in (PROMPT,NEGATIVE):
    assert len(inpaint_pipe.tokenizer(text,truncation=False)['input_ids']) <= inpaint_pipe.tokenizer.model_max_length,'Prompt ยาวเกิน tokenizer'
def segment_labels(image):
    inputs = seg_processor(images=image.convert('RGB'), return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        logits = seg_model(**inputs).logits
        logits = torch.nn.functional.interpolate(logits, size=image.size[::-1], mode='bilinear', align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)

def composite_protected(base, edited, alpha):
    if base.size != edited.size or np.asarray(alpha).shape != (base.height, base.width):
        raise ValueError('ภาพและ alpha ต้องมีขนาดตรงกัน')
    a = np.clip(np.asarray(alpha, dtype=np.float32), 0, 1)[..., None]
    original = np.array(base.convert('RGB'))
    changed = np.asarray(edited.convert('RGB'), dtype=np.float32)
    output = np.rint(original * (1 - a) + changed * a).clip(0, 255).astype(np.uint8)
    output[a[..., 0] == 0] = original[a[..., 0] == 0]
    return Image.fromarray(output)

def padded(image, target=512, mask=False):
    scale = target / max(image.size)
    size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
    resized = image.resize(size, Image.Resampling.NEAREST if mask else Image.Resampling.LANCZOS)
    canvas = Image.new(image.mode, (target, target), 0)
    x, y = (target-size[0])//2, (target-size[1])//2
    canvas.paste(resized, (x, y))
    return canvas, (x, y, x+size[0], y+size[1])

def tensor_to_pil(value, value_range='zero_one'):
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if torch.is_tensor(value):
        t = value.detach().cpu()
        if t.ndim == 4 and t.shape[0] == 1:
            t = t[0]
        if t.ndim != 3 or t.shape[0] != 3:
            raise ValueError(f'Unexpected tensor shape: {tuple(t.shape)}')
        a = t.permute(1, 2, 0).numpy()
    else:
        a = np.asarray(value)
    if a.dtype == np.uint8:
        return Image.fromarray(a).convert('RGB')
    if value_range == 'minus_one_one':
        a = (a + 1) / 2
    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:
        raise ValueError('Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ')
    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert('RGB')

def resolve_inpaint_settings(strength=None, steps=None, guidance=None):
    import math
    strength = float(INPAINT_STRENGTH if strength is None else strength)
    steps = INPAINT_STEPS if steps is None else steps
    guidance = float(INPAINT_GUIDANCE if guidance is None else guidance)
    if isinstance(steps, bool) or not isinstance(steps, (int, np.integer)) or not 1 <= steps <= 100:
        raise ValueError('steps ต้องเป็นจำนวนเต็ม 1–100')
    steps = int(steps)
    if not math.isfinite(strength) or not 0 < strength <= 1:
        raise ValueError('strength ต้องมากกว่า 0 และไม่เกิน 1')
    if not math.isfinite(guidance) or not 1 <= guidance <= 20:
        raise ValueError('guidance ต้องอยู่ในช่วง 1–20')
    if int(steps * strength) < 1:
        raise ValueError('steps × strength ต้องไม่น้อยกว่า 1')
    return {'strength':strength, 'steps':steps, 'guidance':guidance}

def resolve_joint_settings(face_blend=JOINT_FACE_BLEND):
    import math
    face_blend = float(face_blend)
    if not math.isfinite(face_blend) or not 0 <= face_blend <= 1:
        raise ValueError('face_blend ต้องอยู่ในช่วง 0–1')
    return {'face_blend':face_blend}

def prepare_eye_models():
    """One instance per runtime, including repeated execution of this cell."""
    global _EYE_LANDMARK_MODEL, _DLIB_DETECTOR
    import face_alignment, inspect, dlib, hashlib, shutil
    from pathlib import Path
    with _EYE_INIT_LOCK:
        if _EYE_LANDMARK_MODEL is not None:
            if _DLIB_DETECTOR is None:_DLIB_DETECTOR=dlib.get_frontal_face_detector()
            print('ใช้โมเดลตรวจหน้า–ตาที่อยู่ใน RAM แล้ว')
            return _EYE_LANDMARK_MODEL
        # Keep Torch's normal LOCAL cache for speed; mirror checkpoints to Drive.
        local=Path(torch.hub.get_dir())/'checkpoints'
        persistent=Path('/content/drive/MyDrive/HairFastGAN_cache/face_alignment')
        local.mkdir(parents=True,exist_ok=True)
        drive_ready=Path('/content/drive/MyDrive').is_dir()
        manifest_path=persistent/'manifest.json'
        if drive_ready:
            persistent.mkdir(parents=True,exist_ok=True)
            manifest=json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
            for name,entry in manifest.items():
                if Path(name).name!=name:continue
                src=persistent/name;dst=local/name
                if dst.exists() or not src.is_file():continue
                if src.stat().st_size!=entry['size']:continue
                digest=hashlib.sha256(src.read_bytes()).hexdigest()
                if digest!=entry['sha256']:continue
                tmp=dst.with_name(dst.name+'.copying');shutil.copyfile(src,tmp);tmp.replace(dst)
                print('นำ checkpoint จาก Drive มาใช้:',name)
        before={f.name for f in local.iterdir() if f.is_file()}
        kwargs=dict(device='cpu',flip_input=False)
        if 'compile' in inspect.signature(face_alignment.FaceAlignment).parameters:
            kwargs['compile']=False
        model=face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D,**kwargs)
        detector=dlib.get_frontal_face_detector()
        # Cache both known FAN/S3FD files and any new files downloaded by this version.
        if drive_ready:
            manifest=json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
            for src in local.iterdir():
                if not src.is_file():continue
                if src.name in before and not any(k in src.name.lower() for k in ('s3fd','2dfan','3dfan')):continue
                if not src.name.endswith(('.pth','.pth.tar','.pt')):continue
                entry=dict(size=src.stat().st_size,sha256=hashlib.sha256(src.read_bytes()).hexdigest())
                if manifest.get(src.name)!=entry or not (persistent/src.name).is_file():
                    tmp=persistent/(src.name+'.copying');shutil.copyfile(src,tmp);tmp.replace(persistent/src.name)
                manifest[src.name]=entry
            tmp=manifest_path.with_suffix('.tmp');tmp.write_text(json.dumps(manifest,indent=2));tmp.replace(manifest_path)
        _EYE_LANDMARK_MODEL=model;_DLIB_DETECTOR=detector
        print('โมเดลตรวจหน้า–ตาพร้อม ใช้ซ้ำทุกภาพใน runtime นี้')
        return model

def detect_eye_landmarks(base):
    """Inference only after setup; no repeated model construction."""
    if _EYE_LANDMARK_MODEL is None:
        raise RuntimeError('รันส่วน 6 ให้จบก่อน เพื่อเตรียมโมเดลตรวจหน้า–ตา')
    def validate(predictions, size):
        if predictions is None or len(predictions) != 1: return None
        p = np.asarray(predictions[0], dtype=np.float32).copy()
        if p.shape != (68, 2) or not np.isfinite(p).all(): return None
        p[:, 0] *= base.width / size[0]
        p[:, 1] *= base.height / size[1]
        eyes = p[36:48]
        if (eyes < 0).any() or (eyes[:,0] >= base.width).any() or (eyes[:,1] >= base.height).any(): return None
        if any(np.ptp(p[a:b,0]) < 2 for a,b in ((36,42),(42,48))): return None
        return p
    detector = _DLIB_DETECTOR
    with torch.inference_mode():
        predictions = _EYE_LANDMARK_MODEL.get_landmarks(np.asarray(base.convert('RGB')))
        p = validate(predictions, base.size)
        if p is not None: return p
        # Detection-only resizing/contrast. Never alter the image sent to diffusion.
        for side in (512, 768):
            scale = side / max(base.size)
            size = (max(1, round(base.width*scale)), max(1, round(base.height*scale)))
            im = base.convert('RGB').resize(size, Image.Resampling.LANCZOS)
            rgb = np.ascontiguousarray(np.asarray(im))
            for gray in (im.convert('L'), ImageOps.autocontrast(im.convert('L'))):
                rects = detector(np.ascontiguousarray(np.asarray(gray)), 1)
                if len(rects) != 1: continue
                r = rects[0]
                boxes = [[max(0,r.left()),max(0,r.top()),min(size[0]-1,r.right()),min(size[1]-1,r.bottom())]]
                predictions = _EYE_LANDMARK_MODEL.get_landmarks(rgb, detected_faces=boxes)
                p = validate(predictions, size)
                if p is not None:
                    print('Eye protection: ใช้ตัวตรวจสำรองสำเร็จ')
                    return p
    raise ValueError('ยังตรวจตำแหน่งตาที่เชื่อถือได้ไม่ได้ หยุดก่อน diffusion; ส่งภาพ hairfast_B24.png จากโฟลเดอร์ผลเพื่อตรวจต่อ')

def build_eye_protection(size, landmarks):
    """eye_keep=1 คืน A เต็มที่; 0 ใช้ Joint ตามเดิม; ขอบมีช่วงเปลี่ยนที่จำกัด."""
    from scipy.ndimage import distance_transform_edt
    w, h = size
    canvas = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(canvas)
    for eye in (landmarks[36:42], landmarks[42:48]):
        if np.ptp(eye[:, 0]) < 2 or (eye < 0).any() or (eye[:, 0] >= w).any() or (eye[:, 1] >= h).any():
            raise ValueError('Eye protection: ตำแหน่งตาอยู่นอกภาพหรือเล็กเกินไป')
        draw.polygon([tuple(map(float, p)) for p in eye], fill=255)
    scale = min(w, h) / 512.0
    margin = max(0, round(JOINT_EYE_MARGIN_PX * scale))
    if margin:
        canvas = canvas.filter(ImageFilter.MaxFilter(2 * margin + 1))
    core = np.asarray(canvas) > 0
    feather = max(1.0, JOINT_EYE_FEATHER_PX * scale)
    x = np.clip(distance_transform_edt(~core) / feather, 0, 1)
    keep = (1 - x*x*(3-2*x)).astype(np.float32)
    keep[core] = 1.0
    return {'eye_core': core, 'eye_keep': keep}


# ความกว้างรอยต่อ หน่วยพิกเซลที่ภาพ 1024; ปรับตามขนาดภาพอัตโนมัติ
RESTORE_FEATHER_PX = 16.0
HAIR_FACE_FEATHER_PX = 6.0
HAIR_BACKGROUND_FEATHER_PX = 4.0
LABEL_TRANSITION_PX = 8.0

def smootherstep(t):
    t=np.clip(t,0,1).astype(np.float32)
    return t*t*t*(t*(t*6-15)+10)

def outside_distance(mask):
    mask=np.asarray(mask,dtype=bool)
    if not mask.any():return np.full(mask.shape,np.inf,dtype=np.float32)
    return distance_transform_edt(~mask).astype(np.float32)

def outward_feather(mask,width):
    # น้ำหนัก 1 ใน mask แล้วค่อยลดลงถึง 0 ภายนอก ไม่มี Gaussian tail
    return 1-smootherstep(outside_distance(mask)/np.maximum(width,1.0))

def background_confidence(image):
    """Two views of the existing parser; no additional model downloads."""
    bg_id=label_ids['background']
    views=[];base_labels=None
    for flip in (False,True):
        view=ImageOps.mirror(image) if flip else image
        inputs=seg_processor(images=view.convert('RGB'),return_tensors='pt').to(DEVICE)
        with torch.inference_mode():
            logits=seg_model(**inputs).logits
            logits=torch.nn.functional.interpolate(logits,size=image.size[::-1],mode='bilinear',align_corners=False)
            if not flip:base_labels=logits.argmax(1)[0].cpu().numpy().astype(np.uint8)
            probs=logits.softmax(1)[0,[bg_id,HAIR_ID]].cpu().numpy()
        if flip:probs=probs[:,:,::-1].copy()
        views.append(probs)
    # Confidence is a heuristic, not a calibrated probability of correctness.
    bg=np.minimum(views[0][0],views[1][0])
    hair=np.maximum(views[0][1],views[1][1])
    return bg,hair,base_labels

def preservation_and_repair_masks(base,original,lo,lb,background_score,hair_score):
    from scipy.ndimage import binary_dilation
    scale=min(base.size)/1024.;bg_id=label_ids['background']
    old=lo==HAIR_ID
    # Separate liberal keep mask from narrow diffusion hair mask.
    hair_keep=(lb==HAIR_ID)|(hair_score>=.08)
    # New pixels on source background may be missed new hair: preserve them.
    delta=np.linalg.norm(np.asarray(base,dtype=np.float32)-np.asarray(original,dtype=np.float32),axis=2)
    uncertain_new=(lo==bg_id)&(delta>18)
    keep=hair_keep|uncertain_new|(lb!=bg_id)|(background_score<.98)
    guard=binary_dilation(keep,iterations=max(1,round(6*scale)))
    old_zone=binary_dilation(old,iterations=max(1,round(20*scale)))
    candidate=old_zone&(lb==bg_id)&(background_score>=.98)&~guard
    return guard,candidate

def prepare_face_base(base,original,need_landmarks=True):
    if base.size!=original.size:raise ValueError('aligned source กับ B24 ขนาดไม่ตรงกัน')
    lo=segment_labels(original)
    bg_score,hair_score,lb=background_confidence(base)
    old,new=lo==HAIR_ID,lb==HAIR_ID
    if not old.any() or not new.any() or not (lo==FACE_ID).any():
        raise ValueError('ตรวจผมหรือหน้าไม่ได้ โปรดตรวจ segmentation')
    # Full B24 base: no source face is pasted, so no source alignment detector needed.
    if need_landmarks:
        if _EYE_LANDMARK_MODEL is None:prepare_eye_models()
        lm1=detect_eye_landmarks(base)
    else:
        lm1=None
    scale=min(base.size)/1024.
    # Segmentation selects edits only, NEVER cuts the B24 silhouette.
    # Preserve every B24 pixel before diffusion, including missed hair strands.
    patch=np.ones((base.height,base.width),dtype=np.float32)
    restored=base.copy()
    keep_guard,repair_candidate=preservation_and_repair_masks(base,original,lo,lb,bg_score,hair_score)
    eyes=(build_eye_protection(base.size,lm1) if lm1 is not None else
          {'eye_core':np.zeros(lb.shape,bool),'eye_keep':np.zeros(lb.shape,np.float32)})
    # ลดการป้องกันอย่างต่อเนื่องเมื่อมีผมบัง ไม่ใช้ boolean ตัดขอบตา
    eye_keep=(eyes['eye_keep']*(1-patch)).astype(np.float32)
    visible_eyes=eyes['eye_core'] & (patch==0)
    return restored,dict(old_hair=old,hair=new,face=lb==FACE_ID,
        source_keep=patch==0,restore_alpha=patch,
        source_labels=lo,base_labels=lb,
        hair_preserve_guard=keep_guard,background_candidate=repair_candidate,
        background_score=bg_score,
        uncertain_new_hair=(lo==label_ids['background']) & (np.linalg.norm(np.asarray(base,dtype=np.float32)-np.asarray(original,dtype=np.float32),axis=2)>18),
        eye_core=visible_eyes,eye_keep=eye_keep,base_eye_masks=eyes,
        alignment_median_px=None,alignment_eye_max_px=None)

def single_masks(base,face_blend=JOINT_FACE_BLEND,restoration=None):
    if restoration is None:raise ValueError('ต้องมี aligned source restoration')
    face_blend=resolve_joint_settings(face_blend)['face_blend']
    if not 0<=BACKGROUND_BLEND<=1:raise ValueError('BACKGROUND_BLEND ต้องอยู่ในช่วง 0–1')
    r=restoration
    scale=min(base.size)/1024.
    target_labels=r['base_labels']
    if 'background' not in label_ids:raise ValueError('ไม่พบ background label')
    eyes=r['base_eye_masks']
    hair=r['hair']
    visible_eyes=eyes['eye_core'] & ~hair
    # Do not fill holes or union with source-face masks: bangs may cover them.
    face=((target_labels==FACE_ID) & r['face']) & ~hair
    background=(target_labels==label_ids['background']) & ~hair & ~face
    # Face/eye receive .35; hair core receives 1. Feather stays INSIDE the
    # person region so the already-good background is not regenerated.
    person=hair|face
    dist=distance_transform_edt(np.pad(person,1))[1:-1,1:-1]
    boundary=smootherstep(dist/max(1.,HAIR_BACKGROUND_FEATHER_PX*scale))
    hd=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]
    hair_weight=smootherstep(hd/max(1.,HAIR_FACE_FEATHER_PX*scale))
    alpha=(face_blend+(HAIR_DIFFUSION_BLEND-face_blend)*hair_weight)*boundary
    eye_weight=eyes['eye_keep']*(1-outward_feather(hair,max(1.,4*scale)))
    # Blend only within person; do not spill across background.
    alpha=(alpha*(1-eye_weight)+face_blend*eye_weight*boundary)*person
    alpha=alpha.astype(np.float32)
    # Bridge the hair/background boundary rather than fading to zero there.
    # A small bounded band; uncertain new hair outside the parser remains B24.
    safe_background=background & (r['background_score']>=.98) & ~r['uncertain_new_hair']
    to_hair=outside_distance(hair)
    to_bg=outside_distance(background)
    width=max(2.,SEAM_WIDTH_PX*scale)
    seam_support=(hair & (to_bg<width)) | (safe_background & (to_hair<width))
    support_distance=distance_transform_edt(np.pad(seam_support,1))[1:-1,1:-1]
    soft_support=smootherstep(support_distance/max(1.,3*scale))
    distance_to_edge=np.where(hair,to_bg,to_hair)
    seam_alpha=SEAM_BLEND*(1-smootherstep(distance_to_edge/width))*soft_support
    seam_alpha*=seam_support
    # Preserve the face/eye blend; seam does not include skin labels.
    alpha=1-(1-alpha)*(1-seam_alpha)
    alpha=alpha.astype(np.float32)
    raw=(alpha>0).astype('uint8')*255
    source_weight=(1-r['restore_alpha'])*(1-alpha)
    b24_weight=r['restore_alpha']*(1-alpha)
    assert np.allclose(source_weight+b24_weight+alpha,1,atol=1e-6)
    return dict(r,alpha=alpha,raw_mask=Image.fromarray(raw),face=face,
        background=background,eye_core=visible_eyes,eye_keep=eye_weight,
        edit_core=hair|face|background,source_weight=source_weight,b24_weight=b24_weight,
        seam_mask=(alpha>0)&(alpha<1),seam_edit_alpha=seam_alpha)

def refine_single(base,masks,seed,settings):
    if DIFFUSION_SIDE%8 or DIFFUSION_SIDE<512:raise ValueError('DIFFUSION_SIDE ต้อง >=512 และหาร 8 ลงตัว')
    image,box=padded(base,target=DIFFUSION_SIDE)
    mask,_=padded(masks['raw_mask'],target=DIFFUSION_SIDE,mask=True)
    if not np.asarray(mask).any():raise ValueError('พื้นที่แก้ว่าง')
    with torch.inference_mode():
        result=inpaint_pipe(image=image,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,
            prompt=PROMPT,negative_prompt=NEGATIVE,num_images_per_prompt=1,
            num_inference_steps=settings['steps'],strength=settings['strength'],
            guidance_scale=settings['guidance'],
            generator=torch.Generator(device=DEVICE).manual_seed(seed))
    flags=getattr(result,'nsfw_content_detected',None)
    if flags is not None and any(flags):raise ValueError('โมเดลระงับภาพผลลัพธ์')
    edited=result.images[0].convert('RGB').crop(box).resize(base.size,Image.Resampling.LANCZOS)
    if masks.get('_capture_raw',True):masks['_raw_diffusion']=edited.copy()
    return composite_protected(base,edited,masks['alpha'])


def repair_background(image, original, restoration, seed):
    """Conservative automatic mask; never selects damage from dark color alone."""
    from scipy.ndimage import binary_dilation, gaussian_filter
    r=restoration; scale=min(image.size)/1024.
    labels=r['base_labels'];source_labels=r['source_labels']
    bg_id=label_ids['background']
    target_bg=labels==bg_id
    guard=r['hair_preserve_guard']
    candidate=r['background_candidate'].copy()
    if BACKGROUND_REPAIR_EXTRA_MASK:
        with Image.open(BACKGROUND_REPAIR_EXTRA_MASK) as m:
            if m.size!=image.size:raise ValueError('extra mask ต้องมีขนาดเท่าภาพ aligned')
            manual=np.asarray(m.convert('L'))>127
        candidate |= manual & target_bg & ~guard
    if not candidate.any():
        return image.copy(),np.zeros(labels.shape,np.float32),dict(method='skipped',passes=0,
            reason='No confident background repair region; uncertain pixels preserved as B24.')
    # Feather outward around the damage, and inward near protected anatomy.
    width=max(1.,BACKGROUND_REPAIR_FEATHER_PX*scale)
    # Feather INSIDE manual support so repair cannot spill onto unmarked hair.
    inside=distance_transform_edt(np.pad(candidate,1))[1:-1,1:-1]
    alpha=smootherstep(inside/width)
    clearance=distance_transform_edt(~guard)
    alpha*=smootherstep(clearance/width)
    alpha[guard]=0
    # Source edge background samples are independent of generated black/blue defects.
    border=np.zeros(labels.shape,bool);bw=max(2,round(32*scale))
    border[:bw]=True;border[-bw:]=True;border[:,:bw]=True;border[:,-bw:]=True
    clean=(source_labels==bg_id)&border&~r['old_hair']
    samples=np.asarray(original)[clean].astype(np.float32)
    flat=False;color=None;p90=None
    if len(samples)>=100:
        color=np.median(samples,axis=0)
        errors=np.linalg.norm(samples-color,axis=1)
        p90=float(np.percentile(errors,90))
        flat=p90<=12.0
    if BACKGROUND_REPAIR_MODE not in ('manual_only','auto','inpaint','flat'):
        raise ValueError('BACKGROUND_REPAIR_MODE ต้องเป็น auto/inpaint/flat')
    use_flat=BACKGROUND_REPAIR_MODE=='flat' or (BACKGROUND_REPAIR_MODE in ('auto','manual_only') and flat)
    if use_flat and not flat:
        raise ValueError('ตรวจพื้นหลังสีเรียบไม่ผ่าน ใช้ auto หรือ inpaint')
    passes=0
    if use_flat:
        edited=Image.new('RGB',image.size,tuple(np.rint(color).astype(int)))
        method='flat_source_background'
    else:
        prompt=('Continue only the existing background, matching surrounding texture, '
                'colors and lighting, clean detailed seamless background.')
        negative='hair, person, face, ears, clothing, black fragments, new objects, text, artifacts'
        canvas,box=padded(image,target=DIFFUSION_SIDE)
        mask,_=padded(Image.fromarray((alpha>0).astype('uint8')*255),target=DIFFUSION_SIDE,mask=True)
        with torch.inference_mode():
            output=inpaint_pipe(image=canvas,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,
                prompt=prompt,negative_prompt=negative,strength=BACKGROUND_REPAIR_STRENGTH,
                num_inference_steps=BACKGROUND_REPAIR_STEPS,guidance_scale=6.5,
                generator=torch.Generator(device=DEVICE).manual_seed((int(seed)+101)%4294967296))
        flags=getattr(output,'nsfw_content_detected',None)
        if flags is not None and any(flags):raise ValueError('โมเดลระงับภาพซ่อมพื้นหลัง')
        edited=output.images[0].convert('RGB').crop(box).resize(image.size,Image.Resampling.LANCZOS)
        method='background_inpaint';passes=1
    fixed=composite_protected(image,edited,alpha)
    assert np.array_equal(np.asarray(fixed)[guard],np.asarray(image)[guard])
    assert np.array_equal(np.asarray(fixed)[alpha==0],np.asarray(image)[alpha==0])
    return fixed,alpha,dict(method=method,passes=passes,edge_background_p90=p90,
        sampled_color_rgb=None if color is None else np.rint(color).astype(int).tolist(),
        candidate_pixels=int(candidate.sum()),full_replacement_pixels=int((alpha==1).sum()),
        repair_pixels=int((alpha>0).sum()),strength=BACKGROUND_REPAIR_STRENGTH if passes else None)

# เตรียมก่อนอัปโหลด/รับคำขอ API ไม่โหลดโมเดลในคำขอแรก
PRELOAD_EYE_MODELS=False  # เว็บใช้ face segmentation; debug จะโหลด landmark เมื่อจำเป็น
if PRELOAD_EYE_MODELS:prepare_eye_models()


COLOR_LOCK_MODE = 'auto'  # auto / smoke_gray / blue / off

def validate_color_lock(mode):
    mode=str(mode).strip().lower()
    if mode not in ('auto','smoke_gray','blue','off'):
        raise ValueError('color_lock ต้องเป็น auto, smoke_gray, blue หรือ off')
    return mode

def preserve_hair_color(b24, edited, masks, mode=COLOR_LOCK_MODE):
    from skimage.color import rgb2lab,lab2rgb
    from scipy.ndimage import gaussian_filter
    mode=validate_color_lock(mode)
    hair=np.asarray(masks['hair'],bool)
    zero=np.zeros(hair.shape,np.float32)
    if mode=='off' or hair.sum()<32:
        return edited.copy(),zero,dict(requested=mode,applied='off',reason='disabled_or_small_mask')
    src=rgb2lab(np.asarray(b24.convert('RGB'))).astype(np.float32)
    dst=rgb2lab(np.asarray(edited.convert('RGB'))).astype(np.float32)
    interior=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]
    sample=hair&(interior>=max(1.,min(b24.size)/256.))
    if sample.sum()<32:sample=hair
    L=src[:,:,0][sample];ab=src[:,:,1:][sample]
    C=np.linalg.norm(ab,axis=1)
    blue_share=float(np.mean((ab[:,1]<-10)&(C>15)))
    gray_share=float(np.mean((C<10)&(L>30)&(L<85)))
    applied=mode
    if mode=='auto':
        if blue_share>=.70:applied='blue'
        elif gray_share>=.80 and float(np.median(C))<8:applied='smoke_gray'
        else:applied='off'
    info=dict(requested=mode,applied=applied,blue_share=blue_share,gray_share=gray_share,
              reference='hairfast_B24',auto_is_heuristic=True)
    if applied=='off':return edited.copy(),zero,info
    # Mask-normalized low frequencies exclude skin/background from color estimates.
    sigma=max(1.,4.*min(b24.size)/1024.)
    weight=hair.astype(np.float32)
    denom=np.maximum(gaussian_filter(weight,sigma),1e-6)
    def low(lab):
        return np.stack([gaussian_filter(lab[:,:,i]*weight,sigma)/denom for i in range(3)],axis=-1)
    low_src,low_dst=low(src),low(dst)
    corrected=dst.copy()
    # Recover chroma without copying B24's blurred luminance texture.
    corrected[:,:,1:]=.90*low_src[:,:,1:]+.10*low_dst[:,:,1:]+.20*(dst[:,:,1:]-low_dst[:,:,1:])
    luminance_gain=.80 if applied=='smoke_gray' else .35
    corrected[:,:,0]=np.clip(dst[:,:,0]+luminance_gain*(low_src[:,:,0]-low_dst[:,:,0]),0,100)
    rgb=np.rint(lab2rgb(corrected).clip(0,1)*255).astype(np.uint8)
    edge=smootherstep(interior/max(1.,HAIR_FACE_FEATHER_PX*min(b24.size)/1024.))
    alpha=edge*hair*np.asarray(masks['alpha'],np.float32)
    final=composite_protected(edited,Image.fromarray(rgb),alpha)
    changed=rgb2lab(np.asarray(final))
    info.update(luminance_gain=luminance_gain,chroma_gain=.90,
        mean_chroma_error_before=float(np.linalg.norm(dst[:,:,1:]-src[:,:,1:],axis=-1)[sample].mean()),
        mean_chroma_error_after=float(np.linalg.norm(changed[:,:,1:]-src[:,:,1:],axis=-1)[sample].mean()),
        median_L_B24=float(np.median(L)),median_L_before=float(np.median(dst[:,:,0][sample])),
        median_L_after=float(np.median(changed[:,:,0][sample])))
    assert np.array_equal(np.asarray(final)[alpha==0],np.asarray(edited)[alpha==0])
    return final,alpha,info


HAIR_DETAIL_AMOUNT=0.25  # 0 disables; subtle local contrast, not new strands

def hair_detail_finish(image,masks):
    from scipy.ndimage import gaussian_filter,minimum_filter,maximum_filter
    from skimage.color import rgb2lab,lab2rgb
    hair=np.asarray(masks['hair'],bool)
    if HAIR_DETAIL_AMOUNT==0:return image.copy(),np.zeros_like(hair,dtype=np.float32)
    if not 0<=HAIR_DETAIL_AMOUNT<=.5:raise ValueError('HAIR_DETAIL_AMOUNT ต้องอยู่ระหว่าง 0–0.5')
    lab=rgb2lab(np.asarray(image)).astype(np.float32);L=lab[:,:,0]
    sigma=max(.6,.9*min(image.size)/1024.)
    residual=L-gaussian_filter(L,sigma)
    # Threshold noise; cap contrast and clamp to local range to avoid overshoot halos.
    detail=np.sign(residual)*np.maximum(np.abs(residual)-.35,0)
    corrected=np.clip(L+np.clip(HAIR_DETAIL_AMOUNT*detail,-1.5,1.5),
                      minimum_filter(L,3),maximum_filter(L,3))
    out=lab.copy();out[:,:,0]=corrected
    distance=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]
    scale=min(image.size)/1024.
    # No sharpening near the hair silhouette, skin, background or unedited B24.
    alpha=smootherstep((distance-max(2.,8*scale))/max(1.,4*scale))*masks['alpha']
    alpha*=hair
    rgb=np.rint(lab2rgb(out).clip(0,1)*255).astype(np.uint8)
    final=composite_protected(image,Image.fromarray(rgb),alpha)
    assert np.array_equal(np.asarray(final)[alpha==0],np.asarray(image)[alpha==0])
    return final,alpha


DETAIL_GAIN=.65
DETAIL_MAX_RGB=5.0

def detail_only_masks(base,face_blend):
    labels=segment_labels(base)  # exactly one parser pass
    hair=labels==HAIR_ID;face=labels==FACE_ID
    if not hair.any():raise ValueError('ตรวจไม่พบผม')
    def inside(mask,width):
        distance=distance_transform_edt(np.pad(mask,1))[1:-1,1:-1]
        return smootherstep(distance/max(1.,width*min(base.size)/1024.))
    # No hair silhouette displacement: taper inside each edit region only.
    hair_alpha=inside(hair,4.)
    face_alpha=inside(face,6.)*float(face_blend)
    alpha=np.maximum(hair_alpha,face_alpha).astype(np.float32)
    return dict(hair=hair,face=face,alpha=alpha,
        raw_mask=Image.fromarray((alpha>0).astype('uint8')*255))

def transfer_luminance_detail(b24,proposal,alpha):
    from scipy.ndimage import gaussian_filter
    source=np.asarray(b24.convert('RGB'),dtype=np.float32)
    generated=np.asarray(proposal.convert('RGB'),dtype=np.float32)
    if source.shape!=generated.shape:raise ValueError('ภาพต้องมีขนาดเท่ากัน')
    coeff=np.array([.299,.587,.114],np.float32)
    y0=source@coeff;y1=generated@coeff
    sigma=max(.8,1.2*min(b24.size)/1024.)
    detail0=y0-gaussian_filter(y0,sigma)
    detail1=y1-gaussian_filter(y1,sigma)
    # Preserve B24 coarse structure/color; use only bounded high-frequency delta.
    delta=np.clip(DETAIL_GAIN*(detail1-detail0),-DETAIL_MAX_RGB,DETAIL_MAX_RGB)*alpha
    # Equal RGB increments preserve channel differences; no individual-channel clipping.
    delta=np.clip(delta,-source.min(axis=2),255-source.max(axis=2))
    shift=np.rint(delta).astype(np.int16)
    out=(source.astype(np.int16)+shift[:,:,None]).astype(np.uint8)
    out[alpha==0]=source.astype(np.uint8)[alpha==0]
    return Image.fromarray(out)

def refine_b24_details(base,masks,seed,settings):
    canvas,box=padded(base,target=DIFFUSION_SIDE)
    mask,_=padded(masks['raw_mask'],target=DIFFUSION_SIDE,mask=True)
    with torch.inference_mode():
        result=inpaint_pipe(image=canvas,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,
            prompt='Same portrait, same hairstyle and hair color, fine natural hair detail, clear eyes, natural skin texture, realistic photograph.',
            negative_prompt='changed hairstyle, changed hair color, changed face, waxy skin, blur, halos, noise',
            num_images_per_prompt=1,strength=settings['strength'],num_inference_steps=settings['steps'],
            guidance_scale=settings['guidance'],generator=torch.Generator(device=DEVICE).manual_seed(int(seed)))
    flags=getattr(result,'nsfw_content_detected',None)
    if flags is not None and any(flags):raise ValueError('โมเดลระงับภาพผลลัพธ์')
    proposal=result.images[0].convert('RGB').crop(box).resize(base.size,Image.Resampling.LANCZOS)
    return proposal


RAW_BG_GAP_PX=4.0  # pixels at 1024, scaled with output size
RAW_BG_FEATHER_PX=32.0
RAW_BG_DETAIL_AMOUNT=.45

def finish_raw_background(raw,b24_masks,labels=None):
    from scipy.ndimage import gaussian_filter
    if RAW_BG_GAP_PX<0 or RAW_BG_FEATHER_PX<=0 or not 0<=RAW_BG_DETAIL_AMOUNT<=1:
        raise ValueError('ตรวจค่า RAW_BG_GAP_PX / FEATHER / DETAIL_AMOUNT')
    # Hair may move in raw diffusion: protect both B24 and raw hair.
    if labels is None:labels=segment_labels(raw)
    if 'background' not in label_ids:raise ValueError('ไม่พบ background label')
    bg=labels==label_ids['background']
    hair=(labels==HAIR_ID)|b24_masks['hair']
    scale=min(raw.size)/1024.
    hair_distance=outside_distance(hair)
    gap=RAW_BG_GAP_PX*scale;feather=max(1.,RAW_BG_FEATHER_PX*scale)
    # Stay inside background; protect skin, ears, neck and clothes too.
    bg_distance=distance_transform_edt(np.pad(bg,1))[1:-1,1:-1]
    # One continuous ramp from the nearest protected boundary.
    # Feather the adjustment weight, never the output image.
    safe_distance=np.minimum(hair_distance,bg_distance)
    alpha=(smootherstep((safe_distance-gap)/feather)*bg).astype(np.float32)
    rgb=np.asarray(raw.convert('RGB'),dtype=np.float32)
    luminance=rgb@np.array([.299,.587,.114],np.float32)
    detail=luminance-gaussian_filter(luminance,max(.8,1.2*scale))
    detail=np.sign(detail)*np.maximum(np.abs(detail)-.5,0.)
    delta=np.clip(RAW_BG_DETAIL_AMOUNT*detail,-4.,4.)*alpha
    delta=np.clip(delta,-rgb.min(axis=2),255-rgb.max(axis=2))
    out=(rgb.astype(np.int16)+np.rint(delta).astype(np.int16)[:,:,None]).astype(np.uint8)
    protected=(safe_distance<=gap)|~bg
    assert not alpha[protected].any()
    assert np.array_equal(out[protected],rgb.astype(np.uint8)[protected])
    return Image.fromarray(out),alpha,dict(method='raw_background_smooth_gap_detail',
        gap_px_at_1024=RAW_BG_GAP_PX,feather_px_at_1024=RAW_BG_FEATHER_PX,
        amount=RAW_BG_DETAIL_AMOUNT,protected_raw_max_diff=0,extra_diffusion_passes=0)


# Lightweight native-resolution finish. No extra neural model / diffusion pass.
# Defaults keep the ORIGINAL diffusion prompt and sampling parameters unchanged.
NATIVE_SHARP_ENABLED = True
NATIVE_HAIR_AMOUNT = 0.65
NATIVE_FACE_AMOUNT = 0.25
NATIVE_SHARP_MAX_DELTA = 5.0  # maximum equal RGB shift, in 8-bit levels
NATIVE_SHARP_THRESHOLD = 0.8  # suppress amplification of very small residuals

def finish_native_sharp(image, labels):
    from scipy.ndimage import gaussian_filter
    if not (0 <= NATIVE_HAIR_AMOUNT <= 1.5 and 0 <= NATIVE_FACE_AMOUNT <= .75
            and 0 <= NATIVE_SHARP_MAX_DELTA <= 10 and NATIVE_SHARP_THRESHOLD >= 0):
        raise ValueError('ตรวจค่า NATIVE_SHARP: hair 0–1.5, face 0–.75, delta 0–10')
    rgb=np.asarray(image.convert('RGB'),dtype=np.float32)
    if labels.shape != rgb.shape[:2]:raise ValueError('ขนาดมาสก์ไม่ตรงกับภาพ')
    info=dict(method='bounded_native_luminance_sharpen',enabled=bool(NATIVE_SHARP_ENABLED),
              hair_amount=NATIVE_HAIR_AMOUNT,face_amount=NATIVE_FACE_AMOUNT,
              max_delta=NATIVE_SHARP_MAX_DELTA,threshold=NATIVE_SHARP_THRESHOLD,
              output_size=list(image.size),extra_diffusion_passes=0)
    if not NATIVE_SHARP_ENABLED:
        return image.copy(),dict(info,observed_max_delta=0)
    scale=min(image.size)/1024.
    def inward(region, width):
        dist=distance_transform_edt(np.pad(region,1))[1:-1,1:-1]
        # Leave outermost pixel unchanged; taper inward, never blur the image edge.
        return smootherstep((dist-1.)/max(1.,width*scale))*region
    weight=(NATIVE_HAIR_AMOUNT*inward(labels==HAIR_ID,3.)+
            NATIVE_FACE_AMOUNT*inward(labels==FACE_ID,5.)).astype(np.float32)
    lum=rgb@np.array([.299,.587,.114],np.float32)
    fine=lum-gaussian_filter(lum,max(.55,.75*scale))
    medium=lum-gaussian_filter(lum,max(.8,1.35*scale))
    residual=.75*fine+.25*medium
    residual=np.sign(residual)*np.maximum(np.abs(residual)-NATIVE_SHARP_THRESHOLD,0.)
    delta=np.clip(weight*residual,-NATIVE_SHARP_MAX_DELTA,NATIVE_SHARP_MAX_DELTA)
    # Equal-channel increments preserve RGB channel differences without clipping.
    delta=np.clip(delta,-rgb.min(axis=2),255-rgb.max(axis=2))
    shift=np.rint(delta).astype(np.int16)
    out=(rgb.astype(np.int16)+shift[:,:,None]).astype(np.uint8)
    return Image.fromarray(out),dict(info,observed_max_delta=int(np.abs(shift).max()))


In [ ]:
def run_detail_core(inputs=None,before=None,seed=3407,strength=.15,steps=30,guidance=2.5,face_blend=.35):
    # One shared implementation for the comparison cell, upload test and web.
    settings=resolve_inpaint_settings(strength,steps,guidance)
    resolve_joint_settings(face_blend)
    if not 0<=int(seed)<=4294967295:raise ValueError('seed ไม่ถูกต้อง')
    timings={}
    def clock():
        if torch.cuda.is_available():torch.cuda.synchronize()
        return time.perf_counter()
    queued=time.perf_counter()
    with GPU_LOCK:
        timings['queue_wait']=time.perf_counter()-queued
        start=clock()
        if before is None:
            if inputs is None or len(inputs)!=3:raise ValueError('ต้องมีหน้า/ทรง/สี 3 ภาพ')
            t=clock()
            swapped=hair_fast.swap(*(str(v) for v in inputs),align=True,seed=int(seed))
            if not isinstance(swapped,tuple) or len(swapped)!=4:raise RuntimeError('swap format ไม่ตรงรุ่น')
            before=tensor_to_pil(swapped[0]);timings['B24']=clock()-t
            del swapped
        else:
            before=before.convert('RGB').copy();timings['B24']=0.
        t=clock();masks=detail_only_masks(before,face_blend)
        timings['mask']=clock()-t
        t=clock();raw=refine_b24_details(before,masks,seed,settings)
        timings['diffusion']=clock()-t
        t=clock();raw_labels=segment_labels(raw)
        before_sharp,bg_alpha,bg_info=finish_raw_background(raw,masks,labels=raw_labels)
        timings['raw_background_finish']=clock()-t
        t=clock();after,sharp_info=finish_native_sharp(before_sharp,raw_labels)
        timings['native_sharp']=clock()-t
        timings['total_processing']=clock()-start
    return dict(before=before,raw_diffusion=raw,before_sharp=before_sharp,after=after,native_sharp=sharp_info,sharp_labels=raw_labels,masks=masks,background_alpha=bg_alpha,background_info=bg_info,settings=settings,
                seed=int(seed),timings_seconds={k:round(v,4) for k,v in timings.items()})


## 10. รันทั้งสองโมเดลและบันทึก 8 PNG
ตรวจพิกเซลที่ต้องรักษาแยกตามขั้น: background เทียบ raw, Native Sharp เทียบ before_sharp ไม่มีการนำ assertion ของขั้นเดิมไปบังคับผลสุดท้ายที่ตั้งใจปรับผม/หน้า

In [ ]:
import pandas as pd
RESULTS={}; EXPORT_IMAGES=[]; TIMING_ROWS=[]
FINAL_DIR=RUN_DIR/'comparison_native_sharp'; FINAL_DIR.mkdir(exist_ok=True)
MODELS=[('Barbershop_Baseline',BASE_DIR/'Barbershop_Baseline.png'),
        ('HairFastGAN_B24',BASE_DIR/'HairFastGAN_B24.png')]
PROFILE=dict(seed=int(SEED),strength=STRENGTH,steps=STEPS,guidance=GUIDANCE,face_blend=FACE_BLEND)
for model,path in MODELS:
    print('Diffusion:',model)
    with Image.open(path) as im: before=im.convert('RGB').copy()
    result=run_detail_core(before=before,**PROFILE)
    # shared legacy core calls its GAN timing B24; remove this unused zero-valued field
    result['timings_seconds'].pop('B24',None)
    RESULTS[model]=result
    dest=FINAL_DIR/model; dest.mkdir(exist_ok=True)
    for index,key,label in [(1,'before','Before_Diffusion'),(2,'raw_diffusion','Raw_Diffusion'),
                            (3,'before_sharp','After_Background_Before_Sharp'),
                            (4,'after','Final_Native_Sharp')]:
        out=dest/f'{model}_{index:02d}_{label}_{RUN_ID}.png'
        result[key].save(out,format='PNG',compress_level=1)
        EXPORT_IMAGES.append(out)
    result['masks']['raw_mask'].save(dest/'diffusion_mask.png')
    for name in ['hair','face','alpha']:
        Image.fromarray(np.rint(result['masks'][name]*255).astype('uint8')).save(dest/(name+'_mask.png'))
    Image.fromarray(np.rint(result['background_alpha']*255).astype('uint8')).save(dest/'background_alpha.png')
    raw=np.asarray(result['raw_diffusion']);background=np.asarray(result['before_sharp']);after=np.asarray(result['after'])
    keep=result['background_alpha']==0
    assert np.array_equal(raw[keep],background[keep]),'Background finish changed protected raw pixels'
    sharp_protected=~np.isin(result['sharp_labels'],[HAIR_ID,FACE_ID])
    assert np.array_equal(background[sharp_protected],after[sharp_protected]),'Native sharp changed non-hair/face pixels'
    observed=int(np.abs(after.astype('int16')-background.astype('int16')).max())
    assert observed<=np.ceil(NATIVE_SHARP_MAX_DELTA),'Native sharp exceeded bounded delta'
    Image.fromarray(result['sharp_labels'].astype('uint8')).save(dest/'native_sharp_labels.png')
    transfer_seconds=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_RESULT['estimated_cold_compute_seconds']
    TIMING_ROWS.append(dict(model=model,transfer_compute_seconds=transfer_seconds,
        transfer_wall_seconds_this_run=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_WALL_SECONDS,
        **result['timings_seconds']))
    metadata=dict(model=model,profile=PROFILE,settings=result['settings'],
        base_sha256=sha_file(path),background_info=result['background_info'],native_sharp=result['native_sharp'],
        timings=TIMING_ROWS[-1],face_blend_is_composite_weight=False,
        mask_pixels={k:int(result['masks'][k].sum()) for k in ['hair','face']})
    (dest/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))
TIMINGS=pd.DataFrame(TIMING_ROWS)
TIMINGS['transfer_plus_postprocess_seconds']=TIMINGS.transfer_compute_seconds+TIMINGS.total_processing
TIMINGS.to_csv(FINAL_DIR/'timings.csv',index=False)
manifest=dict(method='looklab_native_sharp',native_source_sha256='d4c5f63c6f895bb4b7569a8f5104a373113ef33c046c81c1534a7fea4634072d',run_id=RUN_ID,profile=PROFILE,environment=ENVIRONMENT,
    barbershop=dict(revision=BARBER_REV,weights=weight_hashes,selection='Baseline',
        selected_from_validation_protocol='27f7f146ef0d',worker_result=BARBER_RESULT),
    hairfast=dict(revision=REPO_COMMIT,checkpoint=str(CHECKPOINT),sha256=EXPECTED_SHA256,
        trial='B',epoch=24,mixing=.95,shape_rotation=False,color_rotation=True),
    original_hashes={p.name:sha_file(p) for p in INPUTS},
    aligned_hashes={p.name:sha_file(p) for p in CROP_DIR.glob('*.png')},
    export_images=[str(p) for p in EXPORT_IMAGES],
    notes=['Same aligned crops; Barbershop consumes quantized PNGs.',
           'Same postprocess algorithm/settings; masks depend on each output.',
           'Timing excludes initial weight downloads and model loading for HFG; BS wall includes worker startup.',
           'BS transfer compute includes inversion even when cached; wall time is shown separately.',
           'Native sharp operates at original resolution without extra diffusion; metric results in evaluation folder.'])
(FINAL_DIR/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
display(TIMINGS)
print('บันทึกภาพเต็ม 8 รูป:',FINAL_DIR)


## 11. แสดงภาพเต็มและครอปตำแหน่งเดียวกันทั้ง 8 ภาพ

In [ ]:
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt
stages=[('before','Before diffusion'),('raw_diffusion','Raw diffusion'),('before_sharp','Background finish'),('after','Native sharp')]
shapes={result[key].size for result in RESULTS.values() for key,_ in stages}
if len(shapes)!=1: raise ValueError('ภาพต้องมีขนาดเท่ากันเพื่อใช้ crop เดียว')
w,h=next(iter(shapes))
if COMPARE_CROP is None:
    union=np.logical_or.reduce([v['masks']['hair'] for v in RESULTS.values()])
    distance=distance_transform_edt(np.pad(union,1))[1:-1,1:-1]
    cy,cx=np.unravel_index(np.argmax(distance),distance.shape)
    side=min(256,w,h);left=max(0,min(w-side,int(cx)-side//2));top=max(0,min(h-side,int(cy)-side//2))
    crop=(left,top,left+side,top+side)
else:
    crop=tuple(int(v) for v in COMPARE_CROP)
    if len(crop)!=4 or not (0<=crop[0]<crop[2]<=w and 0<=crop[1]<crop[3]<=h):
        raise ValueError('crop ต้องเป็น (left,top,right,bottom) ภายในภาพ')
for cropped in [False,True]:
    fig,axes=plt.subplots(2,4,figsize=(22,12 if not cropped else 10))
    for row,(model,_) in enumerate(MODELS):
        for col,(key,label) in enumerate(stages):
            im=RESULTS[model][key]
            axes[row,col].imshow(im.crop(crop) if cropped else im,interpolation='nearest')
            axes[row,col].set_title(model+' | '+label);axes[row,col].axis('off')
    fig.tight_layout()
    fig.savefig(FINAL_DIR/('comparison_same_crop.png' if cropped else 'comparison_full.png'),dpi=150)
    display(fig);plt.close(fig)
(FINAL_DIR/'crop.json').write_text(json.dumps(dict(box=crop,coordinate_system='same aligned 1024x1024 canvas'),indent=2))
print('Crop เดียวกันทุกภาพ:',crop)
print('ภาพ grid เป็น preview; PNG แยกในส่วน 12 เก็บ resolution เดิม')


## 12. ดาวน์โหลด PNG แยกทีละรูป
ปุ่มแต่ละอันดาวน์โหลดไฟล์ PNG หนึ่งรูป ใช้ชื่อระบุโมเดลและขั้นตอนครบ ไม่สร้าง ZIP หากเปิดดาวน์โหลดอัตโนมัติแล้ว browser บล็อก ให้ใช้ปุ่มทีละรูป

In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display
if len(EXPORT_IMAGES)!=8 or not all(p.is_file() for p in EXPORT_IMAGES):
    raise RuntimeError('รันส่วน 10 ให้ครบก่อน')
buttons=[]
for path in EXPORT_IMAGES:
    button=widgets.Button(description=path.name.split('_'+RUN_ID)[0],
        layout=widgets.Layout(width='650px'),tooltip=path.name)
    button.on_click(lambda _,p=path: files.download(str(p)))
    buttons.append(button)
display(widgets.VBox(buttons))
if AUTO_DOWNLOAD_IMAGES:
    for path in EXPORT_IMAGES:
        print('ดาวน์โหลด:',path.name)
        files.download(str(path))
print('ผลและ metadata เก็บไว้แล้วที่:',FINAL_DIR)


## 13. เวลาแต่ละขั้น (วินาที)
- HFG: เวลาจริงของ `swap(align=True)` รวม alignment และ transfer แต่ไม่รวมโหลด weights
- Barbershop: W inversion, FS inversion, align shape, align color, blend จาก worker แสดงรวมกับเวลาคำนวณที่บันทึกไว้ **แม้ cache ทำให้รอบนี้ข้ามบางขั้น**; wall time รอบนี้แสดงแยก รวมโหลดโมเดลใน subprocess
- ทั้งสองใช้ crops ที่ HFG จัดแนวไว้ ดังนั้นขอบเขตการจับเวลา transfer ไม่เท่ากันทุกขั้น ห้ามตีความ ratio ว่าเป็น benchmark สถาปัตยกรรมที่ยุติธรรมสมบูรณ์
- postprocess แยก mask / diffusion / background finish; ไม่รวมอัปโหลด ดาวน์โหลด แสดงภาพ และการคำนวณเมตริก
- ตัวเลขหลักเป็น **หนึ่งรอบต่อหนึ่งชุดภาพ** มี CUDA synchronize ก่อน–หลัง ไม่ใช่ค่าเฉลี่ยหลายรอบ; ส่วน 17 มีการวัด postprocess ซ้ำแบบ warm benchmark ให้เลือก

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
from IPython.display import display
if 'RESULTS' not in globals() or len(RESULTS)!=2:
    raise RuntimeError('รันส่วน 10 ให้ครบก่อน')
EVAL_DIR=FINAL_DIR/'evaluation';EVAL_DIR.mkdir(exist_ok=True)
time_rows=[]
def add_time(model,stage,seconds,scope):
    time_rows.append(dict(model=model,stage=stage,seconds=float(seconds),scope=scope))
add_time('HairFastGAN_B24','swap_including_alignment',HFG_SECONDS,'measured once; excludes model load')
inv_records=[json.loads(p.read_text()) for p in (BARBER_ROOT/'inversion').glob('*_complete.json')]
for key in ['W_seconds','FS_seconds']:
    add_time('Barbershop_Baseline',key,sum(r[key] for r in inv_records),'recorded compute; may be reused from cache')
for key in ['align_shape_seconds','align_color_seconds','blend_seconds']:
    add_time('Barbershop_Baseline',key,BARBER_RESULT[key],'recorded compute; may be reused from cache')
add_time('Barbershop_Baseline','worker_wall_this_run',BARBER_WALL_SECONDS,'includes worker/model load; cache may skip optimization')
for model,result in RESULTS.items():
    for key in ['mask','diffusion','raw_background_finish','native_sharp','total_processing']:
        add_time(model,key,result['timings_seconds'][key],'measured postprocess once; total contains preceding 4 stages')
TIME_DETAIL=pd.DataFrame(time_rows)
TIME_DETAIL.to_csv(EVAL_DIR/'timing_stages.csv',index=False)
time_summary=[]
for model,result in RESULTS.items():
    transfer=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_RESULT['estimated_cold_compute_seconds']
    ts=result['timings_seconds']
    time_summary.append(dict(model=model,transfer_recorded_s=transfer,mask_s=ts['mask'],
        diffusion_s=ts['diffusion'],background_s=ts['raw_background_finish'],native_sharp_s=ts['native_sharp'],
        postprocess_total_s=ts['total_processing'],
        transfer_plus_postprocess_s=transfer+ts['total_processing'],
        postprocess_overhead_percent=100*ts['total_processing']/transfer if transfer>0 else np.nan,
        scope='HFG includes alignment; BS excludes shared alignment; loading excluded from summed compute'))
TIME_SUMMARY=pd.DataFrame(time_summary)
TIME_SUMMARY.to_csv(EVAL_DIR/'timing_summary.csv',index=False)
display(TIME_SUMMARY.round(3));display(TIME_DETAIL.round(3))


## 14. เมตริกหลัก: คุณภาพ / ความตรงเป้าหมาย / การรักษาสิ่งเดิม
| ตัววัดหลัก | เทียบอะไร | การตีความ |
|---|---|---|
| MUSIQ ↑ | ให้คะแนนภาพเต็มแต่ละภาพเอง ไม่มีภาพคำตอบ | คุณภาพภาพโดยรวมที่โมเดลคาดการณ์ ไม่ใช่ความถูกต้องของผม |
| ArcFace cosine ↑ | รูปหน้าอัปโหลด | ความคล้ายอัตลักษณ์ |
| Hair Lab quantile error ↓ | รูปสีอ้างอิง | สีเข้าใกล้เป้าหมาย; core hair คงที่ต่อโมเดล |
| Hair-mask IoU ↑ | ผมก่อน diffusion ของโมเดลนั้น | รักษาทรงก่อนแก้ได้แค่ไหน ไม่ใช่ความตรงทรงเป้าหมาย |
| Protected MAE ↓ / SSIM ↑ | ก่อน diffusion เฉพาะนอก edit mask ที่หดขอบ 4px | วัดการเปลี่ยนที่เพิ่มขึ้นจาก postprocess |

MUSIQ ใช้ `pyiqa.create_metric('musiq')` บน CPU ไม่รวมในเวลา inference ดาวน์โหลด weights ครั้งแรก แล้วใช้ซ้ำใน runtime เดิม หากติดตั้งหรือโหลดไม่สำเร็จจะหยุดพร้อม error ไม่แทนด้วยคะแนนอื่น
MUSIQ วัดภาพเต็ม RGB ไม่ crop/mask ดำ เพื่อไม่เปลี่ยนการกระจายภาพที่ evaluator รับ ไม่ใช่คะแนนเส้นผมโดยเฉพาะ
พื้นที่ protected เทียบ before ไม่เปลี่ยนตามผลหลัง; background finish ตั้งใจแก้พื้นหลังบางส่วน ดังนั้นรายงานช่วง raw→final เป็น drift ที่ตั้งใจได้ ไม่เหมารวมว่าแย่
Shape-reference IoU, sharpness และค่าเทียบรูป input เดิมยังเก็บใน CSV เป็นข้อมูลเสริม; ไม่รวมคะแนนทุกอย่างเป็นตัวเดียว
ส่วน 19 เตรียมภาพประเมินโดยคนแบบไม่บอกชื่อโมเดล เพื่อวัดเส้นผม ขอบผม และคุณภาพภาพรวม

แหล่ง implementation: https://github.com/chaofengc/IQA-PyTorch
MUSIQ paper: https://openaccess.thecvf.com/content/ICCV2021/html/Ke_MUSIQ_Multi-Scale_Image_Quality_Transformer_ICCV_2021_paper.html
ArcFace และ SSIM ใช้ implementation จากส่วนเดิม; SSIM ใช้ window 7x7 เฉพาะตำแหน่งที่หน้าต่างอยู่ใน mask ทั้งหมด


In [ ]:
import time, shutil, torch
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import binary_erosion, binary_dilation, gaussian_filter, laplace
from skimage.color import rgb2lab
from skimage.metrics import structural_similarity
from losses.pp_losses import Backbone
from huggingface_hub import hf_hub_download
SHAPE_TARGET_MASK_PATH = ''  # optional: aligned binary 1024x1024 target mask
MIN_METRIC_PIXELS=64
# CPU evaluator avoids contaminating inference GPU timings and avoids diffusion VRAM overlap.
METRIC_DEVICE='cpu'
id_path=Path(LOCAL_REPO)/'pretrained_models/ArcFace/ir_se50.pth'
if not id_path.is_file():
    downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',
        filename='pretrained_models/ArcFace/ir_se50.pth',revision='main')
    id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
metric_id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
metric_id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
metric_id_net.to(METRIC_DEVICE).eval().requires_grad_(False)

def rgb01(image):return np.asarray(image.convert('RGB'),dtype=np.float32)/255.

def identity_embedding(image):
    x=torch.from_numpy(rgb01(image).copy()).permute(2,0,1)[None].to(METRIC_DEVICE)
    x=F.interpolate(x,size=(256,256),mode='bilinear',align_corners=False)*2-1
    x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
    with torch.inference_mode():features=F.normalize(metric_id_net(x),dim=1)
    return features[0].cpu().numpy()

def mask_mean(values,mask):
    return float(np.asarray(values)[mask].mean()) if int(mask.sum())>=MIN_METRIC_PIXELS else float('nan')

def mask_iou(a,b):
    union=(a|b).sum()
    return float((a&b).sum()/union) if union else float('nan')

def masked_ssim(a,b,mask):
    safe=binary_erosion(mask,structure=np.ones((7,7),bool),border_value=0)
    if safe.sum()<MIN_METRIC_PIXELS:return float('nan')
    _,ssmap=structural_similarity(a,b,data_range=1.,channel_axis=2,win_size=7,full=True)
    if ssmap.ndim==3:ssmap=ssmap.mean(axis=2)
    return mask_mean(ssmap,safe)

def color_quantile_error(lab_image,lab_ref,mask,refmask):
    if min(mask.sum(),refmask.sum())<MIN_METRIC_PIXELS:return (float('nan'),float('nan'))
    q=np.linspace(.1,.9,9)
    delta=np.abs(np.quantile(lab_image[mask],q,axis=0)-np.quantile(lab_ref[refmask],q,axis=0))
    return float(delta[:,0].mean()),float(delta[:,1:].mean())

REF_IMAGES={role:Image.open(CROP_DIR/(role+'.png')).convert('RGB') for role in ['face','shape','color']}
if {im.size for im in REF_IMAGES.values()}!={(1024,1024)}:raise ValueError('References ต้องเป็น 1024x1024')
REF_LABELS={role:segment_labels(im) for role,im in REF_IMAGES.items()}
if SHAPE_TARGET_MASK_PATH:
    with Image.open(SHAPE_TARGET_MASK_PATH) as im:
        if im.size!=(1024,1024):raise ValueError('Target mask ต้องตรง canvas 1024x1024 ห้าม resize อัตโนมัติ')
        TARGET_HAIR=np.asarray(im.convert('L'))>127
    TARGET_KIND='user_supplied_aligned_target_mask'
else:
    TARGET_HAIR=REF_LABELS['shape']==HAIR_ID
    TARGET_KIND='aligned_shape_reference_proxy_not_ground_truth'
if TARGET_HAIR.sum()<MIN_METRIC_PIXELS:raise ValueError('ไม่พบพื้นที่ผมเพียงพอใน target/reference')
REF_COLOR_CORE=binary_erosion(REF_LABELS['color']==HAIR_ID,iterations=4)
UNION_HAIR=(REF_LABELS['face']==HAIR_ID)|TARGET_HAIR
for result in RESULTS.values(): UNION_HAIR|=result['masks']['hair']
COMMON_PROTECTED=~binary_dilation(UNION_HAIR,iterations=8)
COMMON_FACE=binary_erosion(REF_LABELS['face']==FACE_ID,iterations=3)&COMMON_PROTECTED
COMMON_BG=binary_erosion(REF_LABELS['face']==label_ids['background'],iterations=3)&COMMON_PROTECTED
SOURCE_RGB=rgb01(REF_IMAGES['face']);COLOR_LAB=rgb2lab(rgb01(REF_IMAGES['color']))
SOURCE_ID=identity_embedding(REF_IMAGES['face'])
EVAL_MASK_DIR=EVAL_DIR/'fixed_masks';EVAL_MASK_DIR.mkdir(exist_ok=True)
for name,mask in [('target_hair',TARGET_HAIR),('color_reference_core',REF_COLOR_CORE),
                  ('protected_common',COMMON_PROTECTED),('face_common',COMMON_FACE),('background_common',COMMON_BG)]:
    Image.fromarray(mask.astype('uint8')*255).save(EVAL_MASK_DIR/(name+'.png'))
print('Shape target:',TARGET_KIND)
print('Fixed mask pixels:',{k:int(v.sum()) for k,v in dict(protected=COMMON_PROTECTED,face=COMMON_FACE,background=COMMON_BG,color_ref=REF_COLOR_CORE).items()})

# MUSIQ runs on CPU, outside generation timing. Preserve current core runtime packages.
import importlib.util, importlib.metadata, tempfile, sys, subprocess
if importlib.util.find_spec('pyiqa') is None:
    constraints=[]
    for package in ['torch','torchvision','torchaudio','numpy','scipy','Pillow']:
        try: constraints.append(package+'=='+importlib.metadata.version(package))
        except importlib.metadata.PackageNotFoundError: pass
    with tempfile.TemporaryDirectory() as tmp:
        constraint_path=Path(tmp)/'runtime_constraints.txt'
        constraint_path.write_text('\n'.join(constraints))
        subprocess.run([sys.executable,'-m','pip','install','-q','pyiqa','-c',str(constraint_path)],check=True)
import pyiqa
MUSIQ_DEVICE='cpu'
if 'musiq_metric' not in globals():
    musiq_metric=pyiqa.create_metric('musiq',device=MUSIQ_DEVICE)
    musiq_metric.eval().requires_grad_(False)
if musiq_metric.lower_better:raise RuntimeError('Unexpected MUSIQ direction')
MUSIQ_CONFIG=dict(metric='musiq',pyiqa_version=importlib.metadata.version('pyiqa'),
    device=MUSIQ_DEVICE,input='full RGB image in [0,1]; no manual resize/crop',
    scope='no-reference global image quality; not a hair correctness or identity score')
def musiq_score(image):
    x=torch.from_numpy(rgb01(image).copy()).permute(2,0,1)[None].to(MUSIQ_DEVICE)
    with torch.inference_mode():score=float(musiq_metric(x).reshape(-1)[0].cpu())
    if not np.isfinite(score):raise ValueError('MUSIQ returned nonfinite score')
    return score


## 15. วัด 8 ภาพและผลต่าง 4 ช่วง
แยก before→raw, raw→before_sharp, before_sharp→after และ before→after เพื่อเห็นผลของ Native Sharp โดยเฉพาะ ค่า preservation ก่อน diffusion เริ่มที่ IoU=1, MAE=0, SSIM=1 เพราะเทียบตัวเอง

In [ ]:
metric_start=time.perf_counter()
METRIC_ROWS=[]
for model,result in RESULTS.items():
    before=result['before'];before_rgb=rgb01(before);before_lab=rgb2lab(before_rgb)
    before_id=identity_embedding(before)
    hair_before=result['masks']['hair'].astype(bool)
    fixed_core=binary_erosion(hair_before,iterations=4)
    outside=result['masks']['alpha']==0
    protected_before=binary_erosion(outside,iterations=4)
    Image.fromarray(protected_before.astype('uint8')*255).save(EVAL_MASK_DIR/(model+'_protected_before.png'))
    Image.fromarray(fixed_core.astype('uint8')*255).save(EVAL_MASK_DIR/(model+'_hair_core.png'))
    for stage in ['before','raw_diffusion','before_sharp','after']:
        image=result[stage]
        if image.size!=before.size or image.size!=REF_IMAGES['face'].size:raise ValueError('Metric images ต้องอยู่ canvas เดียวกัน')
        rgb=rgb01(image);lab_value=rgb2lab(rgb)
        hair=hair_before if stage=='before' else segment_labels(image)==HAIR_ID
        ident=before_id if stage=='before' else identity_embedding(image)
        lerr,aberr=color_quantile_error(lab_value,COLOR_LAB,fixed_core,REF_COLOR_CORE)
        luminance=rgb@np.array([.299,.587,.114],np.float32)
        high=luminance-gaussian_filter(luminance,1.2)
        lap=laplace(luminance)
        row=dict(model=model,stage=stage,shape_target_kind=TARGET_KIND,
            musiq=musiq_score(image),
            identity_cosine_vs_input=float(np.clip(np.dot(ident,SOURCE_ID),-1,1)),
            identity_cosine_vs_before=float(np.clip(np.dot(ident,before_id),-1,1)),
            hair_L_quantile_error=lerr,hair_ab_quantile_error=aberr,
            hair_IoU_vs_shape_reference=mask_iou(hair,TARGET_HAIR),
            hair_IoU_vs_before=mask_iou(hair,hair_before),
            hair_deltaE76_vs_before=mask_mean(np.linalg.norm(lab_value-before_lab,axis=2),fixed_core),
            hair_laplacian_variance=float(lap[fixed_core].var()) if fixed_core.sum()>=MIN_METRIC_PIXELS else np.nan,
            hair_high_frequency_RMS=float(np.sqrt(mask_mean(high**2,fixed_core))),
            hair_extreme_channel_fraction=mask_mean(((rgb<=.5/255)|(rgb>=254.5/255)).any(axis=2),fixed_core),
            outside_edit_RGB_MAE_vs_before=mask_mean(np.abs(rgb-before_rgb),outside),
            protected_RGB_MAE_vs_before=mask_mean(np.abs(rgb-before_rgb),protected_before),
            protected_SSIM_vs_before=masked_ssim(rgb,before_rgb,protected_before),
            protected_before_pixels=int(protected_before.sum()),
            fixed_hair_core_pixels=int(fixed_core.sum()),predicted_hair_pixels=int(hair.sum()))
        for name,mask in [('protected',COMMON_PROTECTED),('face',COMMON_FACE),('background',COMMON_BG)]:
            row[name+'_RGB_MAE_vs_input']=mask_mean(np.abs(rgb-SOURCE_RGB),mask)
            row[name+'_SSIM_vs_input']=masked_ssim(rgb,SOURCE_RGB,mask)
            row[name+'_pixels']=int(mask.sum())
        METRIC_ROWS.append(row)
METRIC_FRAME=pd.DataFrame(METRIC_ROWS)
METRIC_FRAME.to_csv(EVAL_DIR/'metrics_all_stages.csv',index=False)
# desired direction: diagnostics are deliberately left without a good/bad label
DIRECTIONS={
 'musiq':1,'protected_RGB_MAE_vs_before':-1,'protected_SSIM_vs_before':1,
 'identity_cosine_vs_input':1,'identity_cosine_vs_before':1,
 'hair_L_quantile_error':-1,'hair_ab_quantile_error':-1,
 'hair_IoU_vs_shape_reference':1,'hair_IoU_vs_before':1,
 'hair_deltaE76_vs_before':0,'hair_laplacian_variance':0,'hair_high_frequency_RMS':0,
 'hair_extreme_channel_fraction':0,'outside_edit_RGB_MAE_vs_before':0}
for area in ['protected','face','background']:
    DIRECTIONS[area+'_RGB_MAE_vs_input']=-1;DIRECTIONS[area+'_SSIM_vs_input']=1

def interpretation(delta,direction):
    if not np.isfinite(delta):return 'N/A: insufficient region or invalid measurement'
    if abs(delta)<=1e-6:return 'unchanged within numerical tolerance'
    if direction==0:return 'increased; inspect image' if delta>0 else 'decreased; inspect image'
    return 'toward preferred metric direction; not overall quality' if delta*direction>0 else 'away from preferred metric direction; not overall quality'
DELTA_ROWS=[]
for model in RESULTS:
    block=METRIC_FRAME[METRIC_FRAME.model==model].set_index('stage')
    for transition,start,end in [('diffusion','before','raw_diffusion'),('background_finish','raw_diffusion','before_sharp'),('native_sharp','before_sharp','after'),('total','before','after')]:
        for metric,direction in DIRECTIONS.items():
            a=float(block.loc[start,metric]);b=float(block.loc[end,metric]);delta=b-a
            DELTA_ROWS.append(dict(model=model,transition=transition,metric=metric,before=a,after=b,
                delta_after_minus_before=delta,desired_direction={1:'higher',-1:'lower',0:'diagnostic'}[direction],
                interpretation=interpretation(delta,direction)))
DELTA_FRAME=pd.DataFrame(DELTA_ROWS)
DELTA_FRAME.to_csv(EVAL_DIR/'metrics_deltas.csv',index=False)
EVALUATION_SECONDS=time.perf_counter()-metric_start
protocol=dict(version='before_after_native_sharp_metrics_v3',native_sharp={m:r['native_sharp'] for m,r in RESULTS.items()},musiq=MUSIQ_CONFIG,shape_target_kind=TARGET_KIND,
    shape_target_sha256=sha_file(SHAPE_TARGET_MASK_PATH) if SHAPE_TARGET_MASK_PATH else None,
    arcface_sha256=sha_file(id_path),metric_device=METRIC_DEVICE,
    segmentation_model=SEG_MODEL_ID,segmentation_revision=ENVIRONMENT.get('seg_revision_resolved'),
    min_pixels=MIN_METRIC_PIXELS,quantiles=np.linspace(.1,.9,9).tolist(),
    ssim=dict(data_range=1,win_size=7,channel_axis=2,mask_safe_window=True),
    support='fixed own-before hair core and protected area (outside edit mask eroded 4px); input comparisons retained as supplementary',
    image_resolution=[1024,1024],evaluation_seconds_excluding_evaluator_setup=EVALUATION_SECONDS,
    metric_directions=DIRECTIONS,not_comparable_to_old_FT_metrics=True,
    caveats=['One image set is exploratory, not statistical evidence.',
             'Shape reference is a pose-sensitive proxy unless supplied as aligned target mask.',
             'Sharpness can increase with noise; no unconditional quality direction.',
             'Preservation vs before measures drift, not reference correctness.',
             'Per-model hair cores differ: use color/detail deltas mainly within each model.'])
(EVAL_DIR/'evaluation_protocol.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2))
main=['model','stage','musiq','identity_cosine_vs_input','hair_L_quantile_error','hair_ab_quantile_error',
      'hair_IoU_vs_before','protected_RGB_MAE_vs_before','protected_SSIM_vs_before']
display(METRIC_FRAME[main].round(5))
for model in RESULTS:
    print('ก่อน → หลังทั้งหมด:',model)
    display(DELTA_FRAME[(DELTA_FRAME.model==model)&(DELTA_FRAME.transition=='total')].round(6))
print('เวลาประเมินแยกจาก inference:',round(EVALUATION_SECONDS,2),'วินาที')


## 16. กราฟและรายงานผลดี–ผลเสีย พร้อมดาวน์โหลดตาราง
ไม่มีการสรุปว่า diffusion ช่วยก่อนเห็นค่าที่รันจริง รายงานจะบอกทิศทางที่วัดได้ของแต่ละเมตริก และแยก diagnostics ที่ต้องดูภาพประกอบ

In [ ]:
import matplotlib.pyplot as plt
panels=[('musiq','MUSIQ: predicted global quality (higher)'),
 ('identity_cosine_vs_input','Identity vs input (higher)'),
 ('hair_ab_quantile_error','Hair color reference error (lower)'),
 ('hair_IoU_vs_before','Hair shape preservation vs before (higher)'),
 ('protected_RGB_MAE_vs_before','Protected drift vs before (lower)'),
 ('protected_SSIM_vs_before','Protected similarity vs before (higher)')]
fig,axes=plt.subplots(2,3,figsize=(17,9))
stage_order=['before','raw_diffusion','before_sharp','after']
for ax,(metric,title) in zip(axes.flat,panels):
    for model in RESULTS:
        block=METRIC_FRAME[METRIC_FRAME.model==model].set_index('stage').reindex(stage_order)
        ax.plot(range(4),block[metric],marker='o',label=model)
    ax.set_xticks(range(4),['Before','Diffusion','Background','Sharp']);ax.set_title(title);ax.grid(alpha=.25)
axes[0,0].legend(fontsize=8);fig.tight_layout()
fig.savefig(EVAL_DIR/'metrics_before_after.png',dpi=160);display(fig);plt.close(fig)
THAI_VERDICT={'toward preferred metric direction; not overall quality':'เข้าใกล้เป้าหมายของเมตริกนี้ ไม่ใช่คุณภาพรวม', 'away from preferred metric direction; not overall quality':'ออกห่างเป้าหมายของเมตริกนี้ ไม่ใช่คุณภาพรวม', 'improved on this metric':'ดีขึ้นตามเมตริกนี้', 'worsened on this metric':'แย่ลงตามเมตริกนี้', 'unchanged within numerical tolerance':'ไม่เปลี่ยนภายใน tolerance', 'increased; inspect image':'เพิ่มขึ้น — ต้องดูภาพประกอบ', 'decreased; inspect image':'ลดลง — ต้องดูภาพประกอบ', 'N/A: insufficient region or invalid measurement':'วัดไม่ได้: พื้นที่ไม่พอหรือค่าไม่ถูกต้อง'}
lines=['# ผลก่อน–หลัง diffusion',f'Run: {RUN_ID}',f'Shape target: {TARGET_KIND}',
       '', 'MUSIQ เป็นคะแนนคุณภาพภาพโดยรวมจากโมเดล ไม่รับรองความสมจริงของเส้นผมเฉพาะส่วน; ใช้การประเมินคนประกอบ', 'ตัวเลขมาจากภาพชุดนี้เท่านั้น ไม่ใช่ข้อสรุปเชิงสถิติ; higher sharpness ไม่รับประกันว่าภาพดีขึ้น',
       'คะแนนชุดนี้ไม่เทียบตรงกับ FT report เก่า เพราะ protocol/masks/parser ต่างกัน','']
for model in RESULTS:
    lines+=['## '+model]
    t=TIME_SUMMARY.set_index('model').loc[model]
    lines += [f"Transfer recorded: {t.transfer_recorded_s:.3f} s; diffusion: {t.diffusion_s:.3f} s; background: {t.background_s:.3f} s; native sharp: {t.native_sharp_s:.3f} s; postprocess total: {t.postprocess_total_s:.3f} s",
      'เวลา transfer ของสองโมเดลมีขอบเขตต่างกันตาม timing_stages.csv; ไม่ใช้ ratio สรุปความเร็วสถาปัตยกรรม','']
    for transition in ['diffusion','background_finish','native_sharp','total']:
        lines+=['### '+transition]
        rows=DELTA_FRAME[(DELTA_FRAME.model==model)&(DELTA_FRAME.transition==transition)]
        for _,r in rows.iterrows():
            lines.append(f"- {r.metric}: {r.before:.6g} → {r.after:.6g} (Δ {r.delta_after_minus_before:+.6g}); {THAI_VERDICT.get(r.interpretation,r.interpretation)}")
        lines+=['']
(EVAL_DIR/'tradeoff_report.md').write_text('\n'.join(lines),encoding='utf-8')
print('บันทึก CSV, กราฟ และ tradeoff_report.md ที่:',EVAL_DIR)
import ipywidgets as widgets
from google.colab import files
buttons=[]
for filename in ['timing_summary.csv','timing_stages.csv','metrics_all_stages.csv','metrics_deltas.csv','metrics_before_after.png','tradeoff_report.md','evaluation_protocol.json']:
    path=EVAL_DIR/filename
    button=widgets.Button(description=filename,layout=widgets.Layout(width='400px'))
    button.on_click(lambda _,p=path:files.download(str(p)));buttons.append(button)
display(widgets.VBox(buttons))


## 17. ทางเลือก: จับเวลา postprocess ซ้ำ (ไม่รัน GAN ซ้ำ)
ตั้ง `RUN_POSTPROCESS_BENCHMARK=True` ถ้าต้องการ mean / SD / median / min / max ของ postprocess ด้วยภาพเดิม
warmup หนึ่งรอบต่อโมเดลไม่นับในสถิติ แล้ววัด 3 รอบสลับลำดับโมเดล; ไม่เปลี่ยนภาพที่นำไปวัดเมตริก ไม่รวมการประเมิน/บันทึกไฟล์ในเวลา ค่านี้ไม่ใช่ full-pipeline benchmark และไม่ใช่ความแปรปรวนข้ามภาพ
หากต้องการผลวิจัย ให้ใช้หลายชุดภาพคู่เดียวกันทั้งสองโมเดล เก็บ per-case delta แล้วสรุปข้ามกรณี แยกจากการรันภาพเดิมซ้ำเพื่อจับเวลา

In [ ]:
RUN_POSTPROCESS_BENCHMARK=False
BENCHMARK_REPEATS=3
if RUN_POSTPROCESS_BENCHMARK:
    if BENCHMARK_REPEATS<3:raise ValueError('ใช้ repeats อย่างน้อย 3')
    benchmark_rows=[]
    model_names=list(RESULTS)
    for model in model_names:
        warm=run_detail_core(before=RESULTS[model]['before'],**PROFILE)
        del warm
    for repeat in range(BENCHMARK_REPEATS):
        order=model_names if repeat%2==0 else model_names[::-1]
        for model in order:
            measured=run_detail_core(before=RESULTS[model]['before'],**PROFILE)
            benchmark_rows.append(dict(model=model,repeat=repeat+1,
                **{k:measured['timings_seconds'][k] for k in ['mask','diffusion','raw_background_finish','native_sharp','total_processing']}))
            del measured
    benchmark=pd.DataFrame(benchmark_rows)
    benchmark.to_csv(EVAL_DIR/'postprocess_repeated_timings.csv',index=False)
    stats=benchmark.groupby('model')[['mask','diffusion','raw_background_finish','native_sharp','total_processing']].agg(['mean','std','median','min','max','count'])
    stats.to_csv(EVAL_DIR/'postprocess_repeated_summary.csv');display(stats.round(3))
else:
    print('ข้ามการรันซ้ำ; ตารางส่วน 13 แสดงเวลาจริงของรอบที่สร้างภาพแล้ว')


## 18. เปลี่ยนรูปชุดใหม่ — รันเซลล์นี้ครั้งเดียว
หลังรันครั้งแรกเสร็จ ตั้ง `RUN_NEW_IMAGES=True` แล้วรันเซลล์นี้ ระบบเรียกส่วน 5–17 ตามลำดับ อัปโหลดหน้า/ทรง/สีใหม่ แล้วแสดงภาพ เวลา และเมตริกใหม่ ใช้ runtime และโมเดลเดิม
เซลล์นี้ฝังสำเนาโค้ดส่วน 5–17 ในไฟล์รุ่นนี้ หากแก้ค่าหรือโค้ดในส่วนต้นด้วยตัวเอง ให้รันส่วน 5–17 โดยตรงเพื่อใช้การแก้ไขนั้น
ใช้ได้หลังโหลด B24 ด้วยส่วน 4 ของไฟล์รุ่นนี้แล้ว

In [ ]:
RUN_NEW_IMAGES=False
NEW_RUN_STEPS=[(11, 'import io, uuid, gc\nimport numpy as np\nfrom PIL import Image, ImageOps\nfrom datetime import datetime\nfrom zoneinfo import ZoneInfo\nfrom google.colab import files\nfrom IPython.display import display\n\nINPUT_PATHS = []  # ["/content/drive/.../face.jpg", ".../shape.jpg", ".../color.jpg"]\nSEED = 3407\nSTRENGTH = 0.15\nSTEPS = 30\nGUIDANCE = 2.5\nFACE_BLEND = 0.35\nCOMPARE_CROP = None  # (left, top, right, bottom); None = เลือกจาก union hair masks\nAUTO_DOWNLOAD_IMAGES = False  # True = ดาวน์โหลด 6 รูป; browser ต้องอนุญาตหลายไฟล์\nif globals().get(\'_HFG_LAYOUT_OWNER\') != id(hair_fast) or not globals().get(\'_HFG_DEVICE_LAYOUT\'):\n    raise RuntimeError(\'ใช้ส่วน 4 ของไฟล์รุ่นใหม่โหลด B24 ก่อน เพื่อบันทึก device layout ก่อน offload\')\npark_postprocess()\nrestore_hfg_devices(_HFG_DEVICE_LAYOUT)\n# Invalidate only per-case outputs in memory; prior Drive folders stay intact.\nfor _name in [\'RESULTS\',\'EXPORT_IMAGES\',\'TIMINGS\',\'METRIC_FRAME\',\'DELTA_FRAME\',\'TIME_SUMMARY\',\'TIME_DETAIL\']:\n    globals().pop(_name,None)\nif not 0 <= int(SEED) <= 4294967295: raise ValueError(\'seed ไม่ถูกต้อง\')\nRUN_ID=datetime.now(ZoneInfo(\'Asia/Bangkok\')).strftime(\'%Y%m%d_%H%M%S\')+\'_\'+uuid.uuid4().hex[:8]\nRUN_DIR=RESULTS_ROOT/RUN_ID\nRUN_DIR.mkdir(parents=True, exist_ok=False)\nINPUT_DIR=RUN_DIR/\'inputs_original\'; INPUT_DIR.mkdir()\nCROP_DIR=RUN_DIR/\'inputs_aligned\'; CROP_DIR.mkdir()\nBASE_DIR=RUN_DIR/\'base_outputs\'; BASE_DIR.mkdir()\nif INPUT_PATHS and len(INPUT_PATHS)!=3: raise ValueError(\'ต้องระบุ 3 paths ตามลำดับ หน้า/ทรง/สี\')\nINPUTS=[]\nfor i,(role,label) in enumerate([(\'face\',\'รูปหน้า\'),(\'shape\',\'รูปทรงผม\'),(\'color\',\'รูปสีผม\')]):\n    if INPUT_PATHS:\n        with Image.open(INPUT_PATHS[i]) as im: normalized=ImageOps.exif_transpose(im).convert(\'RGB\')\n    else:\n        print(\'อัปโหลด \'+label+\' เพียง 1 ภาพ\')\n        uploaded=files.upload()\n        if len(uploaded)!=1: raise ValueError(\'กรุณาเลือกครั้งละ 1 ภาพ\')\n        with Image.open(io.BytesIO(next(iter(uploaded.values())))) as im:\n            normalized=ImageOps.exif_transpose(im).convert(\'RGB\')\n        del uploaded\n    path=INPUT_DIR/(role+\'.png\'); normalized.save(path); INPUTS.append(path)\n\ndef tensor_to_pil(value, value_range=\'zero_one\'):\n    if isinstance(value, Image.Image):\n        return value.convert(\'RGB\')\n    if torch.is_tensor(value):\n        t = value.detach().cpu()\n        if t.ndim == 4 and t.shape[0] == 1:\n            t = t[0]\n        if t.ndim != 3 or t.shape[0] != 3:\n            raise ValueError(f\'Unexpected tensor shape: {tuple(t.shape)}\')\n        a = t.permute(1, 2, 0).numpy()\n    else:\n        a = np.asarray(value)\n    if a.dtype == np.uint8:\n        return Image.fromarray(a).convert(\'RGB\')\n    if value_range == \'minus_one_one\':\n        a = (a + 1) / 2\n    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:\n        raise ValueError(\'Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ\')\n    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert(\'RGB\')\n\ndef clock():\n    torch.cuda.synchronize()\n    return time.perf_counter()\n# Device restore is done above, before timing the swap.\n_HFG_OFFLOADED=[]\nt=clock()\nwith torch.inference_mode():\n    swapped=hair_fast.swap(*(str(p) for p in INPUTS),align=True,seed=int(SEED))\nHFG_SECONDS=clock()-t\nif not isinstance(swapped,tuple) or len(swapped)!=4: raise RuntimeError(\'swap return format ไม่ตรงรุ่น\')\ntensor_to_pil(swapped[0]).save(BASE_DIR/\'HairFastGAN_B24.png\')\nfor role,value in zip((\'face\',\'shape\',\'color\'),swapped[1:]):\n    image=tensor_to_pil(value)\n    if image.size!=(1024,1024): raise ValueError(\'Aligned crop ต้องเป็น 1024x1024\')\n    image.save(CROP_DIR/(role+\'.png\'))\ndel swapped, value\n# ย้ายโมเดล HFG ลง CPU ก่อนเปิด subprocess Barbershop ป้องกัน VRAM ซ้อน\n# เดินเฉพาะ object tree ของ HFG; nn.Module ย้ายลูกทั้งหมดด้วย .cpu()\ndef offload_hfg(root):\n    import types\n    seen=set(); moved=[]\n    def walk(obj):\n        if id(obj) in seen: return\n        seen.add(id(obj))\n        if isinstance(obj,torch.nn.Module):\n            # CPU + CUDA in one module is valid. .cpu() moves all registered children.\n            obj.cpu()\n            moved.append(obj)\n        elif isinstance(obj,dict):\n            for value in obj.values(): walk(value)\n        elif isinstance(obj,(list,tuple)):\n            for value in obj: walk(value)\n        elif hasattr(obj,\'__dict__\') and not isinstance(obj,(type,types.ModuleType,types.FunctionType,types.MethodType)):\n            for value in vars(obj).values(): walk(value)\n    walk(root)\n    return moved\n_HFG_OFFLOADED=offload_hfg(hair_fast)\ngc.collect();torch.cuda.empty_cache()\nprint(\'HFG seconds:\',round(HFG_SECONDS,2),\'| offloaded modules:\',len(_HFG_OFFLOADED))\nprint(\'ผลชุดนี้:\',RUN_DIR)\ndisplay(Image.open(BASE_DIR/\'HairFastGAN_B24.png\'))\n'), (13, '# Barbershop stages use the SAME Python/Torch/CUDA runtime as HairFastGAN via subprocess isolation.\nBARBER_REV=\'088d7d0f0623b348732fc15d8fbdea8b8cb730c7\'\nBARBER_REPO=Path(\'/content/Barbershop_baseline_comparison\')\nBARBER_ROOT=RUN_DIR/\'barbershop_frontend\';BARBER_ROOT.mkdir(exist_ok=True)\nsha=sha_file\nBARBER_WEIGHTS=Path(\'/content/drive/MyDrive/Barbershop_matched_weights\');BARBER_WEIGHTS.mkdir(exist_ok=True)\nif not (BARBER_REPO/\'.git\').exists():subprocess.run([\'git\',\'clone\',\'https://github.com/ZPdesu/Barbershop.git\',str(BARBER_REPO)],check=True)\n# No destructive checkout of unrelated work: this is a dedicated experiment checkout.\nhead=subprocess.check_output([\'git\',\'-C\',str(BARBER_REPO),\'rev-parse\',\'HEAD\'],text=True).strip()\nif head!=BARBER_REV:\n    assert not subprocess.check_output([\'git\',\'-C\',str(BARBER_REPO),\'status\',\'--porcelain\'],text=True).strip(), \'Dedicated checkout has modifications at another commit\'\n    subprocess.run([\'git\',\'-C\',str(BARBER_REPO),\'checkout\',BARBER_REV],check=True)\nimport gdown, ast, platform\n# Download once; use the SAME source weights through a local/Drive path or manual upload on failure.\n# Optional: fill paths to the original Barbershop weights from your earlier notebook.\nLOCAL_WEIGHT_PATHS = {\'ffhq.pt\': \'\', \'seg.pth\': \'\'}\nWEIGHT_IDS = {\'ffhq.pt\': \'1NVlMTYO4wyXZYVGVlGjP_xoCzBlfXSR1\',\n              \'seg.pth\': \'1bDOOJpST8dJDcSPuPKpTbdAmLKUGUji5\'}\n\ndef check_barber_weight(path, name):\n    if not path.is_file() or path.stat().st_size < 1000000:\n        raise ValueError(\'Missing/incomplete weight file: \'+str(path))\n    # Only open original weights from the sources above or your trusted existing Barbershop copy.\n    obj=torch.load(path,map_location=\'cpu\',weights_only=False)\n    if name==\'ffhq.pt\':\n        if not isinstance(obj,dict) or not {\'g_ema\',\'latent_avg\'}<=set(obj):\n            raise ValueError(\'Not a Barbershop ffhq.pt checkpoint (requires g_ema and latent_avg)\')\n        if not isinstance(obj[\'g_ema\'],dict) or not torch.is_tensor(obj[\'latent_avg\']):\n            raise ValueError(\'Invalid generator checkpoint structure\')\n    elif not isinstance(obj,dict) or not any(torch.is_tensor(v) for v in obj.values()):\n        raise ValueError(\'Expected original Barbershop segmentation state_dict\')\n    del obj\n\nfor name,drive_id in WEIGHT_IDS.items():\n    target=BARBER_WEIGHTS/name\n    if target.is_file():\n        check_barber_weight(target,name)\n        print(\'Using cached weight:\',target)\n        continue\n    temp=BARBER_WEIGHTS/(name+\'.download\')\n    supplied=LOCAL_WEIGHT_PATHS[name].strip()\n    if supplied:\n        source=Path(supplied).expanduser()\n        check_barber_weight(source,name)\n        shutil.copy2(source,temp)\n    else:\n        try:\n            result=gdown.download(id=drive_id,output=str(temp),quiet=False)\n            if result is None:raise RuntimeError(\'gdown returned no file\')\n            check_barber_weight(temp,name)\n        except Exception as err:\n            print(\'Automatic download unavailable for\',name,\':\',type(err).__name__)\n            print(\'Original source: https://drive.google.com/file/d/\'+drive_id+\'/view\')\n            print(\'Use this original file or your existing Barbershop copy. Do not substitute another model checkpoint.\')\n            supplied=input(\'Paste the full /content/drive/... path to \'+name+\'; or press Enter to upload it: \').strip().strip(\'"\')\n            if supplied:\n                source=Path(supplied).expanduser();check_barber_weight(source,name)\n                shutil.copy2(source,temp)\n            else:\n                from google.colab import files as colab_files\n                # Upload into a dedicated folder; avoid overwriting unrelated notebook files.\n                import tempfile\n                with tempfile.TemporaryDirectory(prefix=\'barber_weight_\') as upload_dir:\n                    before=os.getcwd()\n                    try:\n                        os.chdir(upload_dir);uploaded_weights=colab_files.upload()\n                    finally:os.chdir(before)\n                    if len(uploaded_weights)!=1:raise ValueError(\'Upload only \'+name+\' in this dialog\')\n                    source=Path(upload_dir)/next(iter(uploaded_weights))\n                    check_barber_weight(source,name);shutil.copy2(source,temp)\n                    del uploaded_weights\n    check_barber_weight(temp,name)\n    os.replace(temp,target)\n    print(\'Saved validated weight:\',target)\n# Record byte identities; structural validation alone does not establish source provenance.\nweight_hashes={name:sha(BARBER_WEIGHTS/name) for name in WEIGHT_IDS}\nweight_lock=BARBER_WEIGHTS/\'weight_hashes.json\'\nif weight_lock.exists():\n    assert json.loads(weight_lock.read_text())==weight_hashes, \'Cached weights changed; do not mix checkpoints within an experiment\'\nelse:weight_lock.write_text(json.dumps(weight_hashes,indent=2))\n\nEXPECTED_BARBER_HASHES = {\'ffhq.pt\': \'1a0c25c17013e0bb99bd3de4e80ea52cb3efc2b71224ff902e0a29bafac9c531\', \'seg.pth\': \'590a79b3bea1f45e22d39e36f9dc80434075367c2e8d131ca3f5275e54dd3c00\'}\nif weight_hashes != EXPECTED_BARBER_HASHES:\n    raise RuntimeError(\'Barbershop weights ไม่ตรงรายงานที่เลือก Baseline: ตรวจไฟล์เดิมใน Drive\')\n\n# Use the repository\'s fixed PCA; do not rebuild a new random PCA each trial.\npca=BARBER_WEIGHTS/\'ffhq_PCA.npz\'\nif not pca.is_file():shutil.copy2(BARBER_REPO/\'pretrained_models/ffhq_PCA.npz\',pca)\nassert sha(pca)==sha(BARBER_REPO/\'pretrained_models/ffhq_PCA.npz\')\nweights=torch.load(BARBER_WEIGHTS/\'ffhq.pt\',map_location=\'cpu\',weights_only=False)\nassert {\'g_ema\',\'latent_avg\'}<=set(weights);del weights\n\n# Use the same native-PyTorch operator implementation used by HairFastGAN.\npatches=[]\nfor kind in [\'fused_act\',\'upfirdn2d\']:\n    path=BARBER_REPO/\'models/stylegan2/op\'/(kind+\'.py\')\n    original=subprocess.check_output([\'git\',\'-C\',str(BARBER_REPO),\'show\',BARBER_REV+\':models/stylegan2/op/\'+kind+\'.py\'],text=True)\n    patched=_native_stylegan_source(original,kind)\n    compile(patched,str(path),\'exec\');path.write_text(patched)\n    patches.append(dict(file=str(path.relative_to(BARBER_REPO)),sha256=sha(path)))\n\n# Instrument existing optimizer steps without changing losses or latent update rules.\nclass LogOptimizerSteps(ast.NodeTransformer):\n    def __init__(self,stage):self.stage=stage;self.counter=0\n    def visit_Expr(self,node):\n        self.generic_visit(node)\n        call=node.value\n        if isinstance(call,ast.Call) and isinstance(call.func,ast.Attribute) and call.func.attr==\'step\' and isinstance(call.func.value,ast.Name) and call.func.value.id in [\'optimizer_W\',\'optimizer_FS\',\'optimizer_align\',\'opt_blend\']:\n            self.counter+=1\n            log=ast.parse(f"record(\'{self.stage}_{self.counter}\', locals().get(\'step\'), locals().get(\'loss\'))").body[0]\n            return [node,ast.copy_location(log,node)]\n        return node\nfor name,expected in [(\'Embedding\',2),(\'Alignment\',4),(\'Blending\',1)]:\n    rel=\'models/\'+name+\'.py\';original=subprocess.check_output([\'git\',\'-C\',str(BARBER_REPO),\'show\',BARBER_REV+\':\'+rel],text=True)\n    transformer=LogOptimizerSteps(name);tree=transformer.visit(ast.parse(original));assert transformer.counter==expected,(name,transformer.counter)\n    tree.body.insert(0,ast.ImportFrom(module=\'experiment_trace\',names=[ast.alias(name=\'record\')],level=0))\n    ast.fix_missing_locations(tree);code=ast.unparse(tree);compile(code,rel,\'exec\');(BARBER_REPO/rel).write_text(code)\n    patches.append(dict(file=rel,sha256=sha(BARBER_REPO/rel)))\n\n(BARBER_REPO/\'experiment_worker.py\').write_text(\'"""Runs only inside the pinned Barbershop checkout in a separate Python process."""\\nimport os,sys,json,time,random,hashlib,copy,gc,ast,argparse\\nfrom pathlib import Path\\nimport numpy as np\\nimport torch\\nfrom PIL import Image\\nif not hasattr(Image,\\\'ANTIALIAS\\\'):Image.ANTIALIAS=Image.Resampling.LANCZOS\\ncfg=json.loads(Path(sys.argv[1]).read_text())\\n# Read the original CLI defaults without executing its main pipeline.\\nsource=Path(\\\'main.py\\\').read_text();tree=ast.parse(source)\\nblock=next(n for n in tree.body if isinstance(n,ast.If) and \\\'__name__\\\' in ast.unparse(n.test))\\nparser_nodes=[]\\nfor node in block.body:\\n    if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\\\'args\\\' for t in node.targets):break\\n    parser_nodes.append(node)\\nns={\\\'argparse\\\':argparse};exec(compile(ast.Module(body=parser_nodes,type_ignores=[]),\\\'defaults\\\',\\\'exec\\\'),ns)\\nopts=ns[\\\'parser\\\'].parse_args([])\\nopts.ckpt=cfg[\\\'ckpt\\\'];opts.seg_ckpt=cfg[\\\'seg\\\'];opts.seed=int(cfg.get(\\\'seed\\\',3407))\\nopts.W_steps=1100;opts.FS_steps=250;opts.align_steps1=140;opts.align_steps2=100;opts.blend_steps=400\\nopts.save_intermediate=False;opts.verbose=False;opts.sign=\\\'realistic\\\';opts.smooth=5\\n# All torch.load calls are restricted to the trusted local model files and repository LPIPS weights.\\n_original_load=torch.load\\ndef legacy_load(*args,**kwargs):\\n    kwargs.setdefault(\\\'weights_only\\\',False)\\n    return _original_load(*args,**kwargs)\\ntorch.load=legacy_load\\nfrom models.Embedding import Embedding\\nfrom models.Alignment import Alignment\\nfrom models.Blending import Blending\\nimport experiment_trace as trace\\n\\ndef seed(key):\\n    n=(int(cfg.get(\\\'seed\\\',3407))+int(hashlib.sha256(key.encode()).hexdigest()[:8],16))%(2**31)\\n    random.seed(n);np.random.seed(n);torch.manual_seed(n);torch.cuda.manual_seed_all(n)\\n    torch.backends.cudnn.benchmark=False\\n    torch.backends.cudnn.deterministic=True\\n    torch.backends.cuda.matmul.allow_tf32=False\\n    torch.backends.cudnn.allow_tf32=False\\n\\ndef sha(p):\\n    h=hashlib.sha256()\\n    with open(p,\\\'rb\\\') as f:\\n        for chunk in iter(lambda:f.read(8*1024*1024),b\\\'\\\'):h.update(chunk)\\n    return h.hexdigest()\\ndef fingerprint(module):\\n    h=hashlib.sha256()\\n    for k,v in module.state_dict().items():h.update(k.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())\\n    return h.hexdigest()\\ndef atomic(obj,p):\\n    p=Path(p);tmp=p.with_suffix(\\\'.tmp\\\');tmp.write_text(json.dumps(obj,indent=2));os.replace(tmp,p)\\ndef seconds(fn):\\n    torch.cuda.synchronize();tick=time.perf_counter();result=fn();torch.cuda.synchronize()\\n    return time.perf_counter()-tick,result\\n\\nroot=Path(cfg[\\\'root\\\']);inv=root/\\\'inversion\\\';inv.mkdir(exist_ok=True)\\ntrace.configure(root/\\\'optimization_steps.csv\\\')\\nseed(\\\'initialization\\\')\\nopts.learning_rate=.01;opts.output_dir=str(inv)\\nembedding=Embedding(copy.deepcopy(opts))\\n# Use fixed pretrained noise in both frozen preprocessing and subsequent shared-weight decoding.\\nimport functools\\nembedding.net.generator.forward=functools.partial(embedding.net.generator.forward,randomize_noise=False)\\ninitial=fingerprint(embedding.net.generator)\\n# Reuse the exact same generator for all modules; these weights remain frozen.\\naligner=Alignment(copy.deepcopy(opts),net=embedding.net)\\nblender=Blending(copy.deepcopy(opts),net=embedding.net)\\nfor module in [embedding.loss_builder,aligner.loss_builder,blender.loss_builder]:\\n    module.eval().requires_grad_(False)\\n# No GPU training of shared network weights occurs.\\natomic(dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),gpu=torch.cuda.get_device_name(0),defaults=vars(opts),weights_sha256=sha(cfg[\\\'ckpt\\\']),seg_sha256=sha(cfg[\\\'seg\\\']),pca_sha256=sha(cfg[\\\'pca\\\'])),root/\\\'worker_environment.json\\\')\\nids=sorted({r[k] for r in cfg[\\\'cases\\\'] for k in [\\\'face\\\',\\\'shape\\\',\\\'color\\\']})\\nfor ident in ids:\\n    marker=inv/(ident+\\\'_complete.json\\\');image=str(Path(cfg[\\\'input_dir\\\'])/(ident+\\\'.png\\\'))\\n    if marker.exists():\\n        previous=json.loads(marker.read_text());assert previous[\\\'input_sha256\\\']==sha(image)\\n        continue\\n    seed(\\\'inversion/\\\'+ident);trace.context(model=\\\'shared_inversion\\\',case=ident,operation=\\\'inversion\\\')\\n    wsec,_=seconds(lambda:embedding.invert_images_in_W([image]))\\n    fsec,_=seconds(lambda:embedding.invert_images_in_FS([image]))\\n    trace.flush()\\n    atomic(dict(input_sha256=sha(image),W_seconds=wsec,FS_seconds=fsec,total_seconds=wsec+fsec,W_steps=opts.W_steps,FS_steps=opts.FS_steps),marker)\\n\\n# ALIGNMENT_SAVE_GUARD_V1: preserve exact computed latents, never substitute another pair.\\n_alignment_latents={}\\n_original_save_align=aligner.save_align_results\\ndef checked_save_align(im_name_1,im_name_2,sign,gen_im,latent_in,latent_F,save_intermediate=True):\\n    _original_save_align(im_name_1,im_name_2,sign,gen_im,latent_in,latent_F,save_intermediate=save_intermediate)\\n    path=Path(aligner.opts.output_dir)/(\\\'Align_\\\'+sign)/(im_name_1+\\\'_\\\'+im_name_2+\\\'.npz\\\')\\n    payload=dict(latent_in=latent_in.detach().cpu().numpy().copy(),latent_F=latent_F.detach().cpu().numpy().copy())\\n    _alignment_latents[str(path)]=payload\\n    write_checked_alignment(path,payload)\\ndef write_checked_alignment(path,payload):\\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\\n    if not all(np.isfinite(v).all() for v in payload.values()):raise RuntimeError(\\\'Nonfinite alignment: \\\'+str(path))\\n    tmp=path.with_name(path.name+\\\'.pending\\\')\\n    with tmp.open(\\\'wb\\\') as stream:\\n        np.savez(stream,**payload);stream.flush();os.fsync(stream.fileno())\\n    os.replace(tmp,path)\\n    with np.load(path,allow_pickle=False) as saved:\\n        if set(saved.files)!=set(payload) or any(not np.array_equal(saved[k],v) for k,v in payload.items()):\\n            raise RuntimeError(\\\'Alignment write verification failed: \\\'+str(path))\\n    print(\\\'Verified aligned latent:\\\',path,flush=True)\\naligner.save_align_results=checked_save_align\\n\\ndef ensure_alignment_pair(path):\\n    path=Path(path)\\n    if str(path) not in _alignment_latents:raise RuntimeError(\\\'Alignment did not produce expected pair: \\\'+str(path))\\n    # Re-persist both exact computed pairs before blending; avoids stale/missing Drive entries.\\n    write_checked_alignment(path,_alignment_latents[str(path)])\\n\\nrows=[]\\nfor spec in cfg[\\\'trials\\\']:\\n    for row in cfg[\\\'cases\\\']:\\n        key=row[\\\'key\\\'];dest=root/spec[\\\'name\\\']/key;dest.mkdir(parents=True,exist_ok=True)\\n        result_path=dest/\\\'complete.json\\\'\\n        if result_path.exists():\\n            old=json.loads(result_path.read_text());assert old[\\\'spec\\\']==spec\\n            assert Path(old[\\\'final_float\\\']).is_file() and Path(old[\\\'final_png\\\']).is_file(), \\\'Completed result files missing\\\'\\n            rows.append(old);continue\\n        # Shared inversion bytes are read-only and linked into each isolated case directory.\\n        for sub in [\\\'W+\\\',\\\'FS\\\']:\\n            link=dest/sub\\n            if not link.exists():link.symlink_to(inv/sub,target_is_directory=True)\\n        a=copy.deepcopy(opts);a.output_dir=str(dest);a.learning_rate=spec[\\\'align_lr\\\'];aligner.opts=a;aligner.loss_builder.opt=a\\n        b=copy.deepcopy(opts);b.output_dir=str(dest);b.learning_rate=spec[\\\'blend_lr\\\'];blender.opts=b;blender.loss_builder.opt=b\\n        f,s,c=[str(Path(cfg[\\\'input_dir\\\'])/(row[k]+\\\'.png\\\')) for k in [\\\'face\\\',\\\'shape\\\',\\\'color\\\']]\\n        seed(\\\'case/\\\'+key) # same latent/noise initialization for all trials of this case\\n        trace.context(model=spec[\\\'name\\\'],case=key,operation=\\\'align_shape\\\')\\n        align1,_=seconds(lambda:aligner.align_images(f,s,sign=\\\'realistic\\\',align_more_region=False,smooth=5))\\n        trace.context(model=spec[\\\'name\\\'],case=key,operation=\\\'align_color\\\')\\n        align2=0.0\\n        if s!=c:\\n            align2,_=seconds(lambda:aligner.align_images(f,c,sign=\\\'realistic\\\',align_more_region=False,smooth=5,save_intermediate=False))\\n        for ref in sorted({s,c}):\\n            ensure_alignment_pair(dest/\\\'Align_realistic\\\'/(Path(f).stem+\\\'_\\\'+Path(ref).stem+\\\'.npz\\\'))\\n        trace.context(model=spec[\\\'name\\\'],case=key,operation=\\\'blend\\\')\\n        # Capture the float output before PNG quantization for the common evaluator.\\n        capture={};original_save=blender.save_blend_results\\n        def save_float(*args,**kwargs):\\n            capture[\\\'full\\\']=((args[4].detach().float().cpu()+1)/2).clamp(0,1).numpy()\\n            return original_save(*args,**kwargs)\\n        blender.save_blend_results=save_float\\n        try:blendsec,_=seconds(lambda:blender.blend_images(f,s,c,sign=\\\'realistic\\\'))\\n        finally:blender.save_blend_results=original_save\\n        np.save(dest/\\\'final_float.npy\\\',capture[\\\'full\\\']);del capture\\n        from PIL import Image\\n        png=dest/f\\\'{Path(f).stem}_{Path(s).stem}_{Path(c).stem}_realistic.png\\\'\\n        assert png.is_file()\\n        inv_seconds=sum(json.loads((inv/(ident+\\\'_complete.json\\\')).read_text())[\\\'total_seconds\\\'] for ident in sorted({row[k] for k in [\\\'face\\\',\\\'shape\\\',\\\'color\\\']}))\\n        result=dict(case=key,split=cfg[\\\'split\\\'],model=spec[\\\'name\\\'],spec=spec,align_shape_seconds=align1,align_color_seconds=align2,blend_seconds=blendsec,inversion_cold_seconds=inv_seconds,cached_pipeline_seconds=align1+align2+blendsec,estimated_cold_compute_seconds=inv_seconds+align1+align2+blendsec,final_float=str(dest/\\\'final_float.npy\\\'),final_png=str(png))\\n        trace.flush()\\n        atomic(result,result_path);rows.append(result)\\n        atomic(rows,root/\\\'results.json\\\');gc.collect();torch.cuda.empty_cache()\\nassert fingerprint(embedding.net.generator)==initial,\\\'Frozen generator changed\\\'\\natomic(rows,root/\\\'results.json\\\')\\natomic(dict(status=\\\'complete\\\',frozen_generator_unchanged=True,cases=len(cfg[\\\'cases\\\']),trials=len(cfg[\\\'trials\\\']),rows=len(rows)),root/\\\'completion.json\\\')\\n\')\n(BARBER_REPO/\'experiment_trace.py\').write_text("import csv,torch,time\\n_handle=None\\n_writer=None\\n_context={}\\ndef configure(path):\\n    global _handle,_writer\\n    fields=[\'model\',\'case\',\'operation\',\'stage\',\'step\',\'loss\']\\n    # Keep only completed work when retrying; partial cases restart from the shared inversion.\\n    if path.exists():\\n        with open(path,newline=\'\') as src:rows=list(csv.DictReader(src))\\n        def complete(r):\\n            marker=(path.parent/\'inversion\'/(r[\'case\']+\'_complete.json\')) if r[\'model\']==\'shared_inversion\' else (path.parent/r[\'model\']/r[\'case\']/\'complete.json\')\\n            return marker.is_file()\\n        temp=path.with_suffix(\'.tmp\')\\n        with open(temp,\'w\',newline=\'\') as dst:\\n            writer=csv.DictWriter(dst,fieldnames=fields);writer.writeheader();writer.writerows(r for r in rows if complete(r))\\n        temp.replace(path)\\n    exists=path.exists() and path.stat().st_size>0\\n    _handle=open(path,\'a\',newline=\'\')\\n    _writer=csv.DictWriter(_handle,fieldnames=fields)\\n    if not exists:_writer.writeheader()\\ndef flush():\\n    if _handle:_handle.flush()\\ndef context(**kwargs):\\n    global _context\\n    _context=kwargs\\n    flush()\\ndef record(stage,step,loss):\\n    if _writer is None:return\\n    value=float(loss.detach()) if torch.is_tensor(loss) else float(loss)\\n    if not __import__(\'math\').isfinite(value):raise RuntimeError(\'Nonfinite optimization loss\')\\n    _writer.writerow(dict(**_context,stage=stage,step=step,loss=value))\\n")\n\nALIGNMENT_GUARD_PREVIOUS_WORKER_SHA256=\'5baf0074bda54939722b5cc374e8a4b2233332ba283451c63a85eea50d2ce2f8\'\n'), (15, "BARBER_CASE=dict(key='comparison_000',face='face',shape='shape',color='color')\nBARBER_CONFIG=dict(root=str(BARBER_ROOT),input_dir=str(CROP_DIR),\n    ckpt=str(BARBER_WEIGHTS/'ffhq.pt'),seg=str(BARBER_WEIGHTS/'seg.pth'),\n    pca=str(pca),trials=[dict(name='BS_Pretrained',align_lr=.01,blend_lr=.01)],\n    cases=[BARBER_CASE],split='user_comparison',seed=int(SEED),\n    crop_hashes={role:sha_file(CROP_DIR/(role+'.png')) for role in ['face','shape','color']},\n    weight_hashes=weight_hashes,revision=BARBER_REV,\n    worker_sha256=sha_file(BARBER_REPO/'experiment_worker.py'),patches=patches)\nconfig_path=BARBER_ROOT/'worker_config.json'\nif config_path.exists():\n    previous=json.loads(config_path.read_text())\n    if previous!=BARBER_CONFIG:\n        old_fields={k:v for k,v in previous.items() if k!='worker_sha256'}\n        new_fields={k:v for k,v in BARBER_CONFIG.items() if k!='worker_sha256'}\n        compatible=(old_fields==new_fields and previous.get('worker_sha256')==ALIGNMENT_GUARD_PREVIOUS_WORKER_SHA256)\n        if not compatible or (BARBER_ROOT/'completion.json').exists():\n            raise RuntimeError('Config เปลี่ยนหรือเป็นผลที่เสร็จแล้ว ห้ามใช้ cache ปนกัน ให้เริ่มชุดใหม่')\n        backup=config_path.with_name('worker_config.before_alignment_guard.json')\n        if not backup.exists():backup.write_text(json.dumps(previous,indent=2))\n        config_path.write_text(json.dumps(BARBER_CONFIG,indent=2))\n        print('อัปเดตเฉพาะ alignment save guard; คง input/weights/settings และ inversion cache เดิม')\nelse: config_path.write_text(json.dumps(BARBER_CONFIG,indent=2))\nt=clock()\nwith (BARBER_ROOT/'worker.log').open('a') as log:\n    proc=subprocess.Popen([sys.executable,str(BARBER_REPO/'experiment_worker.py'),str(config_path)],\n        cwd=BARBER_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,\n        bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1'})\n    for line in proc.stdout:\n        log.write(line);log.flush();print(line,end='')\n    returncode=proc.wait()\nif returncode: raise RuntimeError('Barbershop ล้มเหลว ดู worker.log ด้านบน แล้วรันส่วน 7 ซ้ำ')\nBARBER_WALL_SECONDS=clock()-t\ncompletion=json.loads((BARBER_ROOT/'completion.json').read_text())\nassert completion['rows']==1 and completion['frozen_generator_unchanged']\nBARBER_RESULT=json.loads((BARBER_ROOT/'results.json').read_text())[0]\n# ใช้ float output จาก worker แล้ว quantize แบบเดียวกับ HFG เพื่อไม่ต่างกันจาก PNG writer\nraw_bs=np.load(BARBER_RESULT['final_float'])\nif raw_bs.ndim!=4 or raw_bs.shape[:2]!=(1,3): raise ValueError('Barbershop tensor format ไม่ตรงรุ่น')\nbarber_image=tensor_to_pil(torch.from_numpy(raw_bs))\nif barber_image.size!=(1024,1024): raise ValueError('Barbershop output ต้องเป็น 1024x1024')\nbarber_image.save(BASE_DIR/'Barbershop_Baseline.png')\ndel raw_bs,barber_image\nprint('Barbershop เสร็จ | wall time รอบนี้:',round(BARBER_WALL_SECONDS,2),'วินาที')\nprint('เวลาคำนวณรวม inversion ของผลนี้:',round(BARBER_RESULT['estimated_cold_compute_seconds'],2),'วินาที')\ndisplay(Image.open(BASE_DIR/'Barbershop_Baseline.png'))\n"), (17, "import numpy as np\nfrom PIL import Image, ImageOps, ImageFilter, ImageDraw\nfrom transformers import SegformerImageProcessor, SegformerForSemanticSegmentation\nfrom diffusers import StableDiffusionInpaintPipeline, UniPCMultistepScheduler\n\nDEVICE = 'cuda'\nSEG_MODEL_ID = 'mattmdjaga/segformer_b2_clothes'\nDIFFUSION_MODEL_ID = 'stablediffusionapi/realistic-vision-v51'\nSEG_REVISION = 'main'           # เปลี่ยนเป็น commit hash เพื่อทำซ้ำข้าม session\nDIFFUSION_REVISION = 'main'\nDIFFUSION_CPU_OFFLOAD = True\nseg_key=(SEG_MODEL_ID,SEG_REVISION,DEVICE)\nif globals().get('_SEG_LOADED_KEY') != seg_key or 'seg_model' not in globals():\n    seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID, revision=SEG_REVISION)\n    seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID, revision=SEG_REVISION).to(DEVICE).eval()\n    _SEG_LOADED_KEY=seg_key\nelse:\n    seg_model.to(DEVICE).eval()\n    print('คืน segmentation จาก RAM ไป GPU แล้ว')\nlabel_ids = {str(name).lower(): int(i) for i, name in seg_model.config.id2label.items()}\nHAIR_ID, FACE_ID = label_ids['hair'], label_ids['face']\nassert (HAIR_ID, FACE_ID) == (2, 11), 'ตรวจ mapping ของโมเดลก่อนทดลอง'\ndiffusion_key=(DIFFUSION_MODEL_ID,DIFFUSION_REVISION,DIFFUSION_CPU_OFFLOAD,DEVICE)\nif globals().get('_DIFFUSION_LOADED_KEY') != diffusion_key or 'inpaint_pipe' not in globals():\n    # # ไม่ปิด safety checker ที่โมเดลมีมา\n    # inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(\n    #     DIFFUSION_MODEL_ID, revision=DIFFUSION_REVISION, torch_dtype=torch.float16)\n\n    # ปิด Safety Checker\n    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(\n        DIFFUSION_MODEL_ID,\n        revision=DIFFUSION_REVISION,\n        torch_dtype=torch.float16,\n        safety_checker=None,\n    )\n\n    inpaint_pipe.scheduler = UniPCMultistepScheduler.from_config(inpaint_pipe.scheduler.config)\n    if DIFFUSION_CPU_OFFLOAD:\n        inpaint_pipe.enable_model_cpu_offload()\n    else:\n        inpaint_pipe.to(DEVICE)\n    # รองรับทั้ง pipeline API เดิม และ VAE API; ถ้าไม่มีให้ข้าม optimization นี้\n    vae = getattr(inpaint_pipe, 'vae', None)\n    if callable(getattr(vae, 'enable_slicing', None)):\n        vae.enable_slicing()\n    elif callable(getattr(inpaint_pipe, 'enable_vae_slicing', None)):\n        inpaint_pipe.enable_vae_slicing()\n    else:\n        print('VAE slicing ไม่รองรับในรุ่นนี้: ใช้งานต่อโดยไม่เปิด slicing')\n    _DIFFUSION_LOADED_KEY=diffusion_key\nelse:\n    if globals().get('_DIFFUSION_PARKED',False):\n        if DIFFUSION_CPU_OFFLOAD: inpaint_pipe.enable_model_cpu_offload()\n        else: inpaint_pipe.to(DEVICE)\n    print('ใช้ diffusion pipeline ที่โหลดไว้แล้ว')\n_DIFFUSION_PARKED=False\nprint('Labels:', HAIR_ID, FACE_ID, '| UNet channels:', inpaint_pipe.unet.config.in_channels)\n# บันทึกรุ่นที่ใช้จริง ไม่เก็บ secrets\nENVIRONMENT = {\n    'repo_commit': REPO_COMMIT, 'torch': torch.__version__,\n    'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda,\n    'seg_model': SEG_MODEL_ID, 'seg_revision_requested': SEG_REVISION,\n    'seg_revision_resolved': getattr(seg_model.config, '_commit_hash', None),\n    'diffusion_model': DIFFUSION_MODEL_ID, 'diffusion_revision_requested': DIFFUSION_REVISION,\n    'unet_channels': inpaint_pipe.unet.config.in_channels,\n    'scheduler': type(inpaint_pipe.scheduler).__name__,\n    'cpu_offload': DIFFUSION_CPU_OFFLOAD,\n}\n(RESULTS_ROOT / 'requirements.freeze.txt').write_text(\n    subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))\n"), (19, 'import io, threading, uuid, zipfile\nfrom scipy.ndimage import binary_fill_holes, distance_transform_edt\nGPU_LOCK=threading.Lock()\nDEFAULT_SEED=3407\nINPAINT_STRENGTH=.15\nINPAINT_STEPS=30\nINPAINT_GUIDANCE=2.5\nJOINT_FACE_BLEND=0.35\nBACKGROUND_BLEND=0.0  # พื้นหลังซ่อมแยก ไม่ผ่าน hair/face diffusion\nBACKGROUND_REPAIR_MODE="auto"  # auto: สีเรียบเมื่อมั่นใจ มิฉะนั้น inpaint\nBACKGROUND_REPAIR_STRENGTH=0.65\nBACKGROUND_REPAIR_STEPS=40\nBACKGROUND_REPAIR_FEATHER_PX=24.0\nBACKGROUND_REPAIR_EXTRA_MASK=""  # optional: PNG mask ขนาดเท่า aligned image; white=repair\nDIFFUSION_SIDE=1024\nJOINT_TRANSITION_PX=24\nJOINT_MARGIN_PX=12\nJOINT_EYE_MARGIN_PX=3\nJOINT_EYE_FEATHER_PX=6\n_EYE_LANDMARK_MODEL=globals().get(\'_EYE_LANDMARK_MODEL\')\n_EYE_INIT_LOCK=globals().get(\'_EYE_INIT_LOCK\',threading.Lock())\n_DLIB_DETECTOR=globals().get(\'_DLIB_DETECTOR\')\n# Limited seam edits share the existing diffusion pass.\nSEAM_WIDTH_PX=4.0\nSEAM_BLEND=0.35\nHAIR_DIFFUSION_BLEND=1.0\nPROMPT = (\'Natural portrait photograph, same person, haircut and hair color. \'\n          \'Real hair with varied strand thickness, irregular fine strands, subtle flyaways, \'\n          \'soft directional highlights and depth between locks. Natural skin texture, \'\n          \'same eyes and gaze. Consistent lighting, natural hair edges against the existing background.\')\nNEGATIVE = (\'wig, plastic hair, painted streaks, uniform parallel strands, flat clumps, \'\n            \'cutout edges, bright halo, oversharpening, blur, waxy skin, \'\n            \'changed face, changed haircut, changed hair color, new background objects\')\nfor text in (PROMPT,NEGATIVE):\n    assert len(inpaint_pipe.tokenizer(text,truncation=False)[\'input_ids\']) <= inpaint_pipe.tokenizer.model_max_length,\'Prompt ยาวเกิน tokenizer\'\ndef segment_labels(image):\n    inputs = seg_processor(images=image.convert(\'RGB\'), return_tensors=\'pt\').to(DEVICE)\n    with torch.inference_mode():\n        logits = seg_model(**inputs).logits\n        logits = torch.nn.functional.interpolate(logits, size=image.size[::-1], mode=\'bilinear\', align_corners=False)\n    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)\n\ndef composite_protected(base, edited, alpha):\n    if base.size != edited.size or np.asarray(alpha).shape != (base.height, base.width):\n        raise ValueError(\'ภาพและ alpha ต้องมีขนาดตรงกัน\')\n    a = np.clip(np.asarray(alpha, dtype=np.float32), 0, 1)[..., None]\n    original = np.array(base.convert(\'RGB\'))\n    changed = np.asarray(edited.convert(\'RGB\'), dtype=np.float32)\n    output = np.rint(original * (1 - a) + changed * a).clip(0, 255).astype(np.uint8)\n    output[a[..., 0] == 0] = original[a[..., 0] == 0]\n    return Image.fromarray(output)\n\ndef padded(image, target=512, mask=False):\n    scale = target / max(image.size)\n    size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))\n    resized = image.resize(size, Image.Resampling.NEAREST if mask else Image.Resampling.LANCZOS)\n    canvas = Image.new(image.mode, (target, target), 0)\n    x, y = (target-size[0])//2, (target-size[1])//2\n    canvas.paste(resized, (x, y))\n    return canvas, (x, y, x+size[0], y+size[1])\n\ndef tensor_to_pil(value, value_range=\'zero_one\'):\n    if isinstance(value, Image.Image):\n        return value.convert(\'RGB\')\n    if torch.is_tensor(value):\n        t = value.detach().cpu()\n        if t.ndim == 4 and t.shape[0] == 1:\n            t = t[0]\n        if t.ndim != 3 or t.shape[0] != 3:\n            raise ValueError(f\'Unexpected tensor shape: {tuple(t.shape)}\')\n        a = t.permute(1, 2, 0).numpy()\n    else:\n        a = np.asarray(value)\n    if a.dtype == np.uint8:\n        return Image.fromarray(a).convert(\'RGB\')\n    if value_range == \'minus_one_one\':\n        a = (a + 1) / 2\n    if not np.isfinite(a).all() or a.min() < -0.02 or a.max() > 1.02:\n        raise ValueError(\'Tensor range ไม่ตรง config: ตรวจ OUTPUT_TENSOR_RANGE ห้าม normalize อัตโนมัติ\')\n    return Image.fromarray(np.rint(a.clip(0, 1) * 255).astype(np.uint8)).convert(\'RGB\')\n\ndef resolve_inpaint_settings(strength=None, steps=None, guidance=None):\n    import math\n    strength = float(INPAINT_STRENGTH if strength is None else strength)\n    steps = INPAINT_STEPS if steps is None else steps\n    guidance = float(INPAINT_GUIDANCE if guidance is None else guidance)\n    if isinstance(steps, bool) or not isinstance(steps, (int, np.integer)) or not 1 <= steps <= 100:\n        raise ValueError(\'steps ต้องเป็นจำนวนเต็ม 1–100\')\n    steps = int(steps)\n    if not math.isfinite(strength) or not 0 < strength <= 1:\n        raise ValueError(\'strength ต้องมากกว่า 0 และไม่เกิน 1\')\n    if not math.isfinite(guidance) or not 1 <= guidance <= 20:\n        raise ValueError(\'guidance ต้องอยู่ในช่วง 1–20\')\n    if int(steps * strength) < 1:\n        raise ValueError(\'steps × strength ต้องไม่น้อยกว่า 1\')\n    return {\'strength\':strength, \'steps\':steps, \'guidance\':guidance}\n\ndef resolve_joint_settings(face_blend=JOINT_FACE_BLEND):\n    import math\n    face_blend = float(face_blend)\n    if not math.isfinite(face_blend) or not 0 <= face_blend <= 1:\n        raise ValueError(\'face_blend ต้องอยู่ในช่วง 0–1\')\n    return {\'face_blend\':face_blend}\n\ndef prepare_eye_models():\n    """One instance per runtime, including repeated execution of this cell."""\n    global _EYE_LANDMARK_MODEL, _DLIB_DETECTOR\n    import face_alignment, inspect, dlib, hashlib, shutil\n    from pathlib import Path\n    with _EYE_INIT_LOCK:\n        if _EYE_LANDMARK_MODEL is not None:\n            if _DLIB_DETECTOR is None:_DLIB_DETECTOR=dlib.get_frontal_face_detector()\n            print(\'ใช้โมเดลตรวจหน้า–ตาที่อยู่ใน RAM แล้ว\')\n            return _EYE_LANDMARK_MODEL\n        # Keep Torch\'s normal LOCAL cache for speed; mirror checkpoints to Drive.\n        local=Path(torch.hub.get_dir())/\'checkpoints\'\n        persistent=Path(\'/content/drive/MyDrive/HairFastGAN_cache/face_alignment\')\n        local.mkdir(parents=True,exist_ok=True)\n        drive_ready=Path(\'/content/drive/MyDrive\').is_dir()\n        manifest_path=persistent/\'manifest.json\'\n        if drive_ready:\n            persistent.mkdir(parents=True,exist_ok=True)\n            manifest=json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}\n            for name,entry in manifest.items():\n                if Path(name).name!=name:continue\n                src=persistent/name;dst=local/name\n                if dst.exists() or not src.is_file():continue\n                if src.stat().st_size!=entry[\'size\']:continue\n                digest=hashlib.sha256(src.read_bytes()).hexdigest()\n                if digest!=entry[\'sha256\']:continue\n                tmp=dst.with_name(dst.name+\'.copying\');shutil.copyfile(src,tmp);tmp.replace(dst)\n                print(\'นำ checkpoint จาก Drive มาใช้:\',name)\n        before={f.name for f in local.iterdir() if f.is_file()}\n        kwargs=dict(device=\'cpu\',flip_input=False)\n        if \'compile\' in inspect.signature(face_alignment.FaceAlignment).parameters:\n            kwargs[\'compile\']=False\n        model=face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D,**kwargs)\n        detector=dlib.get_frontal_face_detector()\n        # Cache both known FAN/S3FD files and any new files downloaded by this version.\n        if drive_ready:\n            manifest=json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}\n            for src in local.iterdir():\n                if not src.is_file():continue\n                if src.name in before and not any(k in src.name.lower() for k in (\'s3fd\',\'2dfan\',\'3dfan\')):continue\n                if not src.name.endswith((\'.pth\',\'.pth.tar\',\'.pt\')):continue\n                entry=dict(size=src.stat().st_size,sha256=hashlib.sha256(src.read_bytes()).hexdigest())\n                if manifest.get(src.name)!=entry or not (persistent/src.name).is_file():\n                    tmp=persistent/(src.name+\'.copying\');shutil.copyfile(src,tmp);tmp.replace(persistent/src.name)\n                manifest[src.name]=entry\n            tmp=manifest_path.with_suffix(\'.tmp\');tmp.write_text(json.dumps(manifest,indent=2));tmp.replace(manifest_path)\n        _EYE_LANDMARK_MODEL=model;_DLIB_DETECTOR=detector\n        print(\'โมเดลตรวจหน้า–ตาพร้อม ใช้ซ้ำทุกภาพใน runtime นี้\')\n        return model\n\ndef detect_eye_landmarks(base):\n    """Inference only after setup; no repeated model construction."""\n    if _EYE_LANDMARK_MODEL is None:\n        raise RuntimeError(\'รันส่วน 6 ให้จบก่อน เพื่อเตรียมโมเดลตรวจหน้า–ตา\')\n    def validate(predictions, size):\n        if predictions is None or len(predictions) != 1: return None\n        p = np.asarray(predictions[0], dtype=np.float32).copy()\n        if p.shape != (68, 2) or not np.isfinite(p).all(): return None\n        p[:, 0] *= base.width / size[0]\n        p[:, 1] *= base.height / size[1]\n        eyes = p[36:48]\n        if (eyes < 0).any() or (eyes[:,0] >= base.width).any() or (eyes[:,1] >= base.height).any(): return None\n        if any(np.ptp(p[a:b,0]) < 2 for a,b in ((36,42),(42,48))): return None\n        return p\n    detector = _DLIB_DETECTOR\n    with torch.inference_mode():\n        predictions = _EYE_LANDMARK_MODEL.get_landmarks(np.asarray(base.convert(\'RGB\')))\n        p = validate(predictions, base.size)\n        if p is not None: return p\n        # Detection-only resizing/contrast. Never alter the image sent to diffusion.\n        for side in (512, 768):\n            scale = side / max(base.size)\n            size = (max(1, round(base.width*scale)), max(1, round(base.height*scale)))\n            im = base.convert(\'RGB\').resize(size, Image.Resampling.LANCZOS)\n            rgb = np.ascontiguousarray(np.asarray(im))\n            for gray in (im.convert(\'L\'), ImageOps.autocontrast(im.convert(\'L\'))):\n                rects = detector(np.ascontiguousarray(np.asarray(gray)), 1)\n                if len(rects) != 1: continue\n                r = rects[0]\n                boxes = [[max(0,r.left()),max(0,r.top()),min(size[0]-1,r.right()),min(size[1]-1,r.bottom())]]\n                predictions = _EYE_LANDMARK_MODEL.get_landmarks(rgb, detected_faces=boxes)\n                p = validate(predictions, size)\n                if p is not None:\n                    print(\'Eye protection: ใช้ตัวตรวจสำรองสำเร็จ\')\n                    return p\n    raise ValueError(\'ยังตรวจตำแหน่งตาที่เชื่อถือได้ไม่ได้ หยุดก่อน diffusion; ส่งภาพ hairfast_B24.png จากโฟลเดอร์ผลเพื่อตรวจต่อ\')\n\ndef build_eye_protection(size, landmarks):\n    """eye_keep=1 คืน A เต็มที่; 0 ใช้ Joint ตามเดิม; ขอบมีช่วงเปลี่ยนที่จำกัด."""\n    from scipy.ndimage import distance_transform_edt\n    w, h = size\n    canvas = Image.new(\'L\', (w, h), 0)\n    draw = ImageDraw.Draw(canvas)\n    for eye in (landmarks[36:42], landmarks[42:48]):\n        if np.ptp(eye[:, 0]) < 2 or (eye < 0).any() or (eye[:, 0] >= w).any() or (eye[:, 1] >= h).any():\n            raise ValueError(\'Eye protection: ตำแหน่งตาอยู่นอกภาพหรือเล็กเกินไป\')\n        draw.polygon([tuple(map(float, p)) for p in eye], fill=255)\n    scale = min(w, h) / 512.0\n    margin = max(0, round(JOINT_EYE_MARGIN_PX * scale))\n    if margin:\n        canvas = canvas.filter(ImageFilter.MaxFilter(2 * margin + 1))\n    core = np.asarray(canvas) > 0\n    feather = max(1.0, JOINT_EYE_FEATHER_PX * scale)\n    x = np.clip(distance_transform_edt(~core) / feather, 0, 1)\n    keep = (1 - x*x*(3-2*x)).astype(np.float32)\n    keep[core] = 1.0\n    return {\'eye_core\': core, \'eye_keep\': keep}\n\n\n# ความกว้างรอยต่อ หน่วยพิกเซลที่ภาพ 1024; ปรับตามขนาดภาพอัตโนมัติ\nRESTORE_FEATHER_PX = 16.0\nHAIR_FACE_FEATHER_PX = 6.0\nHAIR_BACKGROUND_FEATHER_PX = 4.0\nLABEL_TRANSITION_PX = 8.0\n\ndef smootherstep(t):\n    t=np.clip(t,0,1).astype(np.float32)\n    return t*t*t*(t*(t*6-15)+10)\n\ndef outside_distance(mask):\n    mask=np.asarray(mask,dtype=bool)\n    if not mask.any():return np.full(mask.shape,np.inf,dtype=np.float32)\n    return distance_transform_edt(~mask).astype(np.float32)\n\ndef outward_feather(mask,width):\n    # น้ำหนัก 1 ใน mask แล้วค่อยลดลงถึง 0 ภายนอก ไม่มี Gaussian tail\n    return 1-smootherstep(outside_distance(mask)/np.maximum(width,1.0))\n\ndef background_confidence(image):\n    """Two views of the existing parser; no additional model downloads."""\n    bg_id=label_ids[\'background\']\n    views=[];base_labels=None\n    for flip in (False,True):\n        view=ImageOps.mirror(image) if flip else image\n        inputs=seg_processor(images=view.convert(\'RGB\'),return_tensors=\'pt\').to(DEVICE)\n        with torch.inference_mode():\n            logits=seg_model(**inputs).logits\n            logits=torch.nn.functional.interpolate(logits,size=image.size[::-1],mode=\'bilinear\',align_corners=False)\n            if not flip:base_labels=logits.argmax(1)[0].cpu().numpy().astype(np.uint8)\n            probs=logits.softmax(1)[0,[bg_id,HAIR_ID]].cpu().numpy()\n        if flip:probs=probs[:,:,::-1].copy()\n        views.append(probs)\n    # Confidence is a heuristic, not a calibrated probability of correctness.\n    bg=np.minimum(views[0][0],views[1][0])\n    hair=np.maximum(views[0][1],views[1][1])\n    return bg,hair,base_labels\n\ndef preservation_and_repair_masks(base,original,lo,lb,background_score,hair_score):\n    from scipy.ndimage import binary_dilation\n    scale=min(base.size)/1024.;bg_id=label_ids[\'background\']\n    old=lo==HAIR_ID\n    # Separate liberal keep mask from narrow diffusion hair mask.\n    hair_keep=(lb==HAIR_ID)|(hair_score>=.08)\n    # New pixels on source background may be missed new hair: preserve them.\n    delta=np.linalg.norm(np.asarray(base,dtype=np.float32)-np.asarray(original,dtype=np.float32),axis=2)\n    uncertain_new=(lo==bg_id)&(delta>18)\n    keep=hair_keep|uncertain_new|(lb!=bg_id)|(background_score<.98)\n    guard=binary_dilation(keep,iterations=max(1,round(6*scale)))\n    old_zone=binary_dilation(old,iterations=max(1,round(20*scale)))\n    candidate=old_zone&(lb==bg_id)&(background_score>=.98)&~guard\n    return guard,candidate\n\ndef prepare_face_base(base,original,need_landmarks=True):\n    if base.size!=original.size:raise ValueError(\'aligned source กับ B24 ขนาดไม่ตรงกัน\')\n    lo=segment_labels(original)\n    bg_score,hair_score,lb=background_confidence(base)\n    old,new=lo==HAIR_ID,lb==HAIR_ID\n    if not old.any() or not new.any() or not (lo==FACE_ID).any():\n        raise ValueError(\'ตรวจผมหรือหน้าไม่ได้ โปรดตรวจ segmentation\')\n    # Full B24 base: no source face is pasted, so no source alignment detector needed.\n    if need_landmarks:\n        if _EYE_LANDMARK_MODEL is None:prepare_eye_models()\n        lm1=detect_eye_landmarks(base)\n    else:\n        lm1=None\n    scale=min(base.size)/1024.\n    # Segmentation selects edits only, NEVER cuts the B24 silhouette.\n    # Preserve every B24 pixel before diffusion, including missed hair strands.\n    patch=np.ones((base.height,base.width),dtype=np.float32)\n    restored=base.copy()\n    keep_guard,repair_candidate=preservation_and_repair_masks(base,original,lo,lb,bg_score,hair_score)\n    eyes=(build_eye_protection(base.size,lm1) if lm1 is not None else\n          {\'eye_core\':np.zeros(lb.shape,bool),\'eye_keep\':np.zeros(lb.shape,np.float32)})\n    # ลดการป้องกันอย่างต่อเนื่องเมื่อมีผมบัง ไม่ใช้ boolean ตัดขอบตา\n    eye_keep=(eyes[\'eye_keep\']*(1-patch)).astype(np.float32)\n    visible_eyes=eyes[\'eye_core\'] & (patch==0)\n    return restored,dict(old_hair=old,hair=new,face=lb==FACE_ID,\n        source_keep=patch==0,restore_alpha=patch,\n        source_labels=lo,base_labels=lb,\n        hair_preserve_guard=keep_guard,background_candidate=repair_candidate,\n        background_score=bg_score,\n        uncertain_new_hair=(lo==label_ids[\'background\']) & (np.linalg.norm(np.asarray(base,dtype=np.float32)-np.asarray(original,dtype=np.float32),axis=2)>18),\n        eye_core=visible_eyes,eye_keep=eye_keep,base_eye_masks=eyes,\n        alignment_median_px=None,alignment_eye_max_px=None)\n\ndef single_masks(base,face_blend=JOINT_FACE_BLEND,restoration=None):\n    if restoration is None:raise ValueError(\'ต้องมี aligned source restoration\')\n    face_blend=resolve_joint_settings(face_blend)[\'face_blend\']\n    if not 0<=BACKGROUND_BLEND<=1:raise ValueError(\'BACKGROUND_BLEND ต้องอยู่ในช่วง 0–1\')\n    r=restoration\n    scale=min(base.size)/1024.\n    target_labels=r[\'base_labels\']\n    if \'background\' not in label_ids:raise ValueError(\'ไม่พบ background label\')\n    eyes=r[\'base_eye_masks\']\n    hair=r[\'hair\']\n    visible_eyes=eyes[\'eye_core\'] & ~hair\n    # Do not fill holes or union with source-face masks: bangs may cover them.\n    face=((target_labels==FACE_ID) & r[\'face\']) & ~hair\n    background=(target_labels==label_ids[\'background\']) & ~hair & ~face\n    # Face/eye receive .35; hair core receives 1. Feather stays INSIDE the\n    # person region so the already-good background is not regenerated.\n    person=hair|face\n    dist=distance_transform_edt(np.pad(person,1))[1:-1,1:-1]\n    boundary=smootherstep(dist/max(1.,HAIR_BACKGROUND_FEATHER_PX*scale))\n    hd=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]\n    hair_weight=smootherstep(hd/max(1.,HAIR_FACE_FEATHER_PX*scale))\n    alpha=(face_blend+(HAIR_DIFFUSION_BLEND-face_blend)*hair_weight)*boundary\n    eye_weight=eyes[\'eye_keep\']*(1-outward_feather(hair,max(1.,4*scale)))\n    # Blend only within person; do not spill across background.\n    alpha=(alpha*(1-eye_weight)+face_blend*eye_weight*boundary)*person\n    alpha=alpha.astype(np.float32)\n    # Bridge the hair/background boundary rather than fading to zero there.\n    # A small bounded band; uncertain new hair outside the parser remains B24.\n    safe_background=background & (r[\'background_score\']>=.98) & ~r[\'uncertain_new_hair\']\n    to_hair=outside_distance(hair)\n    to_bg=outside_distance(background)\n    width=max(2.,SEAM_WIDTH_PX*scale)\n    seam_support=(hair & (to_bg<width)) | (safe_background & (to_hair<width))\n    support_distance=distance_transform_edt(np.pad(seam_support,1))[1:-1,1:-1]\n    soft_support=smootherstep(support_distance/max(1.,3*scale))\n    distance_to_edge=np.where(hair,to_bg,to_hair)\n    seam_alpha=SEAM_BLEND*(1-smootherstep(distance_to_edge/width))*soft_support\n    seam_alpha*=seam_support\n    # Preserve the face/eye blend; seam does not include skin labels.\n    alpha=1-(1-alpha)*(1-seam_alpha)\n    alpha=alpha.astype(np.float32)\n    raw=(alpha>0).astype(\'uint8\')*255\n    source_weight=(1-r[\'restore_alpha\'])*(1-alpha)\n    b24_weight=r[\'restore_alpha\']*(1-alpha)\n    assert np.allclose(source_weight+b24_weight+alpha,1,atol=1e-6)\n    return dict(r,alpha=alpha,raw_mask=Image.fromarray(raw),face=face,\n        background=background,eye_core=visible_eyes,eye_keep=eye_weight,\n        edit_core=hair|face|background,source_weight=source_weight,b24_weight=b24_weight,\n        seam_mask=(alpha>0)&(alpha<1),seam_edit_alpha=seam_alpha)\n\ndef refine_single(base,masks,seed,settings):\n    if DIFFUSION_SIDE%8 or DIFFUSION_SIDE<512:raise ValueError(\'DIFFUSION_SIDE ต้อง >=512 และหาร 8 ลงตัว\')\n    image,box=padded(base,target=DIFFUSION_SIDE)\n    mask,_=padded(masks[\'raw_mask\'],target=DIFFUSION_SIDE,mask=True)\n    if not np.asarray(mask).any():raise ValueError(\'พื้นที่แก้ว่าง\')\n    with torch.inference_mode():\n        result=inpaint_pipe(image=image,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,\n            prompt=PROMPT,negative_prompt=NEGATIVE,num_images_per_prompt=1,\n            num_inference_steps=settings[\'steps\'],strength=settings[\'strength\'],\n            guidance_scale=settings[\'guidance\'],\n            generator=torch.Generator(device=DEVICE).manual_seed(seed))\n    flags=getattr(result,\'nsfw_content_detected\',None)\n    if flags is not None and any(flags):raise ValueError(\'โมเดลระงับภาพผลลัพธ์\')\n    edited=result.images[0].convert(\'RGB\').crop(box).resize(base.size,Image.Resampling.LANCZOS)\n    if masks.get(\'_capture_raw\',True):masks[\'_raw_diffusion\']=edited.copy()\n    return composite_protected(base,edited,masks[\'alpha\'])\n\n\ndef repair_background(image, original, restoration, seed):\n    """Conservative automatic mask; never selects damage from dark color alone."""\n    from scipy.ndimage import binary_dilation, gaussian_filter\n    r=restoration; scale=min(image.size)/1024.\n    labels=r[\'base_labels\'];source_labels=r[\'source_labels\']\n    bg_id=label_ids[\'background\']\n    target_bg=labels==bg_id\n    guard=r[\'hair_preserve_guard\']\n    candidate=r[\'background_candidate\'].copy()\n    if BACKGROUND_REPAIR_EXTRA_MASK:\n        with Image.open(BACKGROUND_REPAIR_EXTRA_MASK) as m:\n            if m.size!=image.size:raise ValueError(\'extra mask ต้องมีขนาดเท่าภาพ aligned\')\n            manual=np.asarray(m.convert(\'L\'))>127\n        candidate |= manual & target_bg & ~guard\n    if not candidate.any():\n        return image.copy(),np.zeros(labels.shape,np.float32),dict(method=\'skipped\',passes=0,\n            reason=\'No confident background repair region; uncertain pixels preserved as B24.\')\n    # Feather outward around the damage, and inward near protected anatomy.\n    width=max(1.,BACKGROUND_REPAIR_FEATHER_PX*scale)\n    # Feather INSIDE manual support so repair cannot spill onto unmarked hair.\n    inside=distance_transform_edt(np.pad(candidate,1))[1:-1,1:-1]\n    alpha=smootherstep(inside/width)\n    clearance=distance_transform_edt(~guard)\n    alpha*=smootherstep(clearance/width)\n    alpha[guard]=0\n    # Source edge background samples are independent of generated black/blue defects.\n    border=np.zeros(labels.shape,bool);bw=max(2,round(32*scale))\n    border[:bw]=True;border[-bw:]=True;border[:,:bw]=True;border[:,-bw:]=True\n    clean=(source_labels==bg_id)&border&~r[\'old_hair\']\n    samples=np.asarray(original)[clean].astype(np.float32)\n    flat=False;color=None;p90=None\n    if len(samples)>=100:\n        color=np.median(samples,axis=0)\n        errors=np.linalg.norm(samples-color,axis=1)\n        p90=float(np.percentile(errors,90))\n        flat=p90<=12.0\n    if BACKGROUND_REPAIR_MODE not in (\'manual_only\',\'auto\',\'inpaint\',\'flat\'):\n        raise ValueError(\'BACKGROUND_REPAIR_MODE ต้องเป็น auto/inpaint/flat\')\n    use_flat=BACKGROUND_REPAIR_MODE==\'flat\' or (BACKGROUND_REPAIR_MODE in (\'auto\',\'manual_only\') and flat)\n    if use_flat and not flat:\n        raise ValueError(\'ตรวจพื้นหลังสีเรียบไม่ผ่าน ใช้ auto หรือ inpaint\')\n    passes=0\n    if use_flat:\n        edited=Image.new(\'RGB\',image.size,tuple(np.rint(color).astype(int)))\n        method=\'flat_source_background\'\n    else:\n        prompt=(\'Continue only the existing background, matching surrounding texture, \'\n                \'colors and lighting, clean detailed seamless background.\')\n        negative=\'hair, person, face, ears, clothing, black fragments, new objects, text, artifacts\'\n        canvas,box=padded(image,target=DIFFUSION_SIDE)\n        mask,_=padded(Image.fromarray((alpha>0).astype(\'uint8\')*255),target=DIFFUSION_SIDE,mask=True)\n        with torch.inference_mode():\n            output=inpaint_pipe(image=canvas,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,\n                prompt=prompt,negative_prompt=negative,strength=BACKGROUND_REPAIR_STRENGTH,\n                num_inference_steps=BACKGROUND_REPAIR_STEPS,guidance_scale=6.5,\n                generator=torch.Generator(device=DEVICE).manual_seed((int(seed)+101)%4294967296))\n        flags=getattr(output,\'nsfw_content_detected\',None)\n        if flags is not None and any(flags):raise ValueError(\'โมเดลระงับภาพซ่อมพื้นหลัง\')\n        edited=output.images[0].convert(\'RGB\').crop(box).resize(image.size,Image.Resampling.LANCZOS)\n        method=\'background_inpaint\';passes=1\n    fixed=composite_protected(image,edited,alpha)\n    assert np.array_equal(np.asarray(fixed)[guard],np.asarray(image)[guard])\n    assert np.array_equal(np.asarray(fixed)[alpha==0],np.asarray(image)[alpha==0])\n    return fixed,alpha,dict(method=method,passes=passes,edge_background_p90=p90,\n        sampled_color_rgb=None if color is None else np.rint(color).astype(int).tolist(),\n        candidate_pixels=int(candidate.sum()),full_replacement_pixels=int((alpha==1).sum()),\n        repair_pixels=int((alpha>0).sum()),strength=BACKGROUND_REPAIR_STRENGTH if passes else None)\n\n# เตรียมก่อนอัปโหลด/รับคำขอ API ไม่โหลดโมเดลในคำขอแรก\nPRELOAD_EYE_MODELS=False  # เว็บใช้ face segmentation; debug จะโหลด landmark เมื่อจำเป็น\nif PRELOAD_EYE_MODELS:prepare_eye_models()\n\n\nCOLOR_LOCK_MODE = \'auto\'  # auto / smoke_gray / blue / off\n\ndef validate_color_lock(mode):\n    mode=str(mode).strip().lower()\n    if mode not in (\'auto\',\'smoke_gray\',\'blue\',\'off\'):\n        raise ValueError(\'color_lock ต้องเป็น auto, smoke_gray, blue หรือ off\')\n    return mode\n\ndef preserve_hair_color(b24, edited, masks, mode=COLOR_LOCK_MODE):\n    from skimage.color import rgb2lab,lab2rgb\n    from scipy.ndimage import gaussian_filter\n    mode=validate_color_lock(mode)\n    hair=np.asarray(masks[\'hair\'],bool)\n    zero=np.zeros(hair.shape,np.float32)\n    if mode==\'off\' or hair.sum()<32:\n        return edited.copy(),zero,dict(requested=mode,applied=\'off\',reason=\'disabled_or_small_mask\')\n    src=rgb2lab(np.asarray(b24.convert(\'RGB\'))).astype(np.float32)\n    dst=rgb2lab(np.asarray(edited.convert(\'RGB\'))).astype(np.float32)\n    interior=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]\n    sample=hair&(interior>=max(1.,min(b24.size)/256.))\n    if sample.sum()<32:sample=hair\n    L=src[:,:,0][sample];ab=src[:,:,1:][sample]\n    C=np.linalg.norm(ab,axis=1)\n    blue_share=float(np.mean((ab[:,1]<-10)&(C>15)))\n    gray_share=float(np.mean((C<10)&(L>30)&(L<85)))\n    applied=mode\n    if mode==\'auto\':\n        if blue_share>=.70:applied=\'blue\'\n        elif gray_share>=.80 and float(np.median(C))<8:applied=\'smoke_gray\'\n        else:applied=\'off\'\n    info=dict(requested=mode,applied=applied,blue_share=blue_share,gray_share=gray_share,\n              reference=\'hairfast_B24\',auto_is_heuristic=True)\n    if applied==\'off\':return edited.copy(),zero,info\n    # Mask-normalized low frequencies exclude skin/background from color estimates.\n    sigma=max(1.,4.*min(b24.size)/1024.)\n    weight=hair.astype(np.float32)\n    denom=np.maximum(gaussian_filter(weight,sigma),1e-6)\n    def low(lab):\n        return np.stack([gaussian_filter(lab[:,:,i]*weight,sigma)/denom for i in range(3)],axis=-1)\n    low_src,low_dst=low(src),low(dst)\n    corrected=dst.copy()\n    # Recover chroma without copying B24\'s blurred luminance texture.\n    corrected[:,:,1:]=.90*low_src[:,:,1:]+.10*low_dst[:,:,1:]+.20*(dst[:,:,1:]-low_dst[:,:,1:])\n    luminance_gain=.80 if applied==\'smoke_gray\' else .35\n    corrected[:,:,0]=np.clip(dst[:,:,0]+luminance_gain*(low_src[:,:,0]-low_dst[:,:,0]),0,100)\n    rgb=np.rint(lab2rgb(corrected).clip(0,1)*255).astype(np.uint8)\n    edge=smootherstep(interior/max(1.,HAIR_FACE_FEATHER_PX*min(b24.size)/1024.))\n    alpha=edge*hair*np.asarray(masks[\'alpha\'],np.float32)\n    final=composite_protected(edited,Image.fromarray(rgb),alpha)\n    changed=rgb2lab(np.asarray(final))\n    info.update(luminance_gain=luminance_gain,chroma_gain=.90,\n        mean_chroma_error_before=float(np.linalg.norm(dst[:,:,1:]-src[:,:,1:],axis=-1)[sample].mean()),\n        mean_chroma_error_after=float(np.linalg.norm(changed[:,:,1:]-src[:,:,1:],axis=-1)[sample].mean()),\n        median_L_B24=float(np.median(L)),median_L_before=float(np.median(dst[:,:,0][sample])),\n        median_L_after=float(np.median(changed[:,:,0][sample])))\n    assert np.array_equal(np.asarray(final)[alpha==0],np.asarray(edited)[alpha==0])\n    return final,alpha,info\n\n\nHAIR_DETAIL_AMOUNT=0.25  # 0 disables; subtle local contrast, not new strands\n\ndef hair_detail_finish(image,masks):\n    from scipy.ndimage import gaussian_filter,minimum_filter,maximum_filter\n    from skimage.color import rgb2lab,lab2rgb\n    hair=np.asarray(masks[\'hair\'],bool)\n    if HAIR_DETAIL_AMOUNT==0:return image.copy(),np.zeros_like(hair,dtype=np.float32)\n    if not 0<=HAIR_DETAIL_AMOUNT<=.5:raise ValueError(\'HAIR_DETAIL_AMOUNT ต้องอยู่ระหว่าง 0–0.5\')\n    lab=rgb2lab(np.asarray(image)).astype(np.float32);L=lab[:,:,0]\n    sigma=max(.6,.9*min(image.size)/1024.)\n    residual=L-gaussian_filter(L,sigma)\n    # Threshold noise; cap contrast and clamp to local range to avoid overshoot halos.\n    detail=np.sign(residual)*np.maximum(np.abs(residual)-.35,0)\n    corrected=np.clip(L+np.clip(HAIR_DETAIL_AMOUNT*detail,-1.5,1.5),\n                      minimum_filter(L,3),maximum_filter(L,3))\n    out=lab.copy();out[:,:,0]=corrected\n    distance=distance_transform_edt(np.pad(hair,1))[1:-1,1:-1]\n    scale=min(image.size)/1024.\n    # No sharpening near the hair silhouette, skin, background or unedited B24.\n    alpha=smootherstep((distance-max(2.,8*scale))/max(1.,4*scale))*masks[\'alpha\']\n    alpha*=hair\n    rgb=np.rint(lab2rgb(out).clip(0,1)*255).astype(np.uint8)\n    final=composite_protected(image,Image.fromarray(rgb),alpha)\n    assert np.array_equal(np.asarray(final)[alpha==0],np.asarray(image)[alpha==0])\n    return final,alpha\n\n\nDETAIL_GAIN=.65\nDETAIL_MAX_RGB=5.0\n\ndef detail_only_masks(base,face_blend):\n    labels=segment_labels(base)  # exactly one parser pass\n    hair=labels==HAIR_ID;face=labels==FACE_ID\n    if not hair.any():raise ValueError(\'ตรวจไม่พบผม\')\n    def inside(mask,width):\n        distance=distance_transform_edt(np.pad(mask,1))[1:-1,1:-1]\n        return smootherstep(distance/max(1.,width*min(base.size)/1024.))\n    # No hair silhouette displacement: taper inside each edit region only.\n    hair_alpha=inside(hair,4.)\n    face_alpha=inside(face,6.)*float(face_blend)\n    alpha=np.maximum(hair_alpha,face_alpha).astype(np.float32)\n    return dict(hair=hair,face=face,alpha=alpha,\n        raw_mask=Image.fromarray((alpha>0).astype(\'uint8\')*255))\n\ndef transfer_luminance_detail(b24,proposal,alpha):\n    from scipy.ndimage import gaussian_filter\n    source=np.asarray(b24.convert(\'RGB\'),dtype=np.float32)\n    generated=np.asarray(proposal.convert(\'RGB\'),dtype=np.float32)\n    if source.shape!=generated.shape:raise ValueError(\'ภาพต้องมีขนาดเท่ากัน\')\n    coeff=np.array([.299,.587,.114],np.float32)\n    y0=source@coeff;y1=generated@coeff\n    sigma=max(.8,1.2*min(b24.size)/1024.)\n    detail0=y0-gaussian_filter(y0,sigma)\n    detail1=y1-gaussian_filter(y1,sigma)\n    # Preserve B24 coarse structure/color; use only bounded high-frequency delta.\n    delta=np.clip(DETAIL_GAIN*(detail1-detail0),-DETAIL_MAX_RGB,DETAIL_MAX_RGB)*alpha\n    # Equal RGB increments preserve channel differences; no individual-channel clipping.\n    delta=np.clip(delta,-source.min(axis=2),255-source.max(axis=2))\n    shift=np.rint(delta).astype(np.int16)\n    out=(source.astype(np.int16)+shift[:,:,None]).astype(np.uint8)\n    out[alpha==0]=source.astype(np.uint8)[alpha==0]\n    return Image.fromarray(out)\n\ndef refine_b24_details(base,masks,seed,settings):\n    canvas,box=padded(base,target=DIFFUSION_SIDE)\n    mask,_=padded(masks[\'raw_mask\'],target=DIFFUSION_SIDE,mask=True)\n    with torch.inference_mode():\n        result=inpaint_pipe(image=canvas,mask_image=mask,height=DIFFUSION_SIDE,width=DIFFUSION_SIDE,\n            prompt=\'Same portrait, same hairstyle and hair color, fine natural hair detail, clear eyes, natural skin texture, realistic photograph.\',\n            negative_prompt=\'changed hairstyle, changed hair color, changed face, waxy skin, blur, halos, noise\',\n            num_images_per_prompt=1,strength=settings[\'strength\'],num_inference_steps=settings[\'steps\'],\n            guidance_scale=settings[\'guidance\'],generator=torch.Generator(device=DEVICE).manual_seed(int(seed)))\n    flags=getattr(result,\'nsfw_content_detected\',None)\n    if flags is not None and any(flags):raise ValueError(\'โมเดลระงับภาพผลลัพธ์\')\n    proposal=result.images[0].convert(\'RGB\').crop(box).resize(base.size,Image.Resampling.LANCZOS)\n    return proposal\n\n\nRAW_BG_GAP_PX=4.0  # pixels at 1024, scaled with output size\nRAW_BG_FEATHER_PX=32.0\nRAW_BG_DETAIL_AMOUNT=.45\n\ndef finish_raw_background(raw,b24_masks,labels=None):\n    from scipy.ndimage import gaussian_filter\n    if RAW_BG_GAP_PX<0 or RAW_BG_FEATHER_PX<=0 or not 0<=RAW_BG_DETAIL_AMOUNT<=1:\n        raise ValueError(\'ตรวจค่า RAW_BG_GAP_PX / FEATHER / DETAIL_AMOUNT\')\n    # Hair may move in raw diffusion: protect both B24 and raw hair.\n    if labels is None:labels=segment_labels(raw)\n    if \'background\' not in label_ids:raise ValueError(\'ไม่พบ background label\')\n    bg=labels==label_ids[\'background\']\n    hair=(labels==HAIR_ID)|b24_masks[\'hair\']\n    scale=min(raw.size)/1024.\n    hair_distance=outside_distance(hair)\n    gap=RAW_BG_GAP_PX*scale;feather=max(1.,RAW_BG_FEATHER_PX*scale)\n    # Stay inside background; protect skin, ears, neck and clothes too.\n    bg_distance=distance_transform_edt(np.pad(bg,1))[1:-1,1:-1]\n    # One continuous ramp from the nearest protected boundary.\n    # Feather the adjustment weight, never the output image.\n    safe_distance=np.minimum(hair_distance,bg_distance)\n    alpha=(smootherstep((safe_distance-gap)/feather)*bg).astype(np.float32)\n    rgb=np.asarray(raw.convert(\'RGB\'),dtype=np.float32)\n    luminance=rgb@np.array([.299,.587,.114],np.float32)\n    detail=luminance-gaussian_filter(luminance,max(.8,1.2*scale))\n    detail=np.sign(detail)*np.maximum(np.abs(detail)-.5,0.)\n    delta=np.clip(RAW_BG_DETAIL_AMOUNT*detail,-4.,4.)*alpha\n    delta=np.clip(delta,-rgb.min(axis=2),255-rgb.max(axis=2))\n    out=(rgb.astype(np.int16)+np.rint(delta).astype(np.int16)[:,:,None]).astype(np.uint8)\n    protected=(safe_distance<=gap)|~bg\n    assert not alpha[protected].any()\n    assert np.array_equal(out[protected],rgb.astype(np.uint8)[protected])\n    return Image.fromarray(out),alpha,dict(method=\'raw_background_smooth_gap_detail\',\n        gap_px_at_1024=RAW_BG_GAP_PX,feather_px_at_1024=RAW_BG_FEATHER_PX,\n        amount=RAW_BG_DETAIL_AMOUNT,protected_raw_max_diff=0,extra_diffusion_passes=0)\n\n\n# Lightweight native-resolution finish. No extra neural model / diffusion pass.\n# Defaults keep the ORIGINAL diffusion prompt and sampling parameters unchanged.\nNATIVE_SHARP_ENABLED = True\nNATIVE_HAIR_AMOUNT = 0.65\nNATIVE_FACE_AMOUNT = 0.25\nNATIVE_SHARP_MAX_DELTA = 5.0  # maximum equal RGB shift, in 8-bit levels\nNATIVE_SHARP_THRESHOLD = 0.8  # suppress amplification of very small residuals\n\ndef finish_native_sharp(image, labels):\n    from scipy.ndimage import gaussian_filter\n    if not (0 <= NATIVE_HAIR_AMOUNT <= 1.5 and 0 <= NATIVE_FACE_AMOUNT <= .75\n            and 0 <= NATIVE_SHARP_MAX_DELTA <= 10 and NATIVE_SHARP_THRESHOLD >= 0):\n        raise ValueError(\'ตรวจค่า NATIVE_SHARP: hair 0–1.5, face 0–.75, delta 0–10\')\n    rgb=np.asarray(image.convert(\'RGB\'),dtype=np.float32)\n    if labels.shape != rgb.shape[:2]:raise ValueError(\'ขนาดมาสก์ไม่ตรงกับภาพ\')\n    info=dict(method=\'bounded_native_luminance_sharpen\',enabled=bool(NATIVE_SHARP_ENABLED),\n              hair_amount=NATIVE_HAIR_AMOUNT,face_amount=NATIVE_FACE_AMOUNT,\n              max_delta=NATIVE_SHARP_MAX_DELTA,threshold=NATIVE_SHARP_THRESHOLD,\n              output_size=list(image.size),extra_diffusion_passes=0)\n    if not NATIVE_SHARP_ENABLED:\n        return image.copy(),dict(info,observed_max_delta=0)\n    scale=min(image.size)/1024.\n    def inward(region, width):\n        dist=distance_transform_edt(np.pad(region,1))[1:-1,1:-1]\n        # Leave outermost pixel unchanged; taper inward, never blur the image edge.\n        return smootherstep((dist-1.)/max(1.,width*scale))*region\n    weight=(NATIVE_HAIR_AMOUNT*inward(labels==HAIR_ID,3.)+\n            NATIVE_FACE_AMOUNT*inward(labels==FACE_ID,5.)).astype(np.float32)\n    lum=rgb@np.array([.299,.587,.114],np.float32)\n    fine=lum-gaussian_filter(lum,max(.55,.75*scale))\n    medium=lum-gaussian_filter(lum,max(.8,1.35*scale))\n    residual=.75*fine+.25*medium\n    residual=np.sign(residual)*np.maximum(np.abs(residual)-NATIVE_SHARP_THRESHOLD,0.)\n    delta=np.clip(weight*residual,-NATIVE_SHARP_MAX_DELTA,NATIVE_SHARP_MAX_DELTA)\n    # Equal-channel increments preserve RGB channel differences without clipping.\n    delta=np.clip(delta,-rgb.min(axis=2),255-rgb.max(axis=2))\n    shift=np.rint(delta).astype(np.int16)\n    out=(rgb.astype(np.int16)+shift[:,:,None]).astype(np.uint8)\n    return Image.fromarray(out),dict(info,observed_max_delta=int(np.abs(shift).max()))\n'), (20, "def run_detail_core(inputs=None,before=None,seed=3407,strength=.15,steps=30,guidance=2.5,face_blend=.35):\n    # One shared implementation for the comparison cell, upload test and web.\n    settings=resolve_inpaint_settings(strength,steps,guidance)\n    resolve_joint_settings(face_blend)\n    if not 0<=int(seed)<=4294967295:raise ValueError('seed ไม่ถูกต้อง')\n    timings={}\n    def clock():\n        if torch.cuda.is_available():torch.cuda.synchronize()\n        return time.perf_counter()\n    queued=time.perf_counter()\n    with GPU_LOCK:\n        timings['queue_wait']=time.perf_counter()-queued\n        start=clock()\n        if before is None:\n            if inputs is None or len(inputs)!=3:raise ValueError('ต้องมีหน้า/ทรง/สี 3 ภาพ')\n            t=clock()\n            swapped=hair_fast.swap(*(str(v) for v in inputs),align=True,seed=int(seed))\n            if not isinstance(swapped,tuple) or len(swapped)!=4:raise RuntimeError('swap format ไม่ตรงรุ่น')\n            before=tensor_to_pil(swapped[0]);timings['B24']=clock()-t\n            del swapped\n        else:\n            before=before.convert('RGB').copy();timings['B24']=0.\n        t=clock();masks=detail_only_masks(before,face_blend)\n        timings['mask']=clock()-t\n        t=clock();raw=refine_b24_details(before,masks,seed,settings)\n        timings['diffusion']=clock()-t\n        t=clock();raw_labels=segment_labels(raw)\n        before_sharp,bg_alpha,bg_info=finish_raw_background(raw,masks,labels=raw_labels)\n        timings['raw_background_finish']=clock()-t\n        t=clock();after,sharp_info=finish_native_sharp(before_sharp,raw_labels)\n        timings['native_sharp']=clock()-t\n        timings['total_processing']=clock()-start\n    return dict(before=before,raw_diffusion=raw,before_sharp=before_sharp,after=after,native_sharp=sharp_info,sharp_labels=raw_labels,masks=masks,background_alpha=bg_alpha,background_info=bg_info,settings=settings,\n                seed=int(seed),timings_seconds={k:round(v,4) for k,v in timings.items()})\n"), (22, "import pandas as pd\nRESULTS={}; EXPORT_IMAGES=[]; TIMING_ROWS=[]\nFINAL_DIR=RUN_DIR/'comparison_native_sharp'; FINAL_DIR.mkdir(exist_ok=True)\nMODELS=[('Barbershop_Baseline',BASE_DIR/'Barbershop_Baseline.png'),\n        ('HairFastGAN_B24',BASE_DIR/'HairFastGAN_B24.png')]\nPROFILE=dict(seed=int(SEED),strength=STRENGTH,steps=STEPS,guidance=GUIDANCE,face_blend=FACE_BLEND)\nfor model,path in MODELS:\n    print('Diffusion:',model)\n    with Image.open(path) as im: before=im.convert('RGB').copy()\n    result=run_detail_core(before=before,**PROFILE)\n    # shared legacy core calls its GAN timing B24; remove this unused zero-valued field\n    result['timings_seconds'].pop('B24',None)\n    RESULTS[model]=result\n    dest=FINAL_DIR/model; dest.mkdir(exist_ok=True)\n    for index,key,label in [(1,'before','Before_Diffusion'),(2,'raw_diffusion','Raw_Diffusion'),\n                            (3,'before_sharp','After_Background_Before_Sharp'),\n                            (4,'after','Final_Native_Sharp')]:\n        out=dest/f'{model}_{index:02d}_{label}_{RUN_ID}.png'\n        result[key].save(out,format='PNG',compress_level=1)\n        EXPORT_IMAGES.append(out)\n    result['masks']['raw_mask'].save(dest/'diffusion_mask.png')\n    for name in ['hair','face','alpha']:\n        Image.fromarray(np.rint(result['masks'][name]*255).astype('uint8')).save(dest/(name+'_mask.png'))\n    Image.fromarray(np.rint(result['background_alpha']*255).astype('uint8')).save(dest/'background_alpha.png')\n    raw=np.asarray(result['raw_diffusion']);background=np.asarray(result['before_sharp']);after=np.asarray(result['after'])\n    keep=result['background_alpha']==0\n    assert np.array_equal(raw[keep],background[keep]),'Background finish changed protected raw pixels'\n    sharp_protected=~np.isin(result['sharp_labels'],[HAIR_ID,FACE_ID])\n    assert np.array_equal(background[sharp_protected],after[sharp_protected]),'Native sharp changed non-hair/face pixels'\n    observed=int(np.abs(after.astype('int16')-background.astype('int16')).max())\n    assert observed<=np.ceil(NATIVE_SHARP_MAX_DELTA),'Native sharp exceeded bounded delta'\n    Image.fromarray(result['sharp_labels'].astype('uint8')).save(dest/'native_sharp_labels.png')\n    transfer_seconds=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_RESULT['estimated_cold_compute_seconds']\n    TIMING_ROWS.append(dict(model=model,transfer_compute_seconds=transfer_seconds,\n        transfer_wall_seconds_this_run=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_WALL_SECONDS,\n        **result['timings_seconds']))\n    metadata=dict(model=model,profile=PROFILE,settings=result['settings'],\n        base_sha256=sha_file(path),background_info=result['background_info'],native_sharp=result['native_sharp'],\n        timings=TIMING_ROWS[-1],face_blend_is_composite_weight=False,\n        mask_pixels={k:int(result['masks'][k].sum()) for k in ['hair','face']})\n    (dest/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))\nTIMINGS=pd.DataFrame(TIMING_ROWS)\nTIMINGS['transfer_plus_postprocess_seconds']=TIMINGS.transfer_compute_seconds+TIMINGS.total_processing\nTIMINGS.to_csv(FINAL_DIR/'timings.csv',index=False)\nmanifest=dict(method='looklab_native_sharp',native_source_sha256='d4c5f63c6f895bb4b7569a8f5104a373113ef33c046c81c1534a7fea4634072d',run_id=RUN_ID,profile=PROFILE,environment=ENVIRONMENT,\n    barbershop=dict(revision=BARBER_REV,weights=weight_hashes,selection='Baseline',\n        selected_from_validation_protocol='27f7f146ef0d',worker_result=BARBER_RESULT),\n    hairfast=dict(revision=REPO_COMMIT,checkpoint=str(CHECKPOINT),sha256=EXPECTED_SHA256,\n        trial='B',epoch=24,mixing=.95,shape_rotation=False,color_rotation=True),\n    original_hashes={p.name:sha_file(p) for p in INPUTS},\n    aligned_hashes={p.name:sha_file(p) for p in CROP_DIR.glob('*.png')},\n    export_images=[str(p) for p in EXPORT_IMAGES],\n    notes=['Same aligned crops; Barbershop consumes quantized PNGs.',\n           'Same postprocess algorithm/settings; masks depend on each output.',\n           'Timing excludes initial weight downloads and model loading for HFG; BS wall includes worker startup.',\n           'BS transfer compute includes inversion even when cached; wall time is shown separately.',\n           'Native sharp operates at original resolution without extra diffusion; metric results in evaluation folder.'])\n(FINAL_DIR/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\ndisplay(TIMINGS)\nprint('บันทึกภาพเต็ม 8 รูป:',FINAL_DIR)\n"), (24, "import matplotlib.pyplot as plt\nfrom scipy.ndimage import distance_transform_edt\nstages=[('before','Before diffusion'),('raw_diffusion','Raw diffusion'),('before_sharp','Background finish'),('after','Native sharp')]\nshapes={result[key].size for result in RESULTS.values() for key,_ in stages}\nif len(shapes)!=1: raise ValueError('ภาพต้องมีขนาดเท่ากันเพื่อใช้ crop เดียว')\nw,h=next(iter(shapes))\nif COMPARE_CROP is None:\n    union=np.logical_or.reduce([v['masks']['hair'] for v in RESULTS.values()])\n    distance=distance_transform_edt(np.pad(union,1))[1:-1,1:-1]\n    cy,cx=np.unravel_index(np.argmax(distance),distance.shape)\n    side=min(256,w,h);left=max(0,min(w-side,int(cx)-side//2));top=max(0,min(h-side,int(cy)-side//2))\n    crop=(left,top,left+side,top+side)\nelse:\n    crop=tuple(int(v) for v in COMPARE_CROP)\n    if len(crop)!=4 or not (0<=crop[0]<crop[2]<=w and 0<=crop[1]<crop[3]<=h):\n        raise ValueError('crop ต้องเป็น (left,top,right,bottom) ภายในภาพ')\nfor cropped in [False,True]:\n    fig,axes=plt.subplots(2,4,figsize=(22,12 if not cropped else 10))\n    for row,(model,_) in enumerate(MODELS):\n        for col,(key,label) in enumerate(stages):\n            im=RESULTS[model][key]\n            axes[row,col].imshow(im.crop(crop) if cropped else im,interpolation='nearest')\n            axes[row,col].set_title(model+' | '+label);axes[row,col].axis('off')\n    fig.tight_layout()\n    fig.savefig(FINAL_DIR/('comparison_same_crop.png' if cropped else 'comparison_full.png'),dpi=150)\n    display(fig);plt.close(fig)\n(FINAL_DIR/'crop.json').write_text(json.dumps(dict(box=crop,coordinate_system='same aligned 1024x1024 canvas'),indent=2))\nprint('Crop เดียวกันทุกภาพ:',crop)\nprint('ภาพ grid เป็น preview; PNG แยกในส่วน 12 เก็บ resolution เดิม')\n"), (26, "from google.colab import files\nimport ipywidgets as widgets\nfrom IPython.display import display\nif len(EXPORT_IMAGES)!=8 or not all(p.is_file() for p in EXPORT_IMAGES):\n    raise RuntimeError('รันส่วน 10 ให้ครบก่อน')\nbuttons=[]\nfor path in EXPORT_IMAGES:\n    button=widgets.Button(description=path.name.split('_'+RUN_ID)[0],\n        layout=widgets.Layout(width='650px'),tooltip=path.name)\n    button.on_click(lambda _,p=path: files.download(str(p)))\n    buttons.append(button)\ndisplay(widgets.VBox(buttons))\nif AUTO_DOWNLOAD_IMAGES:\n    for path in EXPORT_IMAGES:\n        print('ดาวน์โหลด:',path.name)\n        files.download(str(path))\nprint('ผลและ metadata เก็บไว้แล้วที่:',FINAL_DIR)\n"), (28, "from pathlib import Path\nimport json, numpy as np, pandas as pd\nfrom IPython.display import display\nif 'RESULTS' not in globals() or len(RESULTS)!=2:\n    raise RuntimeError('รันส่วน 10 ให้ครบก่อน')\nEVAL_DIR=FINAL_DIR/'evaluation';EVAL_DIR.mkdir(exist_ok=True)\ntime_rows=[]\ndef add_time(model,stage,seconds,scope):\n    time_rows.append(dict(model=model,stage=stage,seconds=float(seconds),scope=scope))\nadd_time('HairFastGAN_B24','swap_including_alignment',HFG_SECONDS,'measured once; excludes model load')\ninv_records=[json.loads(p.read_text()) for p in (BARBER_ROOT/'inversion').glob('*_complete.json')]\nfor key in ['W_seconds','FS_seconds']:\n    add_time('Barbershop_Baseline',key,sum(r[key] for r in inv_records),'recorded compute; may be reused from cache')\nfor key in ['align_shape_seconds','align_color_seconds','blend_seconds']:\n    add_time('Barbershop_Baseline',key,BARBER_RESULT[key],'recorded compute; may be reused from cache')\nadd_time('Barbershop_Baseline','worker_wall_this_run',BARBER_WALL_SECONDS,'includes worker/model load; cache may skip optimization')\nfor model,result in RESULTS.items():\n    for key in ['mask','diffusion','raw_background_finish','native_sharp','total_processing']:\n        add_time(model,key,result['timings_seconds'][key],'measured postprocess once; total contains preceding 4 stages')\nTIME_DETAIL=pd.DataFrame(time_rows)\nTIME_DETAIL.to_csv(EVAL_DIR/'timing_stages.csv',index=False)\ntime_summary=[]\nfor model,result in RESULTS.items():\n    transfer=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_RESULT['estimated_cold_compute_seconds']\n    ts=result['timings_seconds']\n    time_summary.append(dict(model=model,transfer_recorded_s=transfer,mask_s=ts['mask'],\n        diffusion_s=ts['diffusion'],background_s=ts['raw_background_finish'],native_sharp_s=ts['native_sharp'],\n        postprocess_total_s=ts['total_processing'],\n        transfer_plus_postprocess_s=transfer+ts['total_processing'],\n        postprocess_overhead_percent=100*ts['total_processing']/transfer if transfer>0 else np.nan,\n        scope='HFG includes alignment; BS excludes shared alignment; loading excluded from summed compute'))\nTIME_SUMMARY=pd.DataFrame(time_summary)\nTIME_SUMMARY.to_csv(EVAL_DIR/'timing_summary.csv',index=False)\ndisplay(TIME_SUMMARY.round(3));display(TIME_DETAIL.round(3))\n"), (30, "import time, shutil, torch\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom scipy.ndimage import binary_erosion, binary_dilation, gaussian_filter, laplace\nfrom skimage.color import rgb2lab\nfrom skimage.metrics import structural_similarity\nfrom losses.pp_losses import Backbone\nfrom huggingface_hub import hf_hub_download\nSHAPE_TARGET_MASK_PATH = ''  # optional: aligned binary 1024x1024 target mask\nMIN_METRIC_PIXELS=64\n# CPU evaluator avoids contaminating inference GPU timings and avoids diffusion VRAM overlap.\nMETRIC_DEVICE='cpu'\nid_path=Path(LOCAL_REPO)/'pretrained_models/ArcFace/ir_se50.pth'\nif not id_path.is_file():\n    downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',\n        filename='pretrained_models/ArcFace/ir_se50.pth',revision='main')\n    id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)\nmetric_id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')\nmetric_id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)\nmetric_id_net.to(METRIC_DEVICE).eval().requires_grad_(False)\n\ndef rgb01(image):return np.asarray(image.convert('RGB'),dtype=np.float32)/255.\n\ndef identity_embedding(image):\n    x=torch.from_numpy(rgb01(image).copy()).permute(2,0,1)[None].to(METRIC_DEVICE)\n    x=F.interpolate(x,size=(256,256),mode='bilinear',align_corners=False)*2-1\n    x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))\n    with torch.inference_mode():features=F.normalize(metric_id_net(x),dim=1)\n    return features[0].cpu().numpy()\n\ndef mask_mean(values,mask):\n    return float(np.asarray(values)[mask].mean()) if int(mask.sum())>=MIN_METRIC_PIXELS else float('nan')\n\ndef mask_iou(a,b):\n    union=(a|b).sum()\n    return float((a&b).sum()/union) if union else float('nan')\n\ndef masked_ssim(a,b,mask):\n    safe=binary_erosion(mask,structure=np.ones((7,7),bool),border_value=0)\n    if safe.sum()<MIN_METRIC_PIXELS:return float('nan')\n    _,ssmap=structural_similarity(a,b,data_range=1.,channel_axis=2,win_size=7,full=True)\n    if ssmap.ndim==3:ssmap=ssmap.mean(axis=2)\n    return mask_mean(ssmap,safe)\n\ndef color_quantile_error(lab_image,lab_ref,mask,refmask):\n    if min(mask.sum(),refmask.sum())<MIN_METRIC_PIXELS:return (float('nan'),float('nan'))\n    q=np.linspace(.1,.9,9)\n    delta=np.abs(np.quantile(lab_image[mask],q,axis=0)-np.quantile(lab_ref[refmask],q,axis=0))\n    return float(delta[:,0].mean()),float(delta[:,1:].mean())\n\nREF_IMAGES={role:Image.open(CROP_DIR/(role+'.png')).convert('RGB') for role in ['face','shape','color']}\nif {im.size for im in REF_IMAGES.values()}!={(1024,1024)}:raise ValueError('References ต้องเป็น 1024x1024')\nREF_LABELS={role:segment_labels(im) for role,im in REF_IMAGES.items()}\nif SHAPE_TARGET_MASK_PATH:\n    with Image.open(SHAPE_TARGET_MASK_PATH) as im:\n        if im.size!=(1024,1024):raise ValueError('Target mask ต้องตรง canvas 1024x1024 ห้าม resize อัตโนมัติ')\n        TARGET_HAIR=np.asarray(im.convert('L'))>127\n    TARGET_KIND='user_supplied_aligned_target_mask'\nelse:\n    TARGET_HAIR=REF_LABELS['shape']==HAIR_ID\n    TARGET_KIND='aligned_shape_reference_proxy_not_ground_truth'\nif TARGET_HAIR.sum()<MIN_METRIC_PIXELS:raise ValueError('ไม่พบพื้นที่ผมเพียงพอใน target/reference')\nREF_COLOR_CORE=binary_erosion(REF_LABELS['color']==HAIR_ID,iterations=4)\nUNION_HAIR=(REF_LABELS['face']==HAIR_ID)|TARGET_HAIR\nfor result in RESULTS.values(): UNION_HAIR|=result['masks']['hair']\nCOMMON_PROTECTED=~binary_dilation(UNION_HAIR,iterations=8)\nCOMMON_FACE=binary_erosion(REF_LABELS['face']==FACE_ID,iterations=3)&COMMON_PROTECTED\nCOMMON_BG=binary_erosion(REF_LABELS['face']==label_ids['background'],iterations=3)&COMMON_PROTECTED\nSOURCE_RGB=rgb01(REF_IMAGES['face']);COLOR_LAB=rgb2lab(rgb01(REF_IMAGES['color']))\nSOURCE_ID=identity_embedding(REF_IMAGES['face'])\nEVAL_MASK_DIR=EVAL_DIR/'fixed_masks';EVAL_MASK_DIR.mkdir(exist_ok=True)\nfor name,mask in [('target_hair',TARGET_HAIR),('color_reference_core',REF_COLOR_CORE),\n                  ('protected_common',COMMON_PROTECTED),('face_common',COMMON_FACE),('background_common',COMMON_BG)]:\n    Image.fromarray(mask.astype('uint8')*255).save(EVAL_MASK_DIR/(name+'.png'))\nprint('Shape target:',TARGET_KIND)\nprint('Fixed mask pixels:',{k:int(v.sum()) for k,v in dict(protected=COMMON_PROTECTED,face=COMMON_FACE,background=COMMON_BG,color_ref=REF_COLOR_CORE).items()})\n\n# MUSIQ runs on CPU, outside generation timing. Preserve current core runtime packages.\nimport importlib.util, importlib.metadata, tempfile, sys, subprocess\nif importlib.util.find_spec('pyiqa') is None:\n    constraints=[]\n    for package in ['torch','torchvision','torchaudio','numpy','scipy','Pillow']:\n        try: constraints.append(package+'=='+importlib.metadata.version(package))\n        except importlib.metadata.PackageNotFoundError: pass\n    with tempfile.TemporaryDirectory() as tmp:\n        constraint_path=Path(tmp)/'runtime_constraints.txt'\n        constraint_path.write_text('\\n'.join(constraints))\n        subprocess.run([sys.executable,'-m','pip','install','-q','pyiqa','-c',str(constraint_path)],check=True)\nimport pyiqa\nMUSIQ_DEVICE='cpu'\nif 'musiq_metric' not in globals():\n    musiq_metric=pyiqa.create_metric('musiq',device=MUSIQ_DEVICE)\n    musiq_metric.eval().requires_grad_(False)\nif musiq_metric.lower_better:raise RuntimeError('Unexpected MUSIQ direction')\nMUSIQ_CONFIG=dict(metric='musiq',pyiqa_version=importlib.metadata.version('pyiqa'),\n    device=MUSIQ_DEVICE,input='full RGB image in [0,1]; no manual resize/crop',\n    scope='no-reference global image quality; not a hair correctness or identity score')\ndef musiq_score(image):\n    x=torch.from_numpy(rgb01(image).copy()).permute(2,0,1)[None].to(MUSIQ_DEVICE)\n    with torch.inference_mode():score=float(musiq_metric(x).reshape(-1)[0].cpu())\n    if not np.isfinite(score):raise ValueError('MUSIQ returned nonfinite score')\n    return score\n"), (32, "metric_start=time.perf_counter()\nMETRIC_ROWS=[]\nfor model,result in RESULTS.items():\n    before=result['before'];before_rgb=rgb01(before);before_lab=rgb2lab(before_rgb)\n    before_id=identity_embedding(before)\n    hair_before=result['masks']['hair'].astype(bool)\n    fixed_core=binary_erosion(hair_before,iterations=4)\n    outside=result['masks']['alpha']==0\n    protected_before=binary_erosion(outside,iterations=4)\n    Image.fromarray(protected_before.astype('uint8')*255).save(EVAL_MASK_DIR/(model+'_protected_before.png'))\n    Image.fromarray(fixed_core.astype('uint8')*255).save(EVAL_MASK_DIR/(model+'_hair_core.png'))\n    for stage in ['before','raw_diffusion','before_sharp','after']:\n        image=result[stage]\n        if image.size!=before.size or image.size!=REF_IMAGES['face'].size:raise ValueError('Metric images ต้องอยู่ canvas เดียวกัน')\n        rgb=rgb01(image);lab_value=rgb2lab(rgb)\n        hair=hair_before if stage=='before' else segment_labels(image)==HAIR_ID\n        ident=before_id if stage=='before' else identity_embedding(image)\n        lerr,aberr=color_quantile_error(lab_value,COLOR_LAB,fixed_core,REF_COLOR_CORE)\n        luminance=rgb@np.array([.299,.587,.114],np.float32)\n        high=luminance-gaussian_filter(luminance,1.2)\n        lap=laplace(luminance)\n        row=dict(model=model,stage=stage,shape_target_kind=TARGET_KIND,\n            musiq=musiq_score(image),\n            identity_cosine_vs_input=float(np.clip(np.dot(ident,SOURCE_ID),-1,1)),\n            identity_cosine_vs_before=float(np.clip(np.dot(ident,before_id),-1,1)),\n            hair_L_quantile_error=lerr,hair_ab_quantile_error=aberr,\n            hair_IoU_vs_shape_reference=mask_iou(hair,TARGET_HAIR),\n            hair_IoU_vs_before=mask_iou(hair,hair_before),\n            hair_deltaE76_vs_before=mask_mean(np.linalg.norm(lab_value-before_lab,axis=2),fixed_core),\n            hair_laplacian_variance=float(lap[fixed_core].var()) if fixed_core.sum()>=MIN_METRIC_PIXELS else np.nan,\n            hair_high_frequency_RMS=float(np.sqrt(mask_mean(high**2,fixed_core))),\n            hair_extreme_channel_fraction=mask_mean(((rgb<=.5/255)|(rgb>=254.5/255)).any(axis=2),fixed_core),\n            outside_edit_RGB_MAE_vs_before=mask_mean(np.abs(rgb-before_rgb),outside),\n            protected_RGB_MAE_vs_before=mask_mean(np.abs(rgb-before_rgb),protected_before),\n            protected_SSIM_vs_before=masked_ssim(rgb,before_rgb,protected_before),\n            protected_before_pixels=int(protected_before.sum()),\n            fixed_hair_core_pixels=int(fixed_core.sum()),predicted_hair_pixels=int(hair.sum()))\n        for name,mask in [('protected',COMMON_PROTECTED),('face',COMMON_FACE),('background',COMMON_BG)]:\n            row[name+'_RGB_MAE_vs_input']=mask_mean(np.abs(rgb-SOURCE_RGB),mask)\n            row[name+'_SSIM_vs_input']=masked_ssim(rgb,SOURCE_RGB,mask)\n            row[name+'_pixels']=int(mask.sum())\n        METRIC_ROWS.append(row)\nMETRIC_FRAME=pd.DataFrame(METRIC_ROWS)\nMETRIC_FRAME.to_csv(EVAL_DIR/'metrics_all_stages.csv',index=False)\n# desired direction: diagnostics are deliberately left without a good/bad label\nDIRECTIONS={\n 'musiq':1,'protected_RGB_MAE_vs_before':-1,'protected_SSIM_vs_before':1,\n 'identity_cosine_vs_input':1,'identity_cosine_vs_before':1,\n 'hair_L_quantile_error':-1,'hair_ab_quantile_error':-1,\n 'hair_IoU_vs_shape_reference':1,'hair_IoU_vs_before':1,\n 'hair_deltaE76_vs_before':0,'hair_laplacian_variance':0,'hair_high_frequency_RMS':0,\n 'hair_extreme_channel_fraction':0,'outside_edit_RGB_MAE_vs_before':0}\nfor area in ['protected','face','background']:\n    DIRECTIONS[area+'_RGB_MAE_vs_input']=-1;DIRECTIONS[area+'_SSIM_vs_input']=1\n\ndef interpretation(delta,direction):\n    if not np.isfinite(delta):return 'N/A: insufficient region or invalid measurement'\n    if abs(delta)<=1e-6:return 'unchanged within numerical tolerance'\n    if direction==0:return 'increased; inspect image' if delta>0 else 'decreased; inspect image'\n    return 'toward preferred metric direction; not overall quality' if delta*direction>0 else 'away from preferred metric direction; not overall quality'\nDELTA_ROWS=[]\nfor model in RESULTS:\n    block=METRIC_FRAME[METRIC_FRAME.model==model].set_index('stage')\n    for transition,start,end in [('diffusion','before','raw_diffusion'),('background_finish','raw_diffusion','before_sharp'),('native_sharp','before_sharp','after'),('total','before','after')]:\n        for metric,direction in DIRECTIONS.items():\n            a=float(block.loc[start,metric]);b=float(block.loc[end,metric]);delta=b-a\n            DELTA_ROWS.append(dict(model=model,transition=transition,metric=metric,before=a,after=b,\n                delta_after_minus_before=delta,desired_direction={1:'higher',-1:'lower',0:'diagnostic'}[direction],\n                interpretation=interpretation(delta,direction)))\nDELTA_FRAME=pd.DataFrame(DELTA_ROWS)\nDELTA_FRAME.to_csv(EVAL_DIR/'metrics_deltas.csv',index=False)\nEVALUATION_SECONDS=time.perf_counter()-metric_start\nprotocol=dict(version='before_after_native_sharp_metrics_v3',native_sharp={m:r['native_sharp'] for m,r in RESULTS.items()},musiq=MUSIQ_CONFIG,shape_target_kind=TARGET_KIND,\n    shape_target_sha256=sha_file(SHAPE_TARGET_MASK_PATH) if SHAPE_TARGET_MASK_PATH else None,\n    arcface_sha256=sha_file(id_path),metric_device=METRIC_DEVICE,\n    segmentation_model=SEG_MODEL_ID,segmentation_revision=ENVIRONMENT.get('seg_revision_resolved'),\n    min_pixels=MIN_METRIC_PIXELS,quantiles=np.linspace(.1,.9,9).tolist(),\n    ssim=dict(data_range=1,win_size=7,channel_axis=2,mask_safe_window=True),\n    support='fixed own-before hair core and protected area (outside edit mask eroded 4px); input comparisons retained as supplementary',\n    image_resolution=[1024,1024],evaluation_seconds_excluding_evaluator_setup=EVALUATION_SECONDS,\n    metric_directions=DIRECTIONS,not_comparable_to_old_FT_metrics=True,\n    caveats=['One image set is exploratory, not statistical evidence.',\n             'Shape reference is a pose-sensitive proxy unless supplied as aligned target mask.',\n             'Sharpness can increase with noise; no unconditional quality direction.',\n             'Preservation vs before measures drift, not reference correctness.',\n             'Per-model hair cores differ: use color/detail deltas mainly within each model.'])\n(EVAL_DIR/'evaluation_protocol.json').write_text(json.dumps(protocol,ensure_ascii=False,indent=2))\nmain=['model','stage','musiq','identity_cosine_vs_input','hair_L_quantile_error','hair_ab_quantile_error',\n      'hair_IoU_vs_before','protected_RGB_MAE_vs_before','protected_SSIM_vs_before']\ndisplay(METRIC_FRAME[main].round(5))\nfor model in RESULTS:\n    print('ก่อน → หลังทั้งหมด:',model)\n    display(DELTA_FRAME[(DELTA_FRAME.model==model)&(DELTA_FRAME.transition=='total')].round(6))\nprint('เวลาประเมินแยกจาก inference:',round(EVALUATION_SECONDS,2),'วินาที')\n"), (34, 'import matplotlib.pyplot as plt\npanels=[(\'musiq\',\'MUSIQ: predicted global quality (higher)\'),\n (\'identity_cosine_vs_input\',\'Identity vs input (higher)\'),\n (\'hair_ab_quantile_error\',\'Hair color reference error (lower)\'),\n (\'hair_IoU_vs_before\',\'Hair shape preservation vs before (higher)\'),\n (\'protected_RGB_MAE_vs_before\',\'Protected drift vs before (lower)\'),\n (\'protected_SSIM_vs_before\',\'Protected similarity vs before (higher)\')]\nfig,axes=plt.subplots(2,3,figsize=(17,9))\nstage_order=[\'before\',\'raw_diffusion\',\'before_sharp\',\'after\']\nfor ax,(metric,title) in zip(axes.flat,panels):\n    for model in RESULTS:\n        block=METRIC_FRAME[METRIC_FRAME.model==model].set_index(\'stage\').reindex(stage_order)\n        ax.plot(range(4),block[metric],marker=\'o\',label=model)\n    ax.set_xticks(range(4),[\'Before\',\'Diffusion\',\'Background\',\'Sharp\']);ax.set_title(title);ax.grid(alpha=.25)\naxes[0,0].legend(fontsize=8);fig.tight_layout()\nfig.savefig(EVAL_DIR/\'metrics_before_after.png\',dpi=160);display(fig);plt.close(fig)\nTHAI_VERDICT={\'toward preferred metric direction; not overall quality\':\'เข้าใกล้เป้าหมายของเมตริกนี้ ไม่ใช่คุณภาพรวม\', \'away from preferred metric direction; not overall quality\':\'ออกห่างเป้าหมายของเมตริกนี้ ไม่ใช่คุณภาพรวม\', \'improved on this metric\':\'ดีขึ้นตามเมตริกนี้\', \'worsened on this metric\':\'แย่ลงตามเมตริกนี้\', \'unchanged within numerical tolerance\':\'ไม่เปลี่ยนภายใน tolerance\', \'increased; inspect image\':\'เพิ่มขึ้น — ต้องดูภาพประกอบ\', \'decreased; inspect image\':\'ลดลง — ต้องดูภาพประกอบ\', \'N/A: insufficient region or invalid measurement\':\'วัดไม่ได้: พื้นที่ไม่พอหรือค่าไม่ถูกต้อง\'}\nlines=[\'# ผลก่อน–หลัง diffusion\',f\'Run: {RUN_ID}\',f\'Shape target: {TARGET_KIND}\',\n       \'\', \'MUSIQ เป็นคะแนนคุณภาพภาพโดยรวมจากโมเดล ไม่รับรองความสมจริงของเส้นผมเฉพาะส่วน; ใช้การประเมินคนประกอบ\', \'ตัวเลขมาจากภาพชุดนี้เท่านั้น ไม่ใช่ข้อสรุปเชิงสถิติ; higher sharpness ไม่รับประกันว่าภาพดีขึ้น\',\n       \'คะแนนชุดนี้ไม่เทียบตรงกับ FT report เก่า เพราะ protocol/masks/parser ต่างกัน\',\'\']\nfor model in RESULTS:\n    lines+=[\'## \'+model]\n    t=TIME_SUMMARY.set_index(\'model\').loc[model]\n    lines += [f"Transfer recorded: {t.transfer_recorded_s:.3f} s; diffusion: {t.diffusion_s:.3f} s; background: {t.background_s:.3f} s; native sharp: {t.native_sharp_s:.3f} s; postprocess total: {t.postprocess_total_s:.3f} s",\n      \'เวลา transfer ของสองโมเดลมีขอบเขตต่างกันตาม timing_stages.csv; ไม่ใช้ ratio สรุปความเร็วสถาปัตยกรรม\',\'\']\n    for transition in [\'diffusion\',\'background_finish\',\'native_sharp\',\'total\']:\n        lines+=[\'### \'+transition]\n        rows=DELTA_FRAME[(DELTA_FRAME.model==model)&(DELTA_FRAME.transition==transition)]\n        for _,r in rows.iterrows():\n            lines.append(f"- {r.metric}: {r.before:.6g} → {r.after:.6g} (Δ {r.delta_after_minus_before:+.6g}); {THAI_VERDICT.get(r.interpretation,r.interpretation)}")\n        lines+=[\'\']\n(EVAL_DIR/\'tradeoff_report.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\nprint(\'บันทึก CSV, กราฟ และ tradeoff_report.md ที่:\',EVAL_DIR)\nimport ipywidgets as widgets\nfrom google.colab import files\nbuttons=[]\nfor filename in [\'timing_summary.csv\',\'timing_stages.csv\',\'metrics_all_stages.csv\',\'metrics_deltas.csv\',\'metrics_before_after.png\',\'tradeoff_report.md\',\'evaluation_protocol.json\']:\n    path=EVAL_DIR/filename\n    button=widgets.Button(description=filename,layout=widgets.Layout(width=\'400px\'))\n    button.on_click(lambda _,p=path:files.download(str(p)));buttons.append(button)\ndisplay(widgets.VBox(buttons))\n'), (36, "RUN_POSTPROCESS_BENCHMARK=False\nBENCHMARK_REPEATS=3\nif RUN_POSTPROCESS_BENCHMARK:\n    if BENCHMARK_REPEATS<3:raise ValueError('ใช้ repeats อย่างน้อย 3')\n    benchmark_rows=[]\n    model_names=list(RESULTS)\n    for model in model_names:\n        warm=run_detail_core(before=RESULTS[model]['before'],**PROFILE)\n        del warm\n    for repeat in range(BENCHMARK_REPEATS):\n        order=model_names if repeat%2==0 else model_names[::-1]\n        for model in order:\n            measured=run_detail_core(before=RESULTS[model]['before'],**PROFILE)\n            benchmark_rows.append(dict(model=model,repeat=repeat+1,\n                **{k:measured['timings_seconds'][k] for k in ['mask','diffusion','raw_background_finish','native_sharp','total_processing']}))\n            del measured\n    benchmark=pd.DataFrame(benchmark_rows)\n    benchmark.to_csv(EVAL_DIR/'postprocess_repeated_timings.csv',index=False)\n    stats=benchmark.groupby('model')[['mask','diffusion','raw_background_finish','native_sharp','total_processing']].agg(['mean','std','median','min','max','count'])\n    stats.to_csv(EVAL_DIR/'postprocess_repeated_summary.csv');display(stats.round(3))\nelse:\n    print('ข้ามการรันซ้ำ; ตารางส่วน 13 แสดงเวลาจริงของรอบที่สร้างภาพแล้ว')\n"), (40, "import secrets, re\nfrom PIL import ImageDraw\nRATER_ID='participant_001'\nif not re.fullmatch(r'[A-Za-z0-9_-]{1,64}',RATER_ID):raise ValueError('RATER_ID ใช้ a-z 0-9 _ - ไม่เกิน 64 ตัว')\nHUMAN_ROOT=EVAL_DIR/'human_review_native_sharp';HUMAN_ROOT.mkdir(exist_ok=True)\nPRIVATE=HUMAN_ROOT/'private_keys';PRIVATE.mkdir(exist_ok=True)\npack=HUMAN_ROOT/RATER_ID;pack.mkdir(exist_ok=True)\nkey_path=PRIVATE/(RATER_ID+'.json')\nif key_path.exists():\n    assignment=json.loads(key_path.read_text())\n    if assignment['run_id']!=RUN_ID:raise ValueError('Review key is from another run')\nelse:\n    pairs=[(m,a,b) for m in RESULTS for a,b in [('before','raw_diffusion'),('before','after'),('before_sharp','after')]]\n    rng=secrets.SystemRandom();rng.shuffle(pairs)\n    assignment=dict(run_id=RUN_ID,rater_id=RATER_ID,pairs=[])\n    for index,(model,baseline,stage) in enumerate(pairs,1):\n        left,right=(baseline,stage) if rng.choice([True,False]) else (stage,baseline)\n        assignment['pairs'].append(dict(pair_id=f'P{index:02d}',model=model,left=left,right=right,baseline=baseline,comparison=baseline+'_to_'+stage))\n    key_path.write_text(json.dumps(assignment,indent=2))\nfor pair in assignment['pairs']:\n    out=pack/(pair['pair_id']+'.png')\n    if out.exists():continue\n    left=RESULTS[pair['model']][pair['left']];right=RESULTS[pair['model']][pair['right']]\n    if left.size!=right.size:raise ValueError('Review images need equal size')\n    w,h=left.size;canvas=Image.new('RGB',(2*w,h+48),'white')\n    canvas.paste(left,(0,48));canvas.paste(right,(w,48))\n    draw=ImageDraw.Draw(canvas);draw.text((20,16),pair['pair_id']+'   A (left)',fill='black')\n    draw.text((w+20,16),'B (right)',fill='black');canvas.save(out)\nresponse_path=pack/'responses.csv'\nif not response_path.exists():\n    pd.DataFrame([dict(run_id=RUN_ID,rater_id=RATER_ID,pair_id=p['pair_id'],\n        natural_hair='',natural_boundary='',overall_quality='') for p in assignment['pairs']]).to_csv(response_path,index=False)\n(pack/'instructions.txt').write_text('Choose L = A (left), R = B (right), or Tie for each question.\\nNatural hair; natural blending of hair edges; overall image quality.\\nDo not infer model names. Leave unanswered items blank.\\n',encoding='utf-8')\nbuttons=[]\nfor path in sorted(pack.glob('*.png'))+[response_path,pack/'instructions.txt']:\n    button=widgets.Button(description=RATER_ID+' / '+path.name,layout=widgets.Layout(width='450px'))\n    button.on_click(lambda _,p=path:files.download(str(p)));buttons.append(button)\ndisplay(widgets.VBox(buttons))\nprint('ชุดสำหรับผู้ประเมิน:',pack)\nprint('เฉลยเก็บแยก ไม่รวมในปุ่มดาวน์โหลด')\n")]
if RUN_NEW_IMAGES:
    if '_HFG_DEVICE_LAYOUT' not in globals():raise RuntimeError('โหลดส่วน 4 ของไฟล์รุ่นใหม่ก่อน')
    for index,source in NEW_RUN_STEPS:
        print('Running cell',index)
        exec(compile(source,f'<new_run_{index}>','exec'),globals())
else: print('เปลี่ยนชุดรูป: ตั้ง RUN_NEW_IMAGES=True แล้วรันเซลล์นี้')


## 19. ภาพ blind A/B สำหรับคนประเมิน — 6 คู่
แต่ละโมเดลมี 3 คู่: ก่อน→raw diffusion, ก่อน→final, ก่อนเพิ่มความคม→final ใช้ภาพ 1024px จริงวางคู่กัน สุ่มคู่และฝั่งซ้าย/ขวาแยกต่อผู้ประเมิน
เลือก L (ภาพ A ซ้าย), R (ภาพ B ขวา), Tie แยกเส้นผมเป็นธรรมชาติ / ขอบผมกลืนกับภาพ / คุณภาพรวม
ส่งเฉพาะ PNG, instructions.txt, responses.csv จากโฟลเดอร์ผู้ประเมิน ไม่ส่ง private_keys หรือโน้ตบุ๊กที่มีเฉลย
เปลี่ยน RATER_ID แล้วรันเพื่อสร้างชุดคนถัดไป การรันรหัสเดิมคง mapping และคำตอบเดิม


In [ ]:
import secrets, re
from PIL import ImageDraw
RATER_ID='participant_001'
if not re.fullmatch(r'[A-Za-z0-9_-]{1,64}',RATER_ID):raise ValueError('RATER_ID ใช้ a-z 0-9 _ - ไม่เกิน 64 ตัว')
HUMAN_ROOT=EVAL_DIR/'human_review_native_sharp';HUMAN_ROOT.mkdir(exist_ok=True)
PRIVATE=HUMAN_ROOT/'private_keys';PRIVATE.mkdir(exist_ok=True)
pack=HUMAN_ROOT/RATER_ID;pack.mkdir(exist_ok=True)
key_path=PRIVATE/(RATER_ID+'.json')
if key_path.exists():
    assignment=json.loads(key_path.read_text())
    if assignment['run_id']!=RUN_ID:raise ValueError('Review key is from another run')
else:
    pairs=[(m,a,b) for m in RESULTS for a,b in [('before','raw_diffusion'),('before','after'),('before_sharp','after')]]
    rng=secrets.SystemRandom();rng.shuffle(pairs)
    assignment=dict(run_id=RUN_ID,rater_id=RATER_ID,pairs=[])
    for index,(model,baseline,stage) in enumerate(pairs,1):
        left,right=(baseline,stage) if rng.choice([True,False]) else (stage,baseline)
        assignment['pairs'].append(dict(pair_id=f'P{index:02d}',model=model,left=left,right=right,baseline=baseline,comparison=baseline+'_to_'+stage))
    key_path.write_text(json.dumps(assignment,indent=2))
for pair in assignment['pairs']:
    out=pack/(pair['pair_id']+'.png')
    if out.exists():continue
    left=RESULTS[pair['model']][pair['left']];right=RESULTS[pair['model']][pair['right']]
    if left.size!=right.size:raise ValueError('Review images need equal size')
    w,h=left.size;canvas=Image.new('RGB',(2*w,h+48),'white')
    canvas.paste(left,(0,48));canvas.paste(right,(w,48))
    draw=ImageDraw.Draw(canvas);draw.text((20,16),pair['pair_id']+'   A (left)',fill='black')
    draw.text((w+20,16),'B (right)',fill='black');canvas.save(out)
response_path=pack/'responses.csv'
if not response_path.exists():
    pd.DataFrame([dict(run_id=RUN_ID,rater_id=RATER_ID,pair_id=p['pair_id'],
        natural_hair='',natural_boundary='',overall_quality='') for p in assignment['pairs']]).to_csv(response_path,index=False)
(pack/'instructions.txt').write_text('Choose L = A (left), R = B (right), or Tie for each question.\nNatural hair; natural blending of hair edges; overall image quality.\nDo not infer model names. Leave unanswered items blank.\n',encoding='utf-8')
buttons=[]
for path in sorted(pack.glob('*.png'))+[response_path,pack/'instructions.txt']:
    button=widgets.Button(description=RATER_ID+' / '+path.name,layout=widgets.Layout(width='450px'))
    button.on_click(lambda _,p=path:files.download(str(p)));buttons.append(button)
display(widgets.VBox(buttons))
print('ชุดสำหรับผู้ประเมิน:',pack)
print('เฉลยเก็บแยก ไม่รวมในปุ่มดาวน์โหลด')


## 20. รับคะแนนคนและสรุป preference (เลือกทำเมื่อได้คำตอบจริง)
ตั้ง UPLOAD_HUMAN_RESPONSES=True เพื่ออัปโหลด CSV ที่กรอกแล้ว ระบบตรวจ run/rater/pair/คำตอบก่อนเก็บ รองรับไฟล์หลายคน ไม่รับคะแนนซ้ำคนเดิมต่อคู่
เปอร์เซ็นต์เลือก after = จำนวนเลือก after / จำนวนคำตอบที่ตอบจริง (รวม Tie ในตัวหาร) แสดง before/after/tie และจำนวนคำตอบแยกกัน ไม่รวม raw และ final เข้าด้วยกัน
ผู้ประเมินหลายคนของภาพเดียวไม่ใช่หลายกรณีภาพ ผลนี้ยังไม่ใช่หลักฐานข้าม dataset

In [ ]:
UPLOAD_HUMAN_RESPONSES=False
HUMAN_ROOT=EVAL_DIR/'human_review_native_sharp';PRIVATE=HUMAN_ROOT/'private_keys'
QUESTIONS=['natural_hair','natural_boundary','overall_quality']
def validate_responses(frame):
    required={'run_id','rater_id','pair_id',*QUESTIONS}
    if not required<=set(frame.columns):raise ValueError('Response CSV columns incomplete')
    if frame.duplicated(['run_id','rater_id','pair_id']).any():raise ValueError('Duplicate response pair')
    for row in frame.to_dict('records'):
        if row['run_id']!=RUN_ID:raise ValueError('CSV belongs to another image run')
        rid=str(row['rater_id'])
        if not re.fullmatch(r'[A-Za-z0-9_-]{1,64}',rid):raise ValueError('Invalid rater ID')
        key=json.loads((PRIVATE/(rid+'.json')).read_text())
        if row['pair_id'] not in {p['pair_id'] for p in key['pairs']}:raise ValueError('Unknown pair')
        for question in QUESTIONS:
            if str(row[question]).strip() not in ['','L','R','Tie']:raise ValueError('Use L / R / Tie or blank')
    return frame
if UPLOAD_HUMAN_RESPONSES:
    import io
    uploads=files.upload()
    incoming=[validate_responses(pd.read_csv(io.BytesIO(data),keep_default_na=False)) for data in uploads.values()]
    combined=pd.concat(incoming,ignore_index=True);validate_responses(combined)
    for rid,frame in combined.groupby('rater_id'):
        target=HUMAN_ROOT/rid/'responses.csv'
        old=validate_responses(pd.read_csv(target,keep_default_na=False))
        old=old.set_index('pair_id');new=frame.set_index('pair_id')
        for pair in new.index:
            for q in QUESTIONS:
                value=str(new.at[pair,q]).strip()
                if value:
                    prior=str(old.at[pair,q]).strip()
                    if prior and prior!=value:raise ValueError('Conflicting existing response; edit original CSV explicitly')
                    old.at[pair,q]=value
        old.reset_index().to_csv(target,index=False)
rows=[]
for path in HUMAN_ROOT.glob('*/responses.csv'):
    frame=validate_responses(pd.read_csv(path,keep_default_na=False))
    for row in frame.to_dict('records'):
        mapping=json.loads((PRIVATE/(row['rater_id']+'.json')).read_text())
        pair=next(p for p in mapping['pairs'] if p['pair_id']==row['pair_id'])
        for question in QUESTIONS:
            answer=str(row[question]).strip()
            if not answer:continue
            preferred='tie' if answer=='Tie' else ('before' if pair['left' if answer=='L' else 'right']==pair['baseline'] else 'after')
            rows.append(dict(run_id=RUN_ID,rater_id=row['rater_id'],model=pair['model'],comparison=pair['comparison'],question=question,preference=preferred))
if rows:
    responses=pd.DataFrame(rows);responses.to_csv(EVAL_DIR/'human_preferences.csv',index=False)
    summary=responses.groupby(['model','comparison','question','preference']).size().unstack(fill_value=0).reindex(columns=['before','after','tie'],fill_value=0)
    summary['n_answers']=summary.sum(axis=1)
    summary['after_preference_percent']=100*summary['after']/summary.n_answers
    summary.to_csv(EVAL_DIR/'human_preference_summary.csv');display(summary)
else:print('ยังไม่มีคะแนนคนจริง ไม่สร้างคะแนนหรือสรุปผู้ชนะ')


## 21. ใช้ผล raw เดิมหรือปรับความแรงใหม่ โดยไม่รัน GAN/diffusion ซ้ำ
รันส่วน 9 ก่อน แล้วตั้ง RUN_APPLY_NATIVE_TO_EXISTING=True ในเซลล์นี้ ปรับความแรงได้ด้านใน ส่งออกโฟลเดอร์ใหม่เสมอ แล้วรันส่วน 11–16 เพื่อประเมินใหม่ เวลารวมเป็นผลรวมเวลาขั้นที่เคยวัด ไม่ใช่ wall time ของการกด retune ครั้งเดียว

In [ ]:
RUN_APPLY_NATIVE_TO_EXISTING=False
if RUN_APPLY_NATIVE_TO_EXISTING:
    import time
    if 'finish_native_sharp' not in globals():raise RuntimeError('รันส่วน 9 ของไฟล์นี้ก่อน')
    if 'RESULTS' not in globals() or len(RESULTS)!=2:raise RuntimeError('ต้องมี RESULTS ของทั้งสองโมเดลใน session')
    NATIVE_SHARP_ENABLED=True
    NATIVE_HAIR_AMOUNT=.65
    NATIVE_FACE_AMOUNT=.25
    NATIVE_SHARP_MAX_DELTA=5.0
    NATIVE_SHARP_THRESHOLD=.8
    for model,result in RESULTS.items():
        if 'sharp_labels' not in result:
            t=clock()
            labels=segment_labels(result['raw_diffusion'])
            base,alpha,info=finish_raw_background(result['raw_diffusion'],result['masks'],labels=labels)
            result.update(before_sharp=base,sharp_labels=labels,background_alpha=alpha,background_info=info)
            result['timings_seconds']['raw_background_finish']=clock()-t
        # Always sharpen the saved pre-sharp image; never accumulate sharpening.
        t=clock()
        final,info=finish_native_sharp(result['before_sharp'],result['sharp_labels'])
        result.update(after=final,native_sharp=info)
        result['timings_seconds']['native_sharp']=clock()-t
        result['timings_seconds']['total_processing']=sum(result['timings_seconds'].get(k,0.) for k in ['mask','diffusion','raw_background_finish','native_sharp'])
    NATIVE_REEXPORT_SOURCE="import pandas as pd\nEXPORT_IMAGES=[]; TIMING_ROWS=[]\nFINAL_DIR=RUN_DIR/('comparison_native_sharp_retune_'+uuid.uuid4().hex[:8]); FINAL_DIR.mkdir(exist_ok=True)\nMODELS=[('Barbershop_Baseline',BASE_DIR/'Barbershop_Baseline.png'),\n        ('HairFastGAN_B24',BASE_DIR/'HairFastGAN_B24.png')]\nPROFILE=dict(seed=int(SEED),strength=STRENGTH,steps=STEPS,guidance=GUIDANCE,face_blend=FACE_BLEND)\nfor model,path in MODELS:\n    print('Diffusion:',model)\n    with Image.open(path) as im: before=im.convert('RGB').copy()\n    result=RESULTS[model]\n    # shared legacy core calls its GAN timing B24; remove this unused zero-valued field\n    result['timings_seconds'].pop('B24',None)\n    RESULTS[model]=result\n    dest=FINAL_DIR/model; dest.mkdir(exist_ok=True)\n    for index,key,label in [(1,'before','Before_Diffusion'),(2,'raw_diffusion','Raw_Diffusion'),\n                            (3,'before_sharp','After_Background_Before_Sharp'),\n                            (4,'after','Final_Native_Sharp')]:\n        out=dest/f'{model}_{index:02d}_{label}_{RUN_ID}.png'\n        result[key].save(out,format='PNG',compress_level=1)\n        EXPORT_IMAGES.append(out)\n    result['masks']['raw_mask'].save(dest/'diffusion_mask.png')\n    for name in ['hair','face','alpha']:\n        Image.fromarray(np.rint(result['masks'][name]*255).astype('uint8')).save(dest/(name+'_mask.png'))\n    Image.fromarray(np.rint(result['background_alpha']*255).astype('uint8')).save(dest/'background_alpha.png')\n    raw=np.asarray(result['raw_diffusion']);background=np.asarray(result['before_sharp']);after=np.asarray(result['after'])\n    keep=result['background_alpha']==0\n    assert np.array_equal(raw[keep],background[keep]),'Background finish changed protected raw pixels'\n    sharp_protected=~np.isin(result['sharp_labels'],[HAIR_ID,FACE_ID])\n    assert np.array_equal(background[sharp_protected],after[sharp_protected]),'Native sharp changed non-hair/face pixels'\n    observed=int(np.abs(after.astype('int16')-background.astype('int16')).max())\n    assert observed<=np.ceil(NATIVE_SHARP_MAX_DELTA),'Native sharp exceeded bounded delta'\n    Image.fromarray(result['sharp_labels'].astype('uint8')).save(dest/'native_sharp_labels.png')\n    transfer_seconds=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_RESULT['estimated_cold_compute_seconds']\n    TIMING_ROWS.append(dict(model=model,transfer_compute_seconds=transfer_seconds,\n        transfer_wall_seconds_this_run=HFG_SECONDS if model=='HairFastGAN_B24' else BARBER_WALL_SECONDS,\n        **result['timings_seconds']))\n    metadata=dict(model=model,profile=PROFILE,settings=result['settings'],\n        base_sha256=sha_file(path),background_info=result['background_info'],native_sharp=result['native_sharp'],\n        timings=TIMING_ROWS[-1],face_blend_is_composite_weight=False,\n        mask_pixels={k:int(result['masks'][k].sum()) for k in ['hair','face']})\n    (dest/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2))\nTIMINGS=pd.DataFrame(TIMING_ROWS)\nTIMINGS['transfer_plus_postprocess_seconds']=TIMINGS.transfer_compute_seconds+TIMINGS.total_processing\nTIMINGS.to_csv(FINAL_DIR/'timings.csv',index=False)\nmanifest=dict(method='looklab_native_sharp',native_source_sha256='d4c5f63c6f895bb4b7569a8f5104a373113ef33c046c81c1534a7fea4634072d',run_id=RUN_ID,profile=PROFILE,environment=ENVIRONMENT,\n    barbershop=dict(revision=BARBER_REV,weights=weight_hashes,selection='Baseline',\n        selected_from_validation_protocol='27f7f146ef0d',worker_result=BARBER_RESULT),\n    hairfast=dict(revision=REPO_COMMIT,checkpoint=str(CHECKPOINT),sha256=EXPECTED_SHA256,\n        trial='B',epoch=24,mixing=.95,shape_rotation=False,color_rotation=True),\n    original_hashes={p.name:sha_file(p) for p in INPUTS},\n    aligned_hashes={p.name:sha_file(p) for p in CROP_DIR.glob('*.png')},\n    export_images=[str(p) for p in EXPORT_IMAGES],\n    notes=['Same aligned crops; Barbershop consumes quantized PNGs.',\n           'Same postprocess algorithm/settings; masks depend on each output.',\n           'Timing excludes initial weight downloads and model loading for HFG; BS wall includes worker startup.',\n           'BS transfer compute includes inversion even when cached; wall time is shown separately.',\n           'Native sharp operates at original resolution without extra diffusion; metric results in evaluation folder.'])\n(FINAL_DIR/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))\ndisplay(TIMINGS)\nprint('บันทึกภาพเต็ม 8 รูป:',FINAL_DIR)\n"
    exec(compile(NATIVE_REEXPORT_SOURCE,'<native_reexport>','exec'),globals())
    for model in RESULTS:print(model,RESULTS[model]['native_sharp'])
    print('Exported native outputs. Rerun sections 11–16 for graphs and metrics. Invalidate old human review before changing outputs.')
else:print('ใช้ผลเดิม: ตั้ง RUN_APPLY_NATIVE_TO_EXISTING=True หลังนิยามฟังก์ชันส่วน 9')
